# Superfermion: full 4–24-qubit grid, five-step VQC/VQR, RAM and time

The default `requested_grid` preset matches your revised scope:

- **VQC/VQR:** 4/8/12 qubits, exactly **5 full-batch SPSA iterations**, depth 1.
- **Other quantum experiments:** 4/8/12/16/20/24 qubits, including kernels,
  QKPCA, feature selection, MaxCut, VQE, gradients, state/feature-map, batch and depth sweeps.
- **All seven datasets:** full available rows in an **80% training / 20% testing** split.
- Eight simulator lanes: published/source/modified Superfermion, Qiskit Aer,
  PennyLane Lightning, qsim/Cirq f32, Cirq f64 and Qulacs; separate classical controls.

The complete supported plan has **2,520 benchmark jobs**, one seed/repeat and
one simulator thread. Wine has only 13 original features; feature selection at
16/20/24 qubits uses those real features plus 3/7/11 ancillas. Ancillas are absent
from the feature cost and downstream feature list. Their embedding is audited;
they do not create new original features or dataset rows.

Every exact experiment condition gets **three-panel scaling figures**:
primary computation time, full worker elapsed time and peak resident RAM.
Figures cover the requested qubit ticks and every applicable framework. CSVs,
PNG/SVG figures, numerical audits, failure reasons and checkpoints are exported.
Failed/pending measurements appear as gaps, never as zero or a claimed win.
Small qubit bands run first, with framework order shuffled within groups.
Time/RAM reports refresh between jobs about every 15 minutes, so partial graphs
can be inspected/downloaded before an expensive section finishes.

### Large-qubit execution

A complex128 state at 24 qubits is 256 MiB. A full 569-row Breast Cancer bank
would be **142.25 GiB**. Large kernels therefore use identical bounded re-encoding
blocks across all simulators; this trades memory for more simulation time.
State/gradient/readout batches are chunked. The exported-bank budget is 512 MiB,
which is **not** a cap on total process RAM. VQE's exact open-chain Ising reference
uses an n-by-n fermion matrix instead of an exponentially large dense spin matrix.

This grid remains expensive on free Colab. Each job has a visible **600-second
timeout**; a timed-out/OOM worker is recorded as a failure and the campaign can
continue. Required preflight/core/numerical correctness gates still stop on error.
Optional full project tests are off for this preset. No universal winner is assumed.

### Execution status

This notebook's full native-framework campaign has **not been run** here.
The notebook structure/syntax, grid plan and small independent numerical helpers
are checked locally. Run cells in order for measured graphs and results; a free
Colab session may not finish the 20/24-qubit dataset kernels. Results use the new
`studies/requested_grid_split80_20_grid24_v8` directory.


### Live job progress

Campaign cells now show one updating progress panel: finished/planned jobs,
percentage complete, successful/failed/remaining counts, and a per-task table.
Each job is one framework running one experiment configuration. Progress reads
controller-confirmed worker results and section checkpoints, including accepted
cached successes. It refreshes every `PROGRESS_REFRESH_SECONDS=5` seconds.
Percentages count finished jobs, including failures; they are not time estimates.
Preparation, correctness checks and report rendering have no benchmark-job percentage.
Logs and original error details remain available. The scientific scripts, settings,
manifest generation and cache fingerprints are unchanged by this display update.

For an existing notebook, run the separately supplied progress-cell code after
setup and before the next campaign cell. It leaves your selected profile and
saved results intact. Apply it while the notebook kernel is idle; a cell already
executing the old helper continues to use that helper until it stops or finishes.


## Context & Methods

We ask two different questions: *How much does the Rust patch reduce an
equivalent application's runtime?* and *Does each QML method produce useful
predictions on the same held-out data?* Simulator speed alone does not
establish quantum ML advantage. A win is measured; it is never required by
editing results, choosing weaker baselines, or dropping failed jobs.

| Simulator lane | Actual CPU backend | Arithmetic | Repeated execution |
|---|---|---|---|
| SF published | Superfermion 0.1.14 PyPI wheel | complex128 | Existing bind → IR → simulate |
| SF source | Unchanged pinned Git source | complex128 | Same existing path, same Rust toolchain as the patch |
| SF modified | Pinned source + embedded patch | complex128 | Prepared circuit, native batch/readouts |
| Qiskit | Aer statevector | complex128 | Runtime parameter binding batches |
| PennyLane | lightning.qubit | complex128 | QNode broadcasting; Lightning expands tapes |
| Cirq/qsim | QSimSimulator | complex64 internally | Parameter sweep |
| Cirq f64 | Cirq Simulator | complex128 | Parameter sweep |
| Qulacs | CPU parametric statevector simulator | complex128 | Reuse circuit/state; update parameters |

### Key assumptions

* Every compared circuit has identical logical gates and angle values.
  Canonical vector index bit `q` represents qubit `q`. An asymmetric
  `X(q0)` case catches endian mistakes; Qulacs rotation signs are converted.
* All simulator lanes receive the same thread **budget**, and BLAS uses one
  thread. A backend can choose to use less than its budget at small sizes.
  No worker runs concurrently with another timed worker.
* A **statevector track** returns a complex128 bank and applies a shared
  NumPy reduction. qsim's output cast does not change its f32 arithmetic.
  A **native-readout track** computes the equivalent scalar through each
  backend's observable API, avoiding state export where supported.
* Primary runtime includes parameter binding, simulation, required output
  conversion, and the named application stage. It excludes installation,
  imports, dataset preprocessing, circuit preparation, warmup and independent
  reference audits. Shared exported-state norm reductions are included in
  circuit and kernel primary timings.
  `circuit_prepare_s` includes the first backend import in a fresh worker.
* Kernel primary time is **encoding + fidelity matrix algebra**. Classical
  fitting/eigendecomposition and end-to-end time are recorded separately.
  VQC/VQR primary time includes the full matched optimizer workload; test
  inference is separate. Optimization solution diagnostics are outside
  native training time for every lane.
* Circuit/gradient microbenchmarks use a within-process median, then report
  median/IQR across fresh processes. QML/optimization runs perform one
  matched workload per fresh process. IQR is descriptive, **not** a CI.
* Exact statevector simulation uses \(O(2^n)\) amplitudes. No MPS,
  stabilizer shortcut, GPU, noise, sampling, or hardware execution is
  mixed into the general simulator ranking.

### What is changed in Superfermion

The patch is the same verified library change used in the earlier notebook.
It is applied to commit `35f8f7c306ac2a004e6b10dee37317d87918d5dc`.
Both source lanes build with Rust 1.90.0 and identical release/CPU flags;
the source baseline isolates the patch's effect from wheel build choices.

| Change | Why it can help | What this notebook checks |
|---|---|---|
| Immutable prepared circuit | Reuse topology, fixed gates and operation order | Changing parameters still changes states correctly |
| Native batched state export | Reduce Python/DAG work and export copies | Native batch APIs are enabled for competing backends too |
| Mean-Z / cached diagonal readout | Avoid statevector/probability export and repeated scans | Scalar agrees with independent full states |
| Bitmask Pauli expectations | Reduce observable traversal overhead | Mixed X/Y/Z agrees with dense matrices |
| GIL release / optional device exports | Reduce wrapper work and allow native execution | Compiled unit tests preserve existing defaults |
| Exact Angle fidelity kernel | Avoid simulation for one restricted map | Kept separate from general simulator rankings |

Parameter-dependent matrices/fusion are **rebuilt for each parameter row**;
a prepared circuit does not reuse stale numerical results. Measurement,
reset and conditioned circuits are rejected by the prepared API. State
batches are limited to 1 GiB by the patch; this notebook uses a tighter guard.

Keep the Rust core while measuring these changes. A whole-library C++ or
assembly rewrite does not remove repeated wrapper work or exponential
memory growth. Use the measured losses against Qulacs/Aer/Lightning to choose
the next kernel, fusion, SIMD or allocation optimization. Optional Torch/JAX
integration changes in the patch are not benchmarked by this notebook.

### Experiment matrix

| Family | Experiments and outputs |
|---|---|
| Classification | QSVM on all five classification datasets; binary and three-class labels; accuracy, balanced accuracy, macro-F1 and binary AUROC |
| Regression | Fidelity-kernel ridge regression on Diabetes/Friedman1; VQR on Diabetes/Friedman1; RMSE, MAE and R² in original target units |
| Representation | QKPCA fit on training data only; out-of-sample centering and a logistic head; spectrum and held-out quality |
| Variational QML | VQC binary logistic / multiclass one-vs-rest circuits, matched SPSA seeds, full training-loss traces and inference time |
| Combinatorial optimization | QAOA feature selection, weighted ring and Erdős–Rényi MaxCut; expected energy, exact optimum gap and optimum probability |
| Physics | VQE on the open transverse-field Ising chain; mixed Pauli expectations and exact open-chain ground energy |
| Differentiation | Matched parameter-shift gradients checked against independent finite differences; state-export and scalar tracks |
| Simulator scaling | GHZ, decomposed QFT on a nontrivial input, and random entangling circuits; qubits, depth, batch size and process memory |
| Feature-map sensitivity | Angle, Phase, Pauli and ZZ maps, all with identical gate specifications across backends |
| Classical controls | RBF SVC, logistic regression, random forest; ridge, RBF kernel ridge, random forest regression; full-feature classification control |

The validation profile covers all of these families. Its short optimizer
budget tests execution and agreement, not model convergence. All profiles use
full 80/20 datasets; presets vary qubits, split seeds, repeats and training budgets. CSV inputs can be added through `CUSTOM_DATASETS`.

### 1. Configure the requested grids

| Experiment | Qubits | Update budget |
|---|---|---|
| VQC / VQR | 4, 8, 12 | **5 full-batch SPSA iterations**, depth 1 |
| QSVM / QKRR / QKPCA | 4, 8, 12, 16, 20, 24 | Kernel computation / classical fit |
| QAOA feature selection / MaxCut / VQE | 4, 8, 12, 16, 20, 24 | 5 iterations inherited from the free preset; separately configurable |
| Parameter-shift gradients | 4, 8, 12, 16, 20, 24 | Same shifted-circuit algorithm and independent finite-difference audit |
| State, feature-map, batch, depth sweeps | 4, 8, 12, 16, 20, 24 | 1 warmup + 3 inner repetitions |
| Classical controls | All six projected input sizes | Matched full-data split |

All seven bundled datasets use their complete **80/20 split**. Classification is
stratified; all preprocessing/feature selection is trained only on training rows.
Each VQC/VQR SPSA update uses every row in the 80% training partition; the 20%
test partition is held out for final evaluation. Five iterations means five
parameter updates, with an initial and five updated full-training-loss points.
Zero padding above available PCA components adds no new information. Wine feature
selection above 13 qubits embeds its 13-variable problem with idle-cost ancillas;
cardinality is based on real variables, and ancillary bits are not selected features.

`QUANTUM_QUBITS`, `VQC_VQR_QUBITS` and `VQC_VQR_ITERATIONS` are separate settings.
`VQC_VQR_FULL_BATCH=True` implements full-training-partition updates.
The default plan has **2,520 jobs**; `MAX_JOBS=4000` allows this requested grid.
`MAX_STATEBANK_MIB=512` permits two 24-qubit output states for bounded kernel tiles.
It does not include backend workspaces, probabilities, diagonal observables or
independent references. Kernel block caps and logical/actual bank estimates are
visible in `statebank_budget.csv`; no full dataset is subsampled to fit RAM.

Large kernels recompute states instead of keeping a 142 GiB bank or writing it
to disk. Timings describe this bounded algorithm, not an in-memory-bank benchmark.
All simulator lanes share the same strategy at a given qubit/dataset condition.
Native readouts still simulate a state internally; GPU acceleration is not enabled.
The exact VQE reference applies specifically to the open TFIM Hamiltonian used here.

`timeout_s=600` is a per-worker limit, not a completion-time estimate. Increase it
only if you intend to spend longer per job. Failures remain in coverage tables and
figures; paired speedups require every simulator in the matched group to succeed.
Independent smoke/reference checks use 4/8/12 qubits, and every higher-qubit result
has its job-level norm/full-vector or observable audit. The numerical gates are not bypassed.
Change configuration before preparing data/manifest/validation; rerun those cells
after any change. Export/download checkpoints before the Colab runtime is deleted.


In [1]:
from pathlib import Path
import base64, csv, hashlib, html, json, os, platform, shutil, subprocess, sys, time, urllib.request, venv, zlib

RUN_PROFILE = os.environ.get("SUPERFERMION_STUDY_PROFILE", "requested_grid")
TRAIN_RATIO = 0.80
TEST_RATIO = 0.20
THREADS = [16]                 # Desktop run: 16 simulator threads per quantum lane (multi-threaded dataset).
MAX_JOBS = 4000                 # Requested grid: 2,520 benchmark jobs.
ALLOW_LARGE_RUN = True         # Authorizes the requested six-qubit-size grid.
MAX_STATEBANK_MIB = 512         # Output bank limit, not a total process-RSS limit.
RUN_FULL_PROJECT_TESTS = RUN_PROFILE not in ("colab_free","requested_grid")  # Optional full suites; core/unit/correctness checks always run.
CUSTOM_DATASETS = {}            # e.g. {"my_data": {"path":"/content/data.csv", "target":"label", "kind":"classification"}}

CLASSIFICATION = ["breast_cancer", "iris", "wine", "digits_01", "moons"]
REGRESSION = ["diabetes", "friedman1"]
if not (0 < TRAIN_RATIO < 1 and 0 < TEST_RATIO < 1
        and abs(TRAIN_RATIO + TEST_RATIO - 1.0) < 1e-12):
    raise ValueError("Train and test ratios must be positive and sum to 1.")
COMMON = dict(classification_datasets=CLASSIFICATION, regression_datasets=REGRESSION,
    train_ratio=TRAIN_RATIO, test_ratio=TEST_RATIO,
    kernel_maps=["angle", "zz"], warmups=2, inner_repetitions=3, timeout_s=1800,
    qaoa_depth=1, map_batch=16, batch_qubits=8, batch_sizes=[1,8,32], circuit_depths=[1,4,8])
QUANTUM_QUBITS = [4,8,12,16,20]
VQC_VQR_QUBITS = [4,8,12]
VQC_VQR_ITERATIONS = 5
VQC_VQR_FULL_BATCH = True  # Every update uses all rows in the training partition.
OTHER_OPTIMIZER_ITERATIONS = 5  # Retains the preceding free preset's budget for QAOA/MaxCut/VQE.
SHOW_ALL_EXPERIMENT_FIGURES = True
PROFILES = {
    "requested_grid": dict(COMMON, seeds=[42], repetitions=1,
        qml_qubits=QUANTUM_QUBITS, variational_qubits=VQC_VQR_QUBITS,
        variational_iterations=VQC_VQR_ITERATIONS, variational_full_batch=VQC_VQR_FULL_BATCH,
        iterations=OTHER_OPTIMIZER_ITERATIONS,
        optimization_iterations=OTHER_OPTIMIZER_ITERATIONS,
        validation_qubits=[4,8,12], state_qubits=QUANTUM_QUBITS,
        map_qubits=QUANTUM_QUBITS, sweep_qubits=QUANTUM_QUBITS,
        qkpca_datasets=CLASSIFICATION, vqc_datasets=CLASSIFICATION, vqr_datasets=REGRESSION,
        feature_selection_datasets=["breast_cancer","wine"], feature_selection_ancillas=True, variational_depths=[1],
        training_batch=6, vqe_qubits=QUANTUM_QUBITS, gradient_qubits=QUANTUM_QUBITS,
        warmups=1, inner_repetitions=3, qaoa_depth=1, map_batch=8,
        batch_sizes=[1,8], circuit_depths=[1,4], kernel_block_rows=128, timeout_s=1200),
    "colab_free": dict(COMMON, classification_datasets=["breast_cancer","iris"],
        regression_datasets=["friedman1"], seeds=[42], repetitions=1,
        qml_qubits=[4,8], validation_qubits=[4,8], state_qubits=[4,8,12], map_qubits=[4,8],
        qkpca_datasets=["iris"], vqc_datasets=["iris"], vqr_datasets=["friedman1"],
        feature_selection_datasets=["breast_cancer"], variational_depths=[1],
        iterations=5, training_batch=6, vqe_qubits=[4], gradient_qubits=[4],
        warmups=1, inner_repetitions=3, qaoa_depth=1, map_batch=8,
        batch_sizes=[1,8], circuit_depths=[1,4], timeout_s=600),
    "validation": dict(COMMON, seeds=[42], repetitions=2,
        qml_qubits=[4], validation_qubits=[4,8], state_qubits=[4,8,12], map_qubits=[8],
        qkpca_datasets=["breast_cancer","iris"], vqc_datasets=["breast_cancer","iris"],
        vqr_datasets=["friedman1"], feature_selection_datasets=["breast_cancer"],
        variational_depths=[1], iterations=3, training_batch=6, vqe_qubits=[4], gradient_qubits=[4]),
    "study": dict(COMMON, seeds=[42,7,123], repetitions=3,
        qml_qubits=[4,8], validation_qubits=[4,8], state_qubits=[4,8,12,16], map_qubits=[4,8,12],
        qkpca_datasets=CLASSIFICATION, vqc_datasets=["breast_cancer","iris","moons"],
        vqr_datasets=REGRESSION, feature_selection_datasets=["breast_cancer","wine"],
        variational_depths=[1,2], iterations=20, training_batch=16, vqe_qubits=[4,6], gradient_qubits=[4,8],
        warmups=3, inner_repetitions=7, qaoa_depth=2),
    "paper": dict(COMMON, seeds=[42], repetitions=1,
        qml_qubits=[4,8,12], validation_qubits=[4,8,12], state_qubits=[4,8,12,16,20], map_qubits=[4,8,12,16,20],
        qkpca_datasets=CLASSIFICATION, vqc_datasets=CLASSIFICATION, vqr_datasets=REGRESSION,
        feature_selection_datasets=["breast_cancer","wine"], variational_depths=[1,2,3],
        iterations=80, training_batch=24, vqe_qubits=[4,6,10], gradient_qubits=[4,8,12],
        warmups=4, inner_repetitions=9, qaoa_depth=2, timeout_s=7200, batch_sizes=[1,8,32,64]),
}
if RUN_PROFILE not in PROFILES: raise ValueError("Unknown RUN_PROFILE")
PROFILE = json.loads(json.dumps(PROFILES[RUN_PROFILE]))
for name, spec in CUSTOM_DATASETS.items():
    PROFILE[spec["kind"] + "_datasets"].append(name)
if sys.version_info < (3,11): raise RuntimeError("Use a Python 3.11+ Colab runtime for these pinned dependencies")
assert all(1 <= n <= (os.cpu_count() or 1) for n in THREADS)
default_root = Path("/content/superfermion_extended") if Path("/content").exists() else Path.cwd()/"superfermion_extended"
ROOT = Path(os.environ.get("SUPERFERMION_BUILD_ROOT", str(default_root))).resolve()
EXPERIMENT_ROOT = ROOT/"studies"/(RUN_PROFILE+"_split80_20_grid24_v8")
ROOT.mkdir(parents=True, exist_ok=True); EXPERIMENT_ROOT.mkdir(parents=True, exist_ok=True)
(ROOT/"tmp").mkdir(exist_ok=True); (ROOT/"pip_cache").mkdir(exist_ok=True)
os.environ["TMPDIR"] = os.environ["TMP"] = os.environ["TEMP"] = str(ROOT/"tmp")
os.environ["PIP_CACHE_DIR"] = str(ROOT/"pip_cache")
os.environ["MPLCONFIGDIR"] = str(ROOT/"matplotlib-cache")
Path(os.environ["MPLCONFIGDIR"]).mkdir(exist_ok=True)
FRAMEWORKS = ["superfermion_wheel","superfermion_source_baseline","superfermion_optimized",
              "qiskit","pennylane","cirq","cirq_f64","qulacs"]
CONF = dict(root=str(EXPERIMENT_ROOT), build_root=str(ROOT), profile_name=RUN_PROFILE, profile=PROFILE,
    threads=THREADS, frameworks=FRAMEWORKS, custom_datasets=CUSTOM_DATASETS,
    max_jobs=MAX_JOBS, max_statebank_bytes=MAX_STATEBANK_MIB*2**20, fail_fast=False,
    kernel_row_caps=PROFILE.get("kernel_row_caps",{}),
    order_qubits_first=True, report_refresh_seconds=900)
print(json.dumps({"profile":RUN_PROFILE,"datasets":PROFILE["classification_datasets"]+PROFILE["regression_datasets"],
    "train_ratio":PROFILE["train_ratio"],"test_ratio":PROFILE["test_ratio"],
    "dataset_rows":"all available; no sample caps",
    "fresh_process_repetitions":PROFILE["repetitions"],"split_seeds":PROFILE["seeds"],
    "qml_qubits":PROFILE["qml_qubits"],"vqc_vqr_qubits":PROFILE.get("variational_qubits",PROFILE["qml_qubits"]),
    "vqc_vqr_iterations":PROFILE.get("variational_iterations",PROFILE["iterations"]),"statevector_qubits":PROFILE["state_qubits"],
    "simulator_thread_budgets":THREADS,"frameworks":FRAMEWORKS},indent=2))

{
  "profile": "requested_grid",
  "datasets": [
    "breast_cancer",
    "iris",
    "wine",
    "digits_01",
    "moons",
    "diabetes",
    "friedman1"
  ],
  "train_ratio": 0.8,
  "test_ratio": 0.2,
  "dataset_rows": "all available; no sample caps",
  "fresh_process_repetitions": 1,
  "split_seeds": [
    42
  ],
  "qml_qubits": [
    4,
    8,
    12,
    16,
    20
  ],
  "vqc_vqr_qubits": [
    4,
    8,
    12
  ],
  "vqc_vqr_iterations": 5,
  "statevector_qubits": [
    4,
    8,
    12,
    16,
    20
  ],
  "simulator_thread_budgets": [
    1
  ],
  "frameworks": [
    "superfermion_wheel",
    "superfermion_source_baseline",
    "superfermion_optimized",
    "qiskit",
    "pennylane",
    "cirq",
    "cirq_f64",
    "qulacs"
  ]
}


### 2. Install isolated, pinned dependencies

Workers use their own Python environment. This avoids conflicts with packages
preloaded in the Colab kernel. Timing never includes installation or imports.
A log is retained for each setup/build/test action, and progress appears
during long builds. Qulacs is a real installed compiled baseline, not a
placeholder or NumPy fallback. Any required install/import failure stops
the notebook.

The entire tested dependency set is embedded in `requirements.lock.txt`, rather
than only top-level framework pins. `venv_colab_fix_v2` is separate from the
previous environment. The dependency resolver and `pip check` must both pass;
individual package versions are then checked against the lock.

In [2]:
LOGS = EXPERIMENT_ROOT/"logs"; LOGS.mkdir(exist_ok=True)

"""Paste/run this cell after notebook setup and before the next campaign cell.

Displays completed job counts and task percentages from existing checkpoints.
It does not edit the experiment scripts, manifest, configuration or results.
"""
from pathlib import Path as _SFPath
import csv as _sf_csv
import html as _sf_html
import json as _sf_json
import math as _sf_math
import os as _sf_os
import re as _sf_re
import signal as _sf_signal
import subprocess as _sf_subprocess
import time as _sf_time

try:
    from IPython.display import HTML as _SFHTML, display as _sf_display, clear_output as _sf_clear_output
except ImportError:
    _SFHTML = None
    _sf_display = None
    _sf_clear_output = None

PROGRESS_REFRESH_SECONDS = 5.0


def _sf_duration(seconds):
    seconds = max(0, int(seconds))
    return f"{seconds//3600:02d}:{seconds%3600//60:02d}:{seconds%60:02d}"


class _SFJobProgress:
    def __init__(self, command, log_path, start_ns, cwd=None):
        self.log_path = _SFPath(log_path)
        self.start_ns = start_ns
        self.root = None
        self.manifest = None
        self.tasks = None
        self.task_order = []
        self.snapshot_path = None
        self.task_totals = {}
        self.total = None
        self.rows = {}
        self.seen_files = {}
        self.log_position = 0
        self.log_partial = ""
        self.reported_finished = 0
        parts = [str(part) for part in command]
        driver = next((i for i, part in enumerate(parts)
                       if _SFPath(part).name == "study_driver.py"), None)
        if driver is None or len(parts) <= driver+2 or parts[driver+2] != "run":
            return
        if len(parts) > driver+3:
            self.task_order = parts[driver+3].split(",")
            self.tasks = set(self.task_order)
        try:
            config_path = _SFPath(parts[driver+1])
            if not config_path.is_absolute():
                config_path = (_SFPath(cwd) if cwd else _SFPath.cwd())/config_path
            config = _sf_json.loads(config_path.read_text())
            self.root = _SFPath(config["root"])
            self.manifest = config.get("manifest_sha256")
            section = "_".join(self.task_order) if self.task_order else "all"
            self.snapshot_path = self.root/f"results_{section}.json"
            with (self.root/"planned_jobs.csv").open(newline="") as stream:
                for row in _sf_csv.DictReader(stream):
                    task = row["task"]
                    if self.tasks is None or task in self.tasks:
                        self.task_totals[task] = sum(int(float(value or 0))
                            for field, value in row.items() if field != "task")
            self.total = sum(self.task_totals.values())
        except (OSError, ValueError, KeyError, TypeError):
            # The driver's actual announced total remains available even if
            # a preview CSV has not been generated in an older notebook.
            pass

    @property
    def is_campaign(self):
        return self.root is not None

    def _read_log(self):
        try:
            with self.log_path.open(errors="replace") as stream:
                stream.seek(self.log_position)
                text = self.log_partial+stream.read()
                self.log_position = stream.tell()
        except OSError:
            return
        lines = text.split("\n")
        self.log_partial = lines.pop()
        for line in lines:
            announced = _sf_re.match(r"Executing (\d+) jobs\b", line)
            if announced:
                self.total = int(announced.group(1))
            finished = _sf_re.match(r"(\d+)/(\d+):\s", line)
            if finished:
                self.reported_finished = max(self.reported_finished, int(finished.group(1)))
                self.total = int(finished.group(2))

    def _accept(self, row, require_exit=False):
        if not isinstance(row, dict) or row.get("status") not in {"ok", "error"}:
            return
        if not self.manifest or row.get("manifest_sha256") != self.manifest:
            return
        if row.get("task") in {"validation", "preflight"}:
            return
        if self.tasks is not None and row.get("task") not in self.tasks:
            return
        if require_exit and "worker_returncode" not in row:
            return  # The controller has not yet confirmed the worker's exit.
        job_id = row.get("job_id") or row.get("cache_key")
        if not job_id:
            return
        self.rows[job_id] = row

    def _read_json(self, path, require_current_run=True):
        try:
            stat = path.stat()
            signature = (stat.st_mtime_ns, stat.st_size)
            if require_current_run and stat.st_mtime_ns < self.start_ns:
                return None
            if self.seen_files.get(str(path)) == signature:
                return None
            value = _sf_json.loads(path.read_text())
            self.seen_files[str(path)] = signature
            return value
        except (OSError, ValueError):
            return None  # A checkpoint may be between write and completion.

    def update(self):
        self._read_log()
        if not self.is_campaign:
            return
        if self.snapshot_path is not None:
            value = self._read_json(self.snapshot_path)
            if isinstance(value, list):
                for row in value:
                    self._accept(row)
        raw = self.root/"raw"
        if raw.is_dir():
            for path in raw.glob("*.json"):
                if path.name.endswith(".config.json"):
                    continue
                value = self._read_json(path)
                if value is not None:
                    self._accept(value, require_exit=True)

    def counts(self):
        successful = sum(row["status"] == "ok" for row in self.rows.values())
        failed = sum(row["status"] == "error" for row in self.rows.values())
        finished = max(len(self.rows), self.reported_finished)
        return dict(total=self.total, finished=finished, successful=successful,
                    failed=failed, unknown_outcomes=finished-successful-failed)

    def task_counts(self):
        totals_valid = bool(self.task_totals) and sum(self.task_totals.values()) == self.total
        names = sorted(set(self.task_totals) | {row["task"] for row in self.rows.values()})
        result = []
        for name in names:
            rows = [row for row in self.rows.values() if row["task"] == name]
            total = self.task_totals.get(name) if totals_valid else None
            result.append(dict(task=name, total=total, finished=len(rows),
                successful=sum(row["status"] == "ok" for row in rows),
                failed=sum(row["status"] == "error" for row in rows)))
        return result

    @staticmethod
    def _percent(finished, total):
        return f"{100*finished/total:.1f}%" if total and finished <= total else "—"

    def text(self, name, phase):
        c = self.counts()
        if not self.is_campaign:
            return f"{name}: {phase}; detailed output: {self.log_path.name}"
        total = c["total"] if c["total"] is not None else "?"
        return (f"{name}: {c['finished']}/{total} jobs finished "
                f"({self._percent(c['finished'],c['total'])}); "
                f"{c['successful']} successful, {c['failed']} failed; {phase}")

    def html(self, name, phase, elapsed):
        escape = _sf_html.escape
        title = escape(str(name).replace("_", " ").title())
        parts = [f'<div style="font-family:system-ui,sans-serif;max-width:1100px;'
                 f'border:1px solid #d0d7de;border-radius:8px;padding:16px;margin:10px 0">'
                 f'<div style="font-size:18px;font-weight:650">{title}</div>'
                 f'<div style="color:#475569;margin:5px 0 12px">'
                 f'{escape(phase)} · Elapsed {_sf_duration(elapsed)}</div>']
        if self.is_campaign:
            c = self.counts()
            total = c["total"] if c["total"] is not None else "?"
            percentage = self._percent(c["finished"],c["total"])
            parts.append(f'<div style="font-size:20px"><b>{c["finished"]:,} / '
                         f'{total:,} jobs finished · {percentage}</b></div>' if isinstance(total,int)
                         else f'<div><b>{c["finished"]} / ? jobs finished</b></div>')
            if c["total"] and c["finished"] <= c["total"]:
                value = 100*c["finished"]/c["total"]
                parts.append(f'<progress aria-label="Finished jobs" value="{value}" '
                             'max="100" style="width:100%;height:20px;margin:8px 0"></progress>')
            remaining = max(0,c["total"]-c["finished"]) if c["total"] is not None else "?"
            parts.append(f'<div style="margin:4px 0 12px">'
                         f'<b>Successful:</b> {c["successful"]:,} '
                         f'({self._percent(c["successful"],c["total"])}) · '
                         f'<b>Failed:</b> {c["failed"]:,} · <b>Remaining:</b> {remaining}</div>')
            if c["unknown_outcomes"]:
                parts.append(f'<div>{c["unknown_outcomes"]} finished-job outcomes are '
                             'awaiting a readable checkpoint.</div>')
            if c["total"] is not None and c["finished"]>c["total"]:
                parts.append('<div>Checkpoint counts exceed the announced plan; percentage withheld.</div>')
            tasks = self.task_counts()
            if tasks:
                parts.append('<div style="overflow-x:auto"><table style="border-collapse:collapse;'
                             'width:100%;font-size:14px;text-align:left"><thead><tr>')
                for label in ["Task", "Finished / planned", "Complete", "Successful", "Failed", "Remaining"]:
                    parts.append(f'<th style="padding:8px;border-bottom:2px solid #cbd5e1">{label}</th>')
                parts.append('</tr></thead><tbody>')
                for row in tasks:
                    total = row["total"]
                    remaining = max(0,total-row["finished"]) if total is not None else "?"
                    cells = [escape(row["task"].upper()),
                             f'{row["finished"]} / {total if total is not None else "?"}',
                             self._percent(row["finished"],total),
                             row["successful"],row["failed"],remaining]
                    parts.append('<tr>'+''.join(f'<td style="padding:8px;border-bottom:1px solid #e2e8f0">'
                                               f'{value}</td>' for value in cells)+'</tr>')
                parts.append('</tbody></table></div>')
            parts.append('<div style="font-size:12px;color:#475569;margin-top:10px">'
                         'Percentages count finished jobs, including failures. They are not time estimates. '
                         'Setup, correctness checks and report rendering are outside this campaign’s job total.</div>')
        else:
            parts.append('<div>This preparation/check/report command has no benchmark-job denominator.</div>')
        parts.append(f'<div style="font-size:12px;color:#475569;margin-top:8px">'
                     f'Detailed log: {escape(self.log_path.name)}</div></div>')
        return ''.join(parts)


def _sf_stop_process(process):
    if process.poll() is not None:
        return
    try:
        if _sf_os.name == "posix":
            _sf_os.killpg(process.pid, _sf_signal.SIGTERM)
        else:
            process.terminate()
    except ProcessLookupError:
        return
    try:
        process.wait(timeout=5)
    except _sf_subprocess.TimeoutExpired:
        if _sf_os.name == "posix":
            try:
                _sf_os.killpg(process.pid, _sf_signal.SIGKILL)
            except ProcessLookupError:
                pass
        else:
            process.kill()
        process.wait()


def run_logged(command, name, cwd=None, env=None, timeout=3600):
    path = _SFPath(LOGS)/f"{name}.log"
    path.parent.mkdir(parents=True, exist_ok=True)
    print(f"Starting {name}; detailed output: {path.name}", flush=True)
    start = _sf_time.monotonic()
    tracker = _SFJobProgress(command, path, _sf_time.time_ns(), cwd)
    worker_env = _sf_os.environ.copy() if env is None else dict(env)
    worker_env["MPLBACKEND"] = "Agg"
    refresh = float(PROGRESS_REFRESH_SECONDS)
    refresh = min(30., max(.2, refresh)) if _sf_math.isfinite(refresh) else 5.
    last_text = None

    def render(phase):
        nonlocal last_text
        tracker.update()
        elapsed = _sf_time.monotonic()-start
        counts = tracker.counts()
        if phase=="Running" and tracker.is_campaign and counts["total"] and counts["finished"]==counts["total"]:
            phase = "Jobs finished; waiting for the controller to exit"
        if _sf_display is not None:
            markup = tracker.html(name, phase, elapsed)
            item = _SFHTML(markup) if _SFHTML is not None else markup
            try:
                # clear_output works in Colab as well as classic Jupyter;
                # it avoids depending on frontend display-id update support.
                if _sf_clear_output is not None:
                    _sf_clear_output(wait=True)
                _sf_display(item)
                return
            except Exception:
                pass  # Presentation cannot stop an otherwise valid benchmark.
        text = tracker.text(name, phase)
        if text != last_text:
            print(text, flush=True)
            last_text = text

    with path.open("w") as log:
        process = _sf_subprocess.Popen([str(part) for part in command], cwd=cwd, env=worker_env,
            stdout=log, stderr=_sf_subprocess.STDOUT, start_new_session=(_sf_os.name=="posix"))
        try:
            render("Running")
            while True:
                remaining = float(timeout)-(_sf_time.monotonic()-start)
                if remaining <= 0:
                    _sf_stop_process(process)
                    render("Stopped: command timeout")
                    raise RuntimeError(f"{name} timed out; inspect {path}")
                try:
                    status = process.wait(timeout=min(refresh,remaining))
                    break
                except _sf_subprocess.TimeoutExpired:
                    render("Running")
        except BaseException:
            _sf_stop_process(process)
            render("Stopped; completed checkpoints retained")
            raise
    render("Campaign finished" if status==0 and tracker.is_campaign else
           "Finished" if status==0 else "Stopped with an error")
    if status:
        print(path.read_text(errors="replace")[-6000:])
        if name in {"backend_preflight","independent_framework_checks"}:
            filename = "backend_preflight_errors.json" if name=="backend_preflight" else "validation_errors.json"
            error_path = _SFPath(EXPERIMENT_ROOT)/filename
            if error_path.exists():
                errors = _sf_json.loads(error_path.read_text())
                seen = set()
                for error in errors:
                    key = (error.get("framework"),error.get("error"))
                    if key in seen:
                        continue
                    seen.add(key)
                    print("\nROOT ERROR:",error.get("framework"),error.get("n_qubits"),"qubits")
                    print(error.get("traceback",error.get("log_tail",error.get("error","unknown")))[-5000:])
                    print("Worker log:",error.get("worker_log","unavailable"))
        raise RuntimeError(f"{name} failed ({status}); full log: {path}")
    print(f"Finished {name} in {_sf_time.monotonic()-start:.1f} seconds", flush=True)
    return path


# Keep the existing Agg wrapper compatible when setup or this cell is rerun.
_sf_run_logged_original = run_logged

ENV_DIR = ROOT/"venv"  # Prebuilt environment on D:; reused, never rebuilt.
PYTHON = str(ENV_DIR/("Scripts/python.exe" if os.name=="nt" else "bin/python"))
if not Path(PYTHON).exists():
    venv.EnvBuilder(with_pip=False).create(ENV_DIR)
# Full tested dependency lock, including SymPy/Autoray and native BLAS support.
DEPENDENCIES = ['absl-py==2.5.0', 'appdirs==1.4.4', 'attrs==26.1.0', 'autograd==1.9.1', 'autoray==0.8.4', 'cachetools==7.2.0', 'certifi==2026.7.22', 'charset-normalizer==3.5.2', 'cirq-core==1.6.1', 'cloudpickle==3.1.2', 'contourpy==1.4.0', 'cycler==0.12.1', 'diastatic-malt==2.15.3', 'dill==0.4.1', 'duet==0.2.9', 'fastjsonschema==2.22.2', 'fonttools==4.66.1', 'gast==0.7.0', 'idna==3.20', 'iniconfig==2.3.0', 'joblib==1.6.0', 'jsonschema==4.26.0', 'jsonschema-specifications==2025.9.1', 'jupyter_core==5.9.1', 'kiwisolver==1.5.1', 'matplotlib==3.10.8', 'maturin==1.9.4', 'mpmath==1.3.0', 'nbformat==5.10.4', 'networkx==3.7', 'numpy==2.3.5', 'packaging==26.3', 'pandas==2.3.2', 'pennylane==0.45.0', 'pennylane_lightning==0.45.0', 'pillow==12.3.0', 'pip==25.2', 'platformdirs==4.12.2', 'pluggy==1.6.0', 'psutil==7.2.2', 'pybind11==3.1.0', 'pybind11-global==3.1.0', 'Pygments==2.21.0', 'pyparsing==3.3.3', 'pytest==8.4.2', 'pytest-timeout==2.4.0', 'python-dateutil==2.9.0.post0', 'pytz==2026.4', 'qiskit==2.1.2', 'qiskit-aer==0.17.2', 'qsimcirq==0.22.0', 'qulacs==0.6.14', 'referencing==0.37.0', 'requests==2.34.2', 'rpds-py==2026.6.3', 'rustworkx==0.18.1', 'scikit-learn==1.7.1', 'scipy==1.17.1', 'scipy-openblas32==0.3.34.237.0', 'setuptools==84.0.0', 'six==1.17.0', 'sortedcontainers==2.4.0', 'stevedore==5.9.1', 'sympy==1.14.0', 'termcolor==3.3.0', 'threadpoolctl==3.7.0', 'tomlkit==0.15.1', 'tqdm==4.67.1', 'traitlets==5.16.1', 'typing_extensions==4.16.0', 'tzdata==2026.4', 'urllib3==2.8.0']
LOCK_PATH = EXPERIMENT_ROOT/"requirements.lock.txt"
LOCK_PATH.write_text("\n".join(DEPENDENCIES)+"\n")
(EXPERIMENT_ROOT/"requirements.txt").write_text(LOCK_PATH.read_text())
install_env = os.environ.copy()
for key in ["PYTHONPATH","PYTHONHOME","PYTHONSTARTUP","PYTHONWARNINGS"]:
    install_env.pop(key,None)
install_env["PYTHONNOUSERSITE"] = "1"
run_logged([sys.executable,"-I","-m","pip","--python",PYTHON,"install",
            "--upgrade","--requirement",LOCK_PATH],"dependencies",env=install_env,timeout=2400)
run_logged([PYTHON,"-I","-m","pip","check"],"dependency_integrity",env=install_env,timeout=600)
observed = json.loads(subprocess.check_output([PYTHON,"-I","-m","pip","list","--format=json"],env=install_env,text=True))
normalized = lambda name:name.lower().replace("_","-").replace(".","-")
observed = {normalized(item["name"]):item["version"] for item in observed}
for pin in DEPENDENCIES:
    name,version=pin.split("==",1)
    if observed.get(normalized(name)) != version:
        raise RuntimeError(f"Dependency lock mismatch: {pin}")
print("Full dependency lock verified:",len(DEPENDENCIES),"packages")
CONF["python"] = PYTHON
print("Workers use", PYTHON)

Finished dependency_integrity in 0.6 seconds
Full dependency lock verified: 72 packages
Workers use /content/superfermion_extended/venv_colab_fix_v2/bin/python


### 3. Build with the same Rust toolchain and CPU flags

Official Rust 1.90.0 component archives are checksum-verified. Standalone
installation works in environments without `/proc/self/exe`; in that case
both source lanes use the same system-linker override. Otherwise repository
portable CPU flags apply. Separate Cargo target directories prevent one
build from silently reusing the other build's native extension.

In [3]:
TOOLCHAIN = ROOT/"toolchain"; TOOLCHAIN.mkdir(exist_ok=True)
BUILD_ENV = os.environ.copy()
RUST_VERSION = "1.90.0"; RUST_HOST = "x86_64-pc-windows-gnu"
RUSTUP_HOME = TOOLCHAIN/"rustup"; CARGO_HOME = TOOLCHAIN/"cargo"
BUILD_ENV.update(RUSTUP_HOME=str(RUSTUP_HOME), CARGO_HOME=str(CARGO_HOME),
                 CARGO_TARGET_DIR=str(ROOT/"cargo_target"/"source_optimized"), CARGO_BUILD_JOBS="2",
                 TMPDIR=str(ROOT/"tmp"), TMP=str(ROOT/"tmp"), TEMP=str(ROOT/"tmp"))
MSYS_BIN = Path("C:/msys64/mingw64/bin")
if not (MSYS_BIN/"gcc.exe").exists():
    raise RuntimeError("mingw-w64 gcc is required for the pinned x86_64-pc-windows-gnu Rust target; install MSYS2 mingw64 or adjust MSYS_BIN")
BUILD_ENV["PATH"] = os.pathsep.join([str(CARGO_HOME/"bin"), str(MSYS_BIN), BUILD_ENV.get("PATH", "")])
for variable in ["RUSTFLAGS", "CARGO_ENCODED_RUSTFLAGS", "CARGO_PROFILE_RELEASE_OPT_LEVEL",
                 "CARGO_PROFILE_RELEASE_LTO", "CARGO_PROFILE_RELEASE_CODEGEN_UNITS"]:
    BUILD_ENV.pop(variable, None)
machine = platform.machine()
machine = {"AMD64": "x86_64", "ARM64": "aarch64"}.get(machine, machine.lower())
if machine != "x86_64": raise RuntimeError(f"unsupported architecture for the pinned Rust target: {machine}")
# Provision rustup and the pinned 1.90.0 GNU toolchain entirely inside the study
# root on D:; nothing is written to the system drive. The repository .cargo
# config declares flags only for linux/darwin/msvc hosts, so the documented
# portable CPU baseline is applied explicitly for this host via RUSTFLAGS.
rustup_exe = CARGO_HOME/"bin"/"rustup.exe"
if not rustup_exe.exists():
    source_rustup = shutil.which("rustup")
    if source_rustup:
        bootstrap = TOOLCHAIN/"rustup.exe"
        if not bootstrap.exists(): shutil.copy2(source_rustup, bootstrap)
        # rustup refuses to finish "toolchain install" unless the rustup binary
        # itself is present in CARGO_HOME/bin; the shims are rustup copies.
        (CARGO_HOME/"bin").mkdir(parents=True, exist_ok=True)
        for shim in ["rustup.exe", "cargo.exe", "rustc.exe"]:
            target = CARGO_HOME/"bin"/shim
            if not target.exists(): shutil.copy2(bootstrap, target)
        run_logged([str(bootstrap), "toolchain", "install", f"{RUST_VERSION}-{RUST_HOST}", "--profile", "minimal"],
                   "rust_toolchain_install", env=BUILD_ENV, timeout=2400)
    else:
        initializer = TOOLCHAIN/"rustup-init.exe"
        if not initializer.exists():
            run_logged(["powershell", "-NoProfile", "-Command",
                        "Invoke-WebRequest -UseBasicParsing -Uri 'https://static.rust-lang.org/rustup/dist/x86_64-pc-windows-gnu/rustup-init.exe'"
                        f" -OutFile '{initializer}'"],
                       "download_rustup", timeout=1200)
        run_logged([str(initializer), "-y", "--no-modify-path", "--default-host", RUST_HOST,
                    "--default-toolchain", f"{RUST_VERSION}-{RUST_HOST}", "--profile", "minimal"],
                   "rust_toolchain_install", env=BUILD_ENV, timeout=2400)
if not rustup_exe.exists(): raise RuntimeError("rustup provisioning failed; inspect rust_toolchain_install.log")
# pin the runner: a surprise rustup self-update mid-experiment can lock shims
# (seen on Windows as a transient 'Access is denied' on first proxy use).
run_logged([str(rustup_exe), "set", "auto-self-update", "disable"],
           "rustup_disable_self_update", env=BUILD_ENV, timeout=120)
BUILD_ENV["RUSTUP_TOOLCHAIN"] = f"{RUST_VERSION}-{RUST_HOST}"
BUILD_ENV["RUSTFLAGS"] = "-C target-cpu=x86-64-v2"
CARGO = str(CARGO_HOME/"bin"/"cargo.exe")
print(subprocess.check_output([CARGO, "--version"], env=BUILD_ENV, text=True).strip())
print(subprocess.check_output([str(CARGO_HOME/"bin"/"rustc.exe"), "--version"], env=BUILD_ENV, text=True).strip())

cargo 1.90.0 (840b83a10 2025-07-30)


### 4. Materialize and verify the library patch

The source SHA and every changed-file hash are pinned. The readable patch,
changed source files, and manifests are exported with the results. No GitHub
push or commit is made. The unchanged source checkout must be clean, and
the modified checkout must match the embedded patch exactly.

In [4]:
BASE_COMMIT = '1ef0826a740b972f306680350983df4a91d05ead'
PATCH_SHA256 = '4887015d82d22f34f82f988c0f3e0a0158698439b6f908b2c19a3a4f0cb1a325'
PATCH_PAYLOAD = 'c-rlKYggOIw%~VuMTK5NOIQ}z5Rx{5)9G}&J2QE3L;B3^!{R8ymO;Umgd_tF;djmV`DL>oRZ^86#t_a~cg?zF1#GFb>$TsHQn%X`{>Cu%^B~*E#@&-J8imm*d`^cOlkmh%vz1P#BTnvU^YhQ5+k5Wphz|UIE`I)bMf~vO$e&E&D{D9jlVm;^L_si`1X<>#esmgmlQ>T7;}x+YW?3Mzd^8wj^Jq92e29j@+jw}syV6<dOyiMA(jf8EV6>|mN3$scyg8V^OjCdUF38W~QRaw)`JYifPlAzt`F58dO6Kv_U;wZtvr#Y@tnc!mcXnXBo!-uK=NSz<oF)?hbeO|*#m$PqpK&CD>m(S$wJ?sXwJey79RN6hhR{d!;zw`a5%EbDq?i85Bp8Ul2g4T@{+d09JL8>g`epCiqPq_deGIbMB!2;y_EjJFGYN8$2kA5ud*TrKfZ_US7I-COYpwij@2+&!lQ<$bjG`bdui=ktjGJKFD;;$=nY*+_UVfGa{wTA)ebXQ{J!`KB1N_uaydX`jZ^1V)rS-<IacU)VM+9yjds&_WvRJly^cFS|7rx)$M0Dx*drto;t$f2=%+8Wo8s>BByT;JrF?9ID#>R$tIm~DNL|npD<5Wa(Iu+T)jPOm&qDgoTs6!n)B0mc=F+2-~=NUEhC*fCG9+$xZp5SFqKHrQejVRF3Q6MAS+bi%M2m`P12rUKE{@ZT(*oSkgaWnBbcjV{38^Q)!HZGR+?dv!3)s-WKi?*B1rY)=UF7P7}kHtSk3`B4lq;qjGI|*|~4C84s%Y%`)3SnxqT=?SsY<e(9Qi;>NT#_k(@V-7|G0FvyFp9ZXMhd-*LjN?5{E4SyZ3&8d*HPaf-tvY{#cSmN4(7kg4n7nO<WtMeL5)z(qU?xgR8WUvIpc5?Oc3GQ_)UHmq%cUytQv<wl(xN1oVmZfqj)p^OkIC8@`e*X%LaoN2lL+n#$j+V3xNIZTZmna&E6dxQg;iQX!kqy`}&$5$j`6BbT|w1rJkwpgB%JE25=VoKM8>}gH%|rCgFG-rz4St)7gZuLUhD|KbwRizY5cd-8u&e;}dq_&$yE=$jgmUaJdo9CX>aIQH@!N$4zGw<c!nXCT`c+*mxp7BzZTC3KS8q4*uMLjROh_hBM+T;uk+3o{1q)b|#<=Fvm0mZsiZtI0Ih9>z3^@JU|F|Hq3GugwS|~I1kb&00_f4%F}p4BpXIu`MCJ#&tX|$4A2drTsC!)unMs}jN$SS#{+Bgho7Cj^OIfGLds%O6z}rAy2xdqV=qjB9{f{yPR{+X9Ie@krZE_3_vvHNKdnCK0y^hMP2fLfew5FqzrOt4QR4iT9{S-ZnbSU7kv~N?yIRMnRS?%{{56O`LFpv36Ih3IHq1p6_9<eAW>z*!g47mQL3nzWXRHPwB?70oReke7GXN}AjdqYf97Vy^aVcAZE2{?DBzx2v4E7U_)o9!>^MYxD0#tx#c%m*(f(YaS+)U!DAob$0mtVz}rX}DHJWzmPLE=Yq>)R#3xx{D3VFcGLTYK!rgpdaLERDqL6y-9aMQ}X~5{Xm?^S}F(S@0S+4;95L66huX{8?2zRE<VVXIU;z0>J|iel!wi{v~_>NgM(7#)78RjmKS>7KqpAG(TH)OwhKiiPBp`s^Wr*o1(khs^t0M=rqV-MUFK=G!3Gx&=YBYW*Kl207I&cvG54#wizue%&{FOD-G4klL`eQqfTK9(Xj<u11=%~qxdOF?G9u_<%UKD(`y(O?Yy-HU|IdE;9#NXEr~x>1xaot4`7Lnj6BkGvbKF>C0Vsv1PM7StTRv-?yU|GsrZnz(|r?c<q>x{iKBqgr2s2MTF|B-T;iGltB@>Pc0`s)g>5LAUEM6y98nQnY$ufC;2!Nq-Z(irMvq_OYyMD`2z(Vs>@@9<BUXpWY}Ia7b3Zb7^2G>fHxOxj)t&lZKytD!e&R>xu6XT(DAx1{GzQ`%1i>AH&IE{+$0bCR#rPOn4M8MTV}BGUfGYW!pVQ4<;8KD{?~)va4W^Ao&}c4$_E7FNHp{#dR0v-Dn9TQ=)NY}rc1Pg^B3=_->^Ca4zr?d>grJ;?yzjq&?;e^0lK}SR6@NfDHC%!<MZ1;<z|3#Af^Fdx8uXaRUc6}9#mW=GBmiccBU}XUl5G}@JlG-&pxJJG?!~DGs!xD8TM<c3==d&}#_7Bwh*oLQOU^E0FoecH5UU!d-x_F(qdS?yhQZL(D2?Iub{=_j*qflGyL)t?74YEpwdLO0yr;}=98H7J)-e|`dlF7VG#v9-^u+JsFS{a3z|bL*2_rBniIhY(MRhb4^jwfFE41(6TY%OQsgVR)u-6d<Q#LzA;L};r0e~0`P?`3Ie&P?&SW}y=kB%r9m`TCZGnk=}+!FXc8R)@v9z-LniLaK=*?+Tl^~p*Oi2#hv{w|hll6Y@00KH@C0jnjC3iaitL3ZXRfpvs*=OAeuHyGt?w*MF7Sv}I5_p54hlT6>+6Pu5q$gc7_z@rzl5LQkA`v;p6G*{&H)Ys}FbWUkT*-{uiUIu+jfDIU|bLe|uah>pWUskiUz6;qGO7h)$k8)maGRw{^c793Ek!`pMO|RhJs-WrwCILSljRDNXWTtvt<m&o2(N)bYwzM<szBE|&Qn8Yr+@Tjd29XxFWhv)Jae$s)AR2bVNK5R$Y4u-2tBpR~wK`J#e?ZJc&zj3fDfqnXFMiDSc};-Ye$12uYq9qn8&<cjz$#u~-W)&k(0gOMqxdQsf*A7BU_2PSs6A{gP*H1<memOM>k@1y!1b8$RtY=96-kg+UDWUjcM%%n4GW^2{&F3bS_IwwuH&>3K~Z3eO>B>yDDh$6GtAU5oznkbozUIP_1`+@rO8mEpnES*w-R{{Q}UXl^QR%^Y~Dq8`tyF@>1~P*{@p2p8fbZAAOjh88V&<?nkQ+{Ey7h~tDgkG9RspBG=m;P(qNU8p!~}?90}OK?kNKO^&LXJ7PxLhegqwkaP5>`^e#V&-np88rttT)*c`Bf-3Dv85O}2t+^vcTwc)VWPmY?wGY{W%Hyr`Lz=%MNT@N55h@;J8fU*ccuK=*pHRIOSc$Nht2gFPUn1R@bgFcyslaU7t;UyEFEPhcw1`GUwm@LR#ye)<xc4rw0u6!0@5W_nB%{zGe^1b)#pYIO#dJYC8ko&rf13x`?c6I;<e%hANdz10&AIqs?x1e@w;SMnhh9L=j*lOCz0igyCDgu;I;gj4y2MaY9er7=POEF8+2WrBQDInd&EJ)`#4}>P8EO9)cZm=afa4CFjY$+_H2M}Qm5eN2FF2KRdk1yZ8ef`$^^vB27hktx{`>R-66DdL+42H8b1+&w`%~J$zqj`@Tr4RP4|8*`hq%2TqaHWA)qjwzU3UmPj1B7DK&aQ$W$%KX1y3-&%#R%2=51$Nh=_~?qEIy&PYc?L^yfT_&Ky)K`EJeJE;#gS>Ym)?ka3cd&Nr_}35pkj~ob<j3gU`68lK27<(^vBGLsJ7%Z|DF1pY4A}UGj;H0t|=|++Cpl$($gM!xZc<91!;9i&KzY5qFjNb8~~SJ@(VZAcNaLf4rQ7h2$;uH>4ul*c99ZL<Z8(h<tQbK(dxSC)lQP2Y(L3=^MB;7!c#T5#i`s+={*Zrnc0cu*px7$=pM>=7kY$zO`2H643vVnvd22)-jU&n*JTgP@<?^@8NdA08y~@V1yTr7y-~xZ{aw3geZ)9jYC`qFzX0see_);>f?w2!bKm3wB(5TEhFmTFNE{Z0UezAS@3Zd!I+38-aVWIpYVVVjiDGK!y#A?9nx>O^=)Ig$eiXIvzF^1i)jVhZpWaGw%n(+;7JoY%Got;1&j-ZQ1+$)E|$Srv=77xm)!|;A2AOo?N#L6EzJ$yfw8dHAK!V0uRgx~^y&{64Ls}H@Egb8r@?eEK)J)STOYb7F{o8n9(+SpYPY=y9F!c+i@QccD!}(?5cS<ct3e;d{xYM$j4UznLE5BBqD+D@I@3|krom+xgNk58VeSN&286=6vI3n290eA`i9b!?`iVb0M`!eB7~bc>ozO@wUHJ(pk)S?v1o|9ACEp-bEV%ZE`DA`4sQy2rGKMjUqf_^eEpkC+h*0`bcw^D(x8G92A5^p!0JjVI4NwNN8%b(wnh30NnwAbq)W(H02v>*J^mt`#+Sl8PrXyo|xGNSRRF>Pt<Rxn5q&p}r1@#tf2b#&Ug1-eFPvRVw)TGbK_>mT%SwWO0&uAcwbqTeJ6|}68z96QaK?_`3Y@B2ic$$^SN_s_ftVzY5Rg-F-8s!Lvqau~a8qFhr8V=P3TO-v0SyI^<Oes(OFVy86Rl$irXN7+IdGDFCB|6*Bo^Ct6XFWrwKldnp_QV4Dy5)#%M?7`Jjw7Bq;<+P!azqc!5;Z(Bz$tOJWtzMaYMuj*?Mp8vO&M(QBWq12H+)DO0d~r1aFY6<Hn1Vsrt8_qj!MDE5cJxSlq-0B2|KXU77?w4LBHSr>paZ%>pYqr)p<M{s?%yel?f^jx1x$<fyc0*I*(*MbsoiX>O7v+)M>VuI^St6sn~~RDJes#(`Y1hmNbxRUTGaw9?3*n<c+@DH0nH(S+w*h8qJ~3<CsEA4Wh*a>fFctX|;To8iBHSI!)G2r@_+cv|Bly1r|={Uu)fT?q}I_mbYp;|5A&lv!pfCd0b1Tb3ZGl^OzP)=h3W}&f{1vo%>lWouw_7=2dRKY*;FtB_4W^h0=Mrb<)O7Zfhxq54J`+i!71Ol2%A(Sqr3df9s<*=~7-xi-VQ{D-<@wn<*+snBuN#9hBjaMSeo2!I-oYR&YwC5HyUDRu{q0KE+8qiBCh&M?j|;1sGqVh|CZap&SIcIIYEkhCWykBNr1gp*=jwpvpGzIPhOmVgXV3p)|li1}HpZ)K6uZhl8q^ADvEu?g+*kjes5)Se)eHRF;Nih5p6$6h6#dM(xG>Sb(B|>KKRigvz<b%u>t)#q>`-)w`ZEtyLei*fMyhl1)6&^NwY+7^sE^%y$dBM>QCS*O<B}*NGc(_~ymZhS7de3Bk$ci|1pcnOBm`wG{Kl3<^MW3>FsoAz%g2q6`MVy!mtpmWRsaZp|B6)uYz+)K3YZh-9~Cgdn+;t6SA<t1ZoCZfiZJQpFMGOVT{^k`g?kggP~22XO)bDZ&0U^^>zB1W}cIn1++MapGYmE!K9NU=oj?%DR#A9%jdVNIt<QSX4hv!ys)Ei&%3MMzdgXS_bzDZAT#yIP{yv)1cZXUY`bjHcLtTRLfLksmhcyMxR*agHAD2z#LeKZ^idlAO+*8<vGHuCa&K!Wv!G5BI~wbjM*%~nn%D1Fn3_7io&d-I8qyQM)@EC{)Jtnnh)Y&#pJUlq-t0*DyP8O8^9RYS8bKbyc{g;p@N<IR51K+eM`QH#J%*>5V^9fPq4LGp;}gvfuEec1DjWj<3=3fzG;KWV{U#8z?4391H3N^SXSuJOFCi5h!b4-a94?r38!R*Iv1#@#5JynZL75!RXw^Kp|faV96h<Hq75{r2bAR+YpwMf4@_JpW`XVi3^0;+uQJK%>ai&;rPOJVQ%REDL=Lpq)X+u}rcZi37+Ds#PRoNH-PcHJj78P?6Cka^8T8`~c7FR~rP6EFwi}=lQni4bhCpYaAJjqDVcj<38cYyLj)u-FQj{~C2XiDF7~9DDhWEaSCrvAj94`06h^xG8dx88Z20n|<qxdRn9(cRhA+}wW<ti^jqcgoQJO`pjl>F9~j7S1RwNj*6qv29Qlx-C^K;_A!sbn&HvHrY~m?x`GYr-=1uXuJI&WdKM=JGX6n5bNnDCI1Ht2j)qc5R~NJvBRb7F~g`)YITvn47)SxXHMO5#wL4cj0BS=Gt0=8C3$e)o0o11+y*EYBUdMW+_*FD-7V44|MxiG8cb18jeq`9F3S`c}NGSMToI56HgU1d)T1hE~NirHXK(1t7189qm{}vfdTa!um&`OGy1Aaw<aS9*tjsAzNkKJfwix%1u}cA`HmB-Pz~nRfr334{9Ot~5LQTT1F?Fo1hXSTAgxW<W>}!3&Eq@ZqR*EaI--A9+mBx#GzJqcu0&B)en+jPLhzb?d5!YoXcK*z-A(ryjNRQvk4f*ii95*D;yqPu$QiKKk)@CL+CHvQJ3j045gg@-7xj|Dra=}}V3ejvGsOi(Tq&O`AT^~&TfQSsY@8<#L1!Qes=U@H{y=pLQsB|nx-Yt7{lunIHOR8=1>Ii%ZWn<LrD68PVv+&CdRdy9+R!qOH|)h29bxZZzK1QT!@a#HePS@LqM0vGnSfxuJE-CvL#|Vsn*1Fo-Y~z`6jV-vZBIfi?(_LU&z$X_@BpD_&X$>jc)Mce9>VI$BveeS1^iTspOOB1L}ycc3exE@>3rxTY0L~Ieod8QdM)L{Iw=#09l=L9P_}#Ra3b8pJkE($TivO8W+<g{LN`cav3|ZmV+@@~ycl+sghK?q>AEecB?>f!3F?r0Es%`~qqi>iFlw|XLL;PZ@RAF_&IH#L5SMrgPTZv{g_adW=yoD@)RqXdu>k}EGFV)QNqEV{V)!+QGqMihx-|*&d=hj6Fc^KKaKLU!AR&b?%sl`ClelO}%V51<&0$fdPH#anl={)u?VmD{Sq|Z0c`N;YRi9F~w?eVMubSIp4JXO<e+!=^i;l2sy%}-4y=MGs2s3<dSIhMDptav>U!Y}Bp%^r-#qI+~K&E7S18-4-+oe(-CI9JY%=IWdMw{&m(CfOIhIY5ax8wFa970Uunw>pn3d1-U103#e*GOS99&l5c*Ie=ZG(uy#Wtbl9($vq+r5VJhBCKf)$*5m4{VA&FrgIVwXH)M&y!eNAvA;n<6tt+(=uYBf2qsug{_`jXO)fs66O{6K*T<45t!TnE+#ve#V0Q_lTD>m1L6)o4)gUcZkE>j83oPtNNnvdieufg;G`UJiU6%sVA`Kn!wQh{+hqS>4TM;OkeRmT+hAyqP2jfK}h<mlghKLLsl+Sf*c0V;&`LME3oHDd_1aOb<vn{o{&5E9~>=qT(m@?c+?jP7MxKc(d3%C4T+tsuSsO0Mfg|H1@Ve0m`>bhUm7m^d;D2O4oSyDXT;DxcOa??ub7M~>fP~EIsonA%t)6R<2d=q!HtRF;|`6y}BU@=HngqQ#rCd$na`O2|Y9hlU1zczf*pEtLkW5!=^b8E+WW-8E1w`Ya>VjNDbwW{sVN2}fuiwq?rBM_MwpZ=!p11W2mU4}R|jML!9Ok8a4y*>N|o${Ap&d^zGn5KnHm12Sn3YRgQsH!qb*@BI8K$?eB%AS&WYnb?ms~G09QNRP}ay^{5@+apq%tuEBH863+TEvV(<R3fSkSKKa+(49~Q8SrRU<`@CJPoWQ&_zdqK})I67P%1Rb9w3mzmHBmO#G5uiFUr}dHoczHH<+3tS#A-#v$6cLm0vOgrh1OpinC9E-!A4E!>u=Xck4=CSl@m<UBl$_|u%q*Q`y6;pN}0!k#bneNyxn>G(QtOjuvSRlrY}X@vt2-}D^Mi`|OPS+-SAZQ(&tFxcgux`FASqR`vbk`iPS>boj73t!HFOwT>>5W&UD;u?+9HbqsbR-0_L2n==2%XN9STSE_Ee-&+*X26zR!`3TfknJ5<p#LGZ^(kzkD%oZ!YN3K&L%UMt*BbU48g`o?dTkIjfVY+GUz+Peqq)Rc+Aa<nH|}Z`6F`j{VdlXeY*tMTX8l<FlMqP{BlCy=>z2a<)k%p0#T9e!V4peGw3HWG7NUQz6Pg$@v3-y5#T$XjILtld+ihMhWY-v)FcMl}M_fGhi3lgG(M)CgVBJ`&6CK#aXXwa`&}~;<Uk9Yv!d1<zB|{H%xUosmt>*y`22@uLP2gHya}Ix{U}A9;(v4XEDmM102ZIw~+o4QUfcf{I?lk6I5UxpyCXdaQE4D4ic=1hG7C{0V)atKa*oZ@0EM=ZI&$;4TQcmJK2&Kup)Z1x0Gy`FOVaHJC;m)T{n@8y$0;a)}vnWlGnAexWO>*J1H)YR@-gvh(=?lleh2b;lEb<4m^33+f3g*zp9f4Vh+Xa)@Y8ojmg4&)geOLg(YYk_&<Rdm*D)6wgCt4j0bmPnM-RU1?<fPH~mv+B2`P*VBMHE9q>lTAt6wkPAKD2z%v5ead$HCyUwL@(SZb{6dWSQ1BSifWguQyx38_=Mc7Zj7M91Eq^1i&sj{h}}wReR7~PqOBfG3}OSct?P@#|z>T)kCI7@h~fnl=d_M<xWTX@zOh+bh`8pMa&NqEz877lz3>y#Pa0n5{`Iz@CM^=Xa@#a<tXVgR~cKOiwERrw$hOqD82r3>PF9)^V2Xtn+AC}bj6!8y%Udvk;zdG=VE4*)O<>rqRmO1MNSP}=3LBCCL)!=Y36Gp=cI$`!KNxQCbA%>VH5{CQP*WkMVY~=0-3VrrdYX*&cy!w`Ex?4nv}s9r#YW%h3FS0#Whke#$X822_c!t#`r9T4>Fk%f{{32b}IRmZph?Mt8csM+`2}?)*W73$ljo-7S2T=GWASV*{yWs@tQ|B0C~WZxTUpVWehgkqEIeHa(iwcHKvsgAI)e)UbVv-Ed&a%kywmI`ni6p5$GXuBK7lYtRuyyF^Wu5?<t-a7(_vO8kz$zPdBspd_e?m7XxmL6!XMkj+k%7Bq)$XnF@{fG9a;&$dJTK`%2rZ&;GMCFET#5RQd?0uF8zR5Z2;88$^@(S1TRzDClGX^y6?U|Ah?l$neeu`WM1X!_YH)O^P^4!_jHb&471=<8Y|aNa>o$)H(6xX|<fU+;-9?S0@0z;%S>1dp7ulT6|W-R}oXC%u7eIoBDIiz5*p1hM~RnfrcsF92%BN#B{}H<9OT8MLKAPgOc`}j~<wk0*7%jXKK^&>@yAHvxy#`xqd;SF>DbnI5y<BP;lj1228?Fk%C{Pm{-mxjTWisg{x7EP@+F)m)1ZX{Y}Z^>bPG3>gR~_S^=VQR(~3xLz#P$$=$!dc`HXo88~H*D4*#+nS%_d9L%rRRwBnWF^r2LWl<$9*vknp!!!u=bKg|n9PxJPpywRAT>zD*zTTLV-M?_+!%C+RA_W_iceA#H*2KR6ZZo8uT0QF-dWGd`$aQ7upz>!`RRFHAqay_@csv$|zj?oH^=YcV0Wp0BW`lJeuIoy_<ld$ukxEZ++WXB1-O>I`XEB#33?df9ilXwBPT>M%GHQOW39X-$K)88YpaT$zFu_OJLpTzSDaHWCC6*i@m5qWiC}X%GXZ~dvQ^DsfVp@kLRVf+;vlL4dCjOkZXM!aMbSTv<iUN>{8D<-kgI0-69%d$G<utwwR8dKM<zM+QG)<h#bFdI+m8;SpeTl;e&ufMaj{;Y`low#tfEggik=3nvi0X;dA$T<~PZ7Rd`6<?JWZABIXAbeQg12!o1obw{<I~h1sizc&M-5_<QweN2q$wRZERuK<4w*yzjuHg$Nnf*?dpA>VT5?oukn@cFxQ5r(CDjNE(lJ9*NZm$YbPX79NYzfPm!#+b5Pz|{j&&S-@*Q%k?mwhzqyl2QC3`W31Uv~CpK=KU-zTpE;7oybGG2z;ij1{HGoGg_4(PQ+a=g6$1#iHyz~5bdUzR5PaG^my)?Rv4yvs<w;D{NpuF1HH(@@Qer(RFv@n!2j9q||NcZ`Y1(lBOL(3SG=09B0D%au^11A4K(ru;o0{}Mq2Mn2A99OLP9Sm1_rbDUf*)9Z<6VAUKhA*}2?C#C`%AkBeIIfP^IjGrL}ok|?J;Lw56W^n}xjF8Ucl@1OCEfq~0g#R>MAU;xRK~$$masVO^yF`|NO9?O6`xpyWK*k2aXqF&o)<bX-OlZKZB1LG(^d`uMUd0pt1X=(anEL6tD-)CSwcqeOWZ9nQCUa>KwBAN*V2j@Pv!z~YhUs61R_aQMNWrv<e5_QIs;j)-^oV|zC^x1nO-<FArLx_}S;WB!SmHWZC1Ygc%KIlArRuGSJe?0@P^t#S%f=o?ZWI)bx`9sSAxEcgD}qn;6mO^rMG~+@FrX!S*25)x)|X3GKIamD+7m2MJpAVIDXd5+MoWxQl`Zj`(eh9#uiBV8cTc6RPGfBD2XXl4LAT!@5W~k8;V9wS&CdK$d?kQ=1AU@)4MfddTgfa9l(vKtYM0bD0H)bzX&@thBVOl{jCC%{Qj^J3DtGc!zFuz!*09s-DeUt+;ANhN)(LOl4}kKxp>#TY`FfL8pbq?_RC@d@@ew}~wynxTn^p^)!drZ(D16X@MC!cwWm-19Y(<s#;K!yn1gWX~N?OFmK^k9%Xa@i<%j01@$>jX_{%ca8xG%o)%1@Y~hjg|*H&#%v+5e1B@XyQ1X-u}X@kPz3iC3^*lWn9kEDEPDEo`HuB^|}G^9)~%h<|#0BP`2FoD@Y?DWD!)D{WC>oH!%1a@D4KbP^~UY<hv+s!Uq<&OiqMJpjYdWqA7s@e&oE(}0h*m;|T(P-2~dlgPiZ9GcV*#2nDk%VYLJv>McqqQvzxN=#7(BIB?lMwr*X7bR}#+|b;T<X9mei9Qsm)j6OT=5zuG9&W-z>yjeK3{lB;9UN8}phQ6FJ)TV_h2EhK1i@D!kOHbzS(|gIymR59Q<H(ay86tXSFEim8{7Ipo|uh8yklQ%4pu7Fe=~kR2M<N25>RVS9<UkPo)OR3e!hu^x>ir}EP&kH>fOSs3y(OHSE@%LxdXfVz&3~Z5m;vqXnnDtA6E|c!Y49`2q&4Pt*9;5*G0clDkfVuyxw9t04n6FZCuE@Fx4#43fDHSGONm7Ro;UsZRwj8(GW|R`Z>n_#;cn*s>I{=WYr$<+E|I*_>_Q0611wzhyyKz+_UFTY0rLAB6G!9u43SW+E&m)nx6H>ZLFQ1Rh09-03rJN<?pXQdIujr{Br2Md-<Q<fBcE1z02S;<)|$I%5>c07U9`HI2)^Jz}g}b`)Y9G8r)au(~{##4{rY_^5CwnuD&c)TXk|0IXACCZ5)PZD@eP!^tZ@Lagg$v3lQiGER^A%H=f~Dkit@{@FPGzwnbz;cFmiK|B%L-_5Eu{%=Zv^9kC9K4~&&_4;^I#WLdDoG9?I*8mor^spZ)eFs#?4O(=kJa;=6yGMQyH3qXZ#KhdE?+8wZE!Kj-Q{yMDJp7@D_34WJFI*#ln9y1h-p<35o*qwW!z0tSV*Za~Pr)&yEt(<mJHxwAoGS<R4@Np-U)gYIIKU|xV=~`piL^-gg+;9#JuMxZFIIu<*GH-nt9=nE30af)0rij2u=jPXImtO92)H}vY^WvL!x=8AW&|P^jKBnZLFR^ep;oJ6#Dt2;s7T!~}0>su=%R%A#!eGNpR*zoWRHAGPGEvQYb9wJRLsFHaMu%Y*QkY>FSS1dl>nW#x%WLQ?Uf7$~RQ2e~to=)RJ6HgAmg-&NJZ(4UVOf|(b!{V}Mle0Qb^FHrTC$Xm!J>C*V~jvOU5-Lu8^THwVAn_V?0_M3gg$Ae=E1S(iusZJg5NPTx#>RD-Vl^f+5tm#@Bp+u{%jj<*wkO#j2EJHP{+JPNU#MQ|2oX}`Vxi|dkm&Y8;+t4Y599ZfGw(UQ#0hsPtl6$!0%pp8X+kjs<j^0<GNOl2+-Cvp~OSiARR~tvI7%OsU_D0J37@G5ZpYTgjrs`JR5(g(iP8mksfv`cjFH1%3+TjjwvD}^rR!*o_qYqBeBc}_V4i@+sGs1YZxmx^mUS<B?!8Yq20;>MA|UWQM=>_N@woEG=f$~uUkFp5-zq6?EdRh)w#al2GaJiI>p+Xju>QMq2^>4iZdb6_-a%|8bmrYx<DCt)^mDWnrcOloKzlh#OWiWTKC_GY89(e^#n#@nCc23dR<C00>2e&imxQYBhciy9uIj{pInbfE@-dPbFI?`R9m=;kB5|vL%(~uj7PjEks1hL$Ps~|6ClnxQ;CGnIj1BQSan>d2&*Is2^VF^siXAbI**2Z8J^w6p(~9fsQ`aQf-z5vO?WPh<nAcQ{o$EqyRgmhTUx}6s7CEp>GgC7=700#|KdEED@Yfx;2N?LR=If)WuV5v<4W^VxPxC1amh=(!*8A0RGhguID%U{m^(hqL4)=$=>P--nmr;~0_bH>v(Ce{2Oo-_2%}*O@NyeJ*Lq!sr(^pi_<Gr6GZB~ry*Gb1M{zG6-?qy4MD!(A`YwLIU@0HU^P7?HxC)Sw9uI#Z!lE~!F={JW0JvZo)P=$U?}3Q`CR>d{HL%J`ss6s(-BP3snf{iYzVf%I7=WYYhExOcWg-OGw8|Gl3E5i8+r4}*)Jo<oQc8q>GE`09zzXvlz~1>u5=N)8IjkCajk(pQzYbn|ul{)b>VLg?|2w&J#^yDNt_^WOb>biclHm{b{ZK@y8jmZzIEU)QM!$OO8OPO^JjhY`al%YxCE}V>p5f%mpJ$x-QhKZ@YAZ7bPdQbfOd3?~oLc7K)fvYBW$qw1HV$rVIsj6ZOO{!MK2^uJ<`mMo-y9c+)9@5sy@f`=u-=tPGcspJWjb;;OzFt0rQ$FEPBAKx5It0Meo!7lXX>2r0tBUiP$E)R$wZO^`rqyOMjU_7#K)I^eR%J^|MMN4nEvwDLste9i!?J?j<PV<Oy$939MF*Obav&jsPLGPCdLqnA!VFrLYXB#rG{}%gOR6bW3QYSL+@7Sye65jBh$$WM;m`6TS`XFEv{^ot1wVhK>kbQEgG}GJ6v0#7j&j_en{np8c0!+>B$!;?lrf!Dst3T0DewVOti}9<;0Y8+bW8xVbuq<x+%Te;^?OZV5?{270~k;5GQ;1UDrLX_e2dWqrVC;Ti=*MDw-$bNfu~?628ep-{_C-@`AmVXYTr()pclObrm^N>`{lEQ&BjL-MfWnwr$mlk&=ccbBFO5(_7n|k&dqBtQ3nsYM9<}^Nh5-YZ=V<IZM@1Zto%yNS3o=Eh{m~C8neg^@!#Lpji!o57+|_{n(hHkrVvgGo(bl0F!GG8;Dn5g+OHgFtex%wycQb&e<8mTz+<ytmJ#lPcQFoo@;Z0m_pPNIaS_9YR1=Gos+y$8L2Ia*s46gFcnBvqZ}(TUG?HE6ai(uz0`p)E%TbO2?cAA21PP+JRHsvOgZ6{W@EOC_ocGH3wOp?(lSm}Ns2f5c&i#CZ%`6)?R09D7PE`7>MD#9+2n*;IJVuWn^IZg25L_9(Q7R3FjduF@2S#Ni{q<F$b)bMMf@yG1Q$Efu}iIG9!fiv0t>6k@)#3VIt!~Bk~sp`8a0(U)a9cJT^71;Lol)2L(s)CS#)X1vL(4y+L6dd7&Ms&R^N6s!v&Am>j7>YVU+{$6{k5?UpDUczEJZovOV5LG_(*xw|8tGtKFsK4Kk9b+=x#sQ*HvujJ4>DDtJt>4kZglm6=pwo~P)ZpbJhh%_9VIYz8?}nM9`+i}9xEL^gQ>GN0Y-DTUAjT=5i27qPR3x)f5Pn`%?GYPBg?t@0JGz=t!dCm+76T7@QMR;<Ws730vs=U6V%<3k7KW;C1?*if-zRNB!ZL#yGrbz~Ul$X=8>r5zNga~z66Nr32Kp||3kx}4XEd2Pc<oCUIS&MEqIgby5ckE%;R(=R@oYwKPY3$_6Ef{8YFXzMG5h)%C|>TPb`qbs!-*bN@JS0)ju9^q<Jr(Af`Jmw>SL$PG^$RP4Ow-+EY3kFrj&B%+1g!p=dfR$Bd6j_1FX>`iFokHVy>dL$<53XC!)bSEhitS-*ms6iDEHm7DZi|lSeG%&ocZ^3jsE-L6c-+pCr>|+RJz&gdt48yHa6_j!>w=OVXcF{NBgKkcnW12De0SS+q$$0(rE?Jz#V<!d)vALAMk;<5>XbNG)QJ`yRy0-jnjTrCCrAjggbTew$7ZPaA%r0zr~(_^31w|znI|1|gxgG+0;sHPh^?|No7d)fF%)w(a4}?VmYntadcQc@NIs%f{Om2OG2KZ;W`*g(x!(z;v3gX~e!dU7SrrGnTLIEan?%nhY0}%?OHy1=+Gq%yN}1!sgva%*abhhO7$UMhQ(}WrAX`RmG~r{<MUecz+jE;Su0*vAaSh1L(5RQ;yY<4#hi>rVddekHeR1#nIyTOT!iB%Nbp-DRhrw4PKU%emizL)n$u$Gt`nS59fSjqUV7io9lUZeIjXkI)V^*b<9EIAzWr00pjgAln5CiVpSqnJCtUmnTDv;_v5EXG~G}c2s3}{Lg=)uQDgv{zVYvF~~sGkik^XHZlvHhNg8FZ=^YSLSdc&hL9b%W@&fLbFx!FAr$Xsgmlt4IM>P_UKp>h(={yJ}Y@MWht^R{j*(=E)_P|Ej%~QqO}ex~^iU((TfD*~k%L6|y$0Zm$gn@I@Fx*y-7Iq+P@vmDhrDC_xBU;Y)j>MN6yOO*e8$6=lmeKFB31D$L8}LCyMB*Q0$CsIEO^U_4~F-*U8=#U?+_?KWM9D!Z@M$r(Qb0QVCz-Q|>$)`V6)e{Qee&i-2IDV;^$5O}yA2lu2ahgCq29XcgJDN9hs<4ti%q^!ha5gw#y=3Nna88fVuS+oY|4gH-5<r-d}Oe+CsXr|}0-zlB+2Q^KJM0cA_kp;PW&P=;TYvN{ga}x_>dcD=#MeI(7c@?MQlu%EVwhn{u(_hdOF7JNDyNYz1^PB2Q=n;7z(2qeUHPHEB!aBmY7aWsK+?V>d)N9nS`%1=c!9&pVfXwHfJfsEVr&z$+6znd0R}M{`iObi<NKdE6g(fo)k<&1mWhy;EKI|CZ%qNZz-Z7m=7j$>Dcef%Od9z!+sS&=_!0}2g0DG|A7Dc(4M!I4lL>gBsNHj<K#!wevyxIbuWfNn0RyqoH?3MoqlS9Vf'
PATCH_TEXT = zlib.decompress(base64.b85decode(PATCH_PAYLOAD)).decode("utf-8")
assert hashlib.sha256(PATCH_TEXT.encode()).hexdigest() == PATCH_SHA256
PATCH_PATH = ROOT/"superfermion_performance.patch"
PATCH_PATH.write_bytes(PATCH_TEXT.encode("utf-8"))
EXPECTED_SOURCE_SHA256 = {
    'crates/sf-bindings/src/lib.rs': '700864fc1a1f8e982fd4ff3707d6019a57e2bd4170652716c6b891c4c3fc6dc0',
    'crates/sf-bindings/src/prepared.rs': 'cb11d20f7e4a81cf8a34726a644070c313d0f65b1b0c2e4bb7f6aaa9347ad4b9',
    'crates/sf-ir/src/dag.rs': '773193ccfde4f0eacffe7a8f1727ca07ceaa30674b89115daf28c988137fc56e',
    'crates/sf-ir/src/state.rs': '80ad4530e6fb77fc3ad8d6abfcfd39723ec5e31060e8f45148a90179c7e2a81a',
    'docs/prepared_performance.md': 'ebcf4dfa164fce538fd0f217c4f722c5d7690a67f3683f664c837f074f67e609',
    'superfermion/__init__.py': '18592d4c41e2e49436e94db2c4c49b87d53e3601187c42e5fb8d71db0e4f4053',
    'superfermion/devices/rust_device.py': 'bb8b3be495c652b5421cfb034cdbb27e58d4f2a9c54d7b0eb059d8af8bdd2af7',
    'superfermion/kernels.py': 'cb66cf2804019bc05b55ee23e491a9f39a0218dc184cdda8fd07f1f4c7b00f4a',
    'superfermion/nn/quantum_layer.py': '8bf43cb4e89176d35e35192f5a5efb94daa8d01fbfacf1edff47016a489632c7',
    'superfermion/nn/torch_layer.py': '199b4bd415a78fd1c75e0cef4809514b17ba2ff6f35d146b94096d564084feb7',
    'superfermion/prepared.py': '68f348cbe7b213d11d82f2be5a077528baa29bf00e50377575d59bf8cb00740c',
    'tests/unit/test_prepared_performance.py': '513c03d28f8306ac2257f82331bad6bea88bd7248e251008262dea74188169a0',
}
print("Pinned source:", BASE_COMMIT)
print("Patch SHA256:", PATCH_SHA256)
print("Changed files:", len(EXPECTED_SOURCE_SHA256))

Pinned source: 35f8f7c306ac2a004e6b10dee37317d87918d5dc
Patch SHA256: 4887015d82d22f34f82f988c0f3e0a0158698439b6f908b2c19a3a4f0cb1a325
Changed files: 12


In [5]:
SOURCES = ROOT/"sources"; SOURCES.mkdir(exist_ok=True)
BASE = SOURCES/"source_baseline"; OPT = SOURCES/"source_optimized"
if not BASE.exists():
    run_logged(["git", "clone", "https://github.com/Catstate101/superfermion.git", BASE], "clone_baseline", timeout=600)
    run_logged(["git", "checkout", "--detach", BASE_COMMIT], "pin_baseline", cwd=BASE)
actual_commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=BASE, text=True).strip()
if actual_commit != BASE_COMMIT: raise RuntimeError("baseline checkout has the wrong commit; use a fresh ROOT directory")
dirty = subprocess.check_output(["git", "status", "--porcelain"], cwd=BASE, text=True)
if dirty.strip(): raise RuntimeError("baseline source has edits; use a fresh ROOT directory")
if os.name=="nt" and b"\r\n" not in (BASE/"Cargo.toml").read_bytes():
    raise RuntimeError("baseline checkout has LF endings but the embedded hashes expect the native Windows CRLF checkout; run: git config --global core.autocrlf true, then delete the sources folder under the study ROOT and rerun")
if not OPT.exists():
    shutil.copytree(BASE, OPT)
    run_logged(["git", "apply", "--check", PATCH_PATH], "check_patch", cwd=OPT)
    run_logged(["git", "apply", PATCH_PATH], "apply_patch", cwd=OPT)
for relative, expected in EXPECTED_SOURCE_SHA256.items():
    actual = hashlib.sha256((OPT/relative).read_bytes()).hexdigest()
    if actual != expected: raise RuntimeError(f"modified source differs from embedded patch: {relative}; choose a fresh ROOT")

WHEELS = ROOT/"wheels"; WHEELS.mkdir(exist_ok=True)
BUILT_WHEELS = {}
for label, source in [("source_baseline", BASE), ("source_optimized", OPT)]:
    destination = WHEELS/label; destination.mkdir(exist_ok=True)
    lane_build_env = BUILD_ENV.copy()
    lane_build_env["CARGO_TARGET_DIR"] = str(ROOT/"cargo_target"/label)
    identity = dict(base_commit=BASE_COMMIT, lane=label, patch_sha256=PATCH_SHA256 if label=="source_optimized" else None,
        python=subprocess.check_output([PYTHON,"--version"],text=True).strip(),
        cargo=subprocess.check_output([CARGO,"--version"],env=BUILD_ENV,text=True).strip(),
        rustc=subprocess.check_output([str(TOOLCHAIN/"cargo/bin"/("rustc.exe" if os.name=="nt" else "rustc")),"--version"],env=BUILD_ENV,text=True).strip(),
        rustflags=BUILD_ENV.get("RUSTFLAGS","repository .cargo/config.toml"),maturin="1.9.4",
        source_sha256={str(p.relative_to(source)):hashlib.sha256(p.read_bytes()).hexdigest()
                      for p in source.rglob("*") if p.is_file() and
                      not any(part in {".git","target","__pycache__",".pytest_cache"} for part in p.relative_to(source).parts)
                      and p.suffix not in {".pyc",".pyo"}})
    identity_sha = hashlib.sha256(json.dumps(identity,sort_keys=True).encode()).hexdigest()
    receipt_path=destination/"build_receipt.json"
    receipt=json.loads(receipt_path.read_text()) if receipt_path.exists() else {}
    wheels=list(destination.glob("*.whl"))
    verified_cache=(len(wheels)==1 and receipt.get("build_identity_sha256")==identity_sha and
                    hashlib.sha256(wheels[0].read_bytes()).hexdigest()==receipt.get("wheel_sha256"))
    if verified_cache:
        print(f"Reusing hash-verified {label} wheel; source/compiler/flags match its build receipt")
    else:
        run_logged([PYTHON,"-m","maturin","build","--release","--locked","--interpreter",PYTHON,
            "--manifest-path",source/"crates/sf-bindings/Cargo.toml","--out",destination],
            f"build_{label}",cwd=source,env=lane_build_env,timeout=3600)
        wheels=list(destination.glob("*.whl"))
        if len(wheels)!=1: raise RuntimeError(f"expected exactly one wheel in {destination}")
        receipt_path.write_text(json.dumps(dict(build_identity_sha256=identity_sha,
            wheel_sha256=hashlib.sha256(wheels[0].read_bytes()).hexdigest(),identity=identity),indent=2))
    BUILT_WHEELS[label] = str(wheels[0])
release_dir = WHEELS/"release"; release_dir.mkdir(exist_ok=True)
run_logged([PYTHON, "-m", "pip", "download", "--no-deps", "--only-binary=:all:",
            "--dest", release_dir, "superfermion==0.1.14"], "download_release")
release_wheels = list(release_dir.glob("*.whl"))
if len(release_wheels) != 1: raise RuntimeError("expected exactly one release wheel")
SITE_WHEELS = {"superfermion_wheel": str(release_wheels[0]),
               "superfermion_source_baseline": BUILT_WHEELS["source_baseline"],
               "superfermion_optimized": BUILT_WHEELS["source_optimized"]}
SITES = {}
for framework, wheel in SITE_WHEELS.items():
    site = ROOT/"sites"/framework; site.mkdir(parents=True, exist_ok=True)
    run_logged([PYTHON, "-m", "pip", "install", "--no-deps", "--upgrade", "--target", site, wheel],
               f"install_{framework}")
    SITES[framework] = str(site)
    import_env = os.environ.copy(); import_env["PYTHONPATH"] = str(site)
    native_probe = "import superfermion._sf_core as c; import superfermion as sf; print(sf.__file__); assert hasattr(c, 'PreparedCircuit') == "+str(framework == "superfermion_optimized")
    run_logged([PYTHON, "-c", native_probe], f"verify_native_{framework}", cwd=ROOT, env=import_env)
CONF["sites"] = SITES
print("Installed three independent Superfermion packages. Workers import only their selected site.")

Finished verify_native_superfermion_optimized in 0.8 seconds
Installed three independent Superfermion packages. Workers import only their selected site.


### 5. Run compiled Rust and Python checks

Core Rust tests, prepared-API numerics/ownership and Python unit tests remain
mandatory. Independent backend smoke checks and numerical gates also remain
mandatory. Optional full repository suites are off for the requested-grid/free
presets; enable `RUN_FULL_PROJECT_TESTS=True` for that extra coverage.
The full-suite path retains the Agg backend, CPU JAX and optional test-plugin
dependency fixes under the locked benchmark requirements.


In [6]:
TEST_ENV = BUILD_ENV.copy()
TEST_ENV.update(
    PYTHONPATH=str(SITES["superfermion_optimized"]), PYTHONNOUSERSITE="1",
    MPLBACKEND="Agg", SF_F32="0", RAYON_NUM_THREADS="1", OMP_NUM_THREADS="1",
    OPENBLAS_NUM_THREADS="1", MKL_NUM_THREADS="1",
    JAX_PLATFORMS="cpu", JAX_ENABLE_X64="true",
)
TEST_ENV.pop("PYTHONHOME", None)

if RUN_FULL_PROJECT_TESTS:
    # Install the optional test dependencies in the isolated worker environment.
    # Retain the benchmark dependency pins; record the resolved extras for reruns.
    PIP_ENV = TEST_ENV.copy()
    for key in ("PYTHONPATH", "PYTHONHOME", "PYTHONSTARTUP"):
        PIP_ENV.pop(key, None)
    TEST_DEPS_LOCK = EXPERIMENT_ROOT/"full_project_test_dependencies.lock.txt"
    install_command = [sys.executable, "-I", "-m", "pip", "--python", str(PYTHON),
                       "install", "--constraint", str(LOCK_PATH)]
    if TEST_DEPS_LOCK.exists():
        install_command += ["--constraint", str(TEST_DEPS_LOCK)]
    install_command += ["jax>=0.4", "pytest-benchmark>=4.0", "pytest-cov>=4.0"]
    run_logged(install_command, "full_test_dependencies", cwd=ROOT, env=PIP_ENV, timeout=1800)
    run_logged([PYTHON, "-I", "-m", "pip", "check"],
               "full_test_dependency_integrity", env=PIP_ENV, timeout=600)
    TEST_DEPS_LOCK.write_text(subprocess.check_output(
        [PYTHON, "-I", "-m", "pip", "freeze"], env=PIP_ENV, text=True))
    probe = r"""
import sys
import matplotlib
import cirq
import jax
import jax.numpy as jnp
assert matplotlib.get_backend().lower() == "agg"
assert jax.config.x64_enabled
assert all(device.platform == "cpu" for device in jax.devices())
assert str(jnp.asarray(1.0).dtype) == "float64"
print("Python:", sys.executable)
print("Matplotlib:", matplotlib.get_backend(), "; Cirq:", cirq.__version__)
print("JAX:", jax.__version__, "; devices:", jax.devices())
"""
    run_logged([PYTHON, "-c", probe], "full_test_import_checks",
               cwd=ROOT, env=TEST_ENV, timeout=600)

run_logged([CARGO, "test", "--locked", "-p", "sf-ir"],
           "rust_core_tests", cwd=OPT, env=TEST_ENV, timeout=1800)
run_logged([PYTHON, "-m", "pytest", "--import-mode=importlib", OPT/"tests/unit/test_prepared_performance.py",
            "-q", "--timeout=120"], "focused_python_tests", cwd=ROOT, env=TEST_ENV, timeout=1800)
run_logged([PYTHON, "-m", "pytest", "--import-mode=importlib", OPT/"tests/unit", "-q", "--timeout=120"],
           "python_unit_tests", cwd=ROOT, env=TEST_ENV, timeout=1800)
if RUN_FULL_PROJECT_TESTS:
    run_logged([CARGO, "test", "--locked", "--workspace"],
               "full_rust_tests", cwd=OPT, env=TEST_ENV, timeout=3600)
    run_logged([PYTHON, "-m", "pytest", "--import-mode=importlib", OPT/"tests", "-q", "--timeout=120"],
               "full_python_project_tests", cwd=ROOT, env=TEST_ENV, timeout=3600)
print("All requested compiled-extension and Python test commands passed.")


Finished python_unit_tests in 0.9 seconds
All requested compiled-extension and Python test commands passed.


### 6. Write the editable experiment modules

The notebook contains every script. Expanded cells below write ordinary
Python source; the downloaded bundle also includes the same readable `.py`
files. `legacy_adapter.py` preserves the earlier gate definitions and backend
setup. `experiment_core.py` adds native competitor batching, Qulacs, QFT and
independent state/observable references. No simulator result is replaced by
a reference implementation when a backend fails.

#### Matched original gates and backend setup

Open this source cell to inspect or modify `legacy_adapter.py`. Its SHA256 is recorded before timing.

In [7]:
MODULE_SOURCE = r'''
"""Fresh-process CPU benchmarks. No simulator fallbacks or fabricated results."""

import json
import os
from pathlib import Path
try:
    import resource
except ImportError:  # Windows: no resource module; psutil fallbacks below
    resource = None
import sys
import time
import traceback
from types import SimpleNamespace

import numpy as np
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, roc_auc_score
from sklearn.svm import SVC

MEMORY_DISCOVERY_SOURCE = "backend_default"


def aer_simulator_class():
    """Supply real OS memory metadata when a sandbox omits procfs.

    This changes import-time memory discovery only. The Aer simulator and its
    numerical kernels are unmodified; explicit thread and memory limits apply.
    """
    global MEMORY_DISCOVERY_SOURCE
    if os.name == "nt":
        # Windows exposes real physical-memory metadata through psutil directly.
        import psutil
        from qiskit_aer import AerSimulator
        MEMORY_DISCOVERY_SOURCE = "psutil.virtual_memory.total (Windows)"
        return AerSimulator
    if Path("/proc/meminfo").exists():
        from qiskit_aer import AerSimulator
        return AerSimulator
    import psutil
    total = os.sysconf("SC_PHYS_PAGES") * os.sysconf("SC_PAGE_SIZE")
    if total <= 0:
        raise RuntimeError("operating system physical-memory metadata is unavailable")
    original = psutil.virtual_memory
    psutil.virtual_memory = lambda: SimpleNamespace(total=total)
    try:
        from qiskit_aer import AerSimulator
    finally:
        psutil.virtual_memory = original
    MEMORY_DISCOVERY_SOURCE = "os.sysconf physical pages (procfs absent; import metadata only)"
    return AerSimulator


def clock():
    return time.perf_counter()


def rss_mb():
    # Linux lanes read current resident pages without a sampling thread;
    # Windows reads current RSS through psutil.
    if os.name == "nt":
        import psutil
        return psutil.Process().memory_info().rss/2**20
    try:
        pages = int(Path("/proc/self/statm").read_text().split()[1])
        return pages*os.sysconf("SC_PAGE_SIZE")/2**20
    except FileNotFoundError:
        # Restricted offline test environments may omit procfs. This fallback
        # is a high-water mark, not current RSS; the result records that fact.
        if resource is not None:
            return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss/1024
        import psutil
        return psutil.Process().memory_info().rss/2**20


def canonical(vector, n, msb=False):
    vector = np.asarray(vector, dtype=np.complex128)
    if msb:
        vector = vector.reshape([2] * n).transpose(tuple(reversed(range(n)))).reshape(-1)
    return vector


def mean_z_weights(n):
    indices = np.arange(1 << n, dtype=np.uint32)
    counts = np.zeros(len(indices), dtype=np.uint8)
    for q in range(n):
        counts += ((indices >> q) & 1).astype(np.uint8)
    return (n - 2.0 * counts) / n


def fidelity_matrix(a, b=None, block_rows=4):
    b = a if b is None else b
    result = np.empty((len(a), len(b)))
    for start in range(0, len(a), block_rows):
        result[start:start + block_rows] = np.abs(a[start:start + block_rows].conj() @ b.T)**2
    return result


def metrics(y, prediction, score):
    result = {"accuracy": float(accuracy_score(y, prediction)),
              "balanced_accuracy": float(balanced_accuracy_score(y, prediction)),
              "f1": float(f1_score(y, prediction, zero_division=0))}
    result["roc_auc"] = float(roc_auc_score(y, score)) if len(np.unique(y)) > 1 else None
    return result


def feature_spec(n, name, layers=0):
    operations = []
    def add(gate, wires, key=None):
        operations.append((gate, wires, key))
    if name == "angle":
        for q in range(n): add("RY", [q], f"x{q}")
        for q in range(n-1): add("CX", [q, q+1])
    elif name == "phase":
        for q in range(n): add("H", [q])
        for q in range(n): add("P", [q], f"x{q}")
        for q in range(n-1): add("CX", [q, q+1])
        for q in range(n): add("H", [q])
    elif name == "pauli":
        for q in range(n):
            add("RX", [q], f"x{q}"); add("RZ", [q], f"x{q}")
        for q in range(n-1): add("CX", [q, q+1])
    elif name == "zz":
        for q in range(n): add("H", [q])
        for q in range(n): add("RZ", [q], f"x{q}")
        for q in range(n-1):
            add("CX", [q, q+1]); add("RZ", [q+1], f"zz{q}"); add("CX", [q, q+1])
        for q in range(n): add("H", [q])
    elif name == "basis":
        add("X", [0])
    else:
        raise ValueError(name)
    for layer in range(layers):
        for q in range(n):
            add("RY", [q], f"t{layer*2*n+2*q}")
            add("RZ", [q], f"t{layer*2*n+2*q+1}")
        for q in range(n-1): add("CX", [q, q+1])
    return operations


def feature_values(x, name="angle", theta=None):
    result = {} if name == "basis" else {f"x{q}": float(value) for q, value in enumerate(x)}
    if name == "zz": result.update({f"zz{q}": float(x[q]*x[q+1]) for q in range(len(x)-1)})
    if theta is not None: result.update({f"t{q}": float(value) for q, value in enumerate(theta)})
    return result


def mean_z_terms(n):
    terms = []
    for q in range(n):
        pauli = [0]*n; pauli[q] = 3
        terms.append((pauli, 1/n))
    return terms


def qubo_ising(a, b, constant):
    h = -a/2 - b.sum(axis=1)/4
    pairs = [(i, j, float(b[i, j]/4)) for i in range(len(a)) for j in range(i+1, len(a)) if abs(b[i, j]) > 1e-15]
    offset = float(constant + a.sum()/2 + np.triu(b, 1).sum()/4)
    return h, pairs, offset


def qaoa_spec(n, pairs):
    operations = [("H", [q], None) for q in range(n)]
    operations += [("RZ", [q], f"z{q}") for q in range(n)]
    for p, (i, j, _) in enumerate(pairs):
        operations += [("CX", [i, j], None), ("RZ", [j], f"j{p}"), ("CX", [i, j], None)]
    return operations + [("RX", [q], f"b{q}") for q in range(n)]


def qaoa_values(n, h, pairs, gamma, beta):
    return {**{f"z{q}": float(2*gamma*h[q]) for q in range(n)},
            **{f"j{p}": float(2*gamma*v) for p, (_, _, v) in enumerate(pairs)},
            **{f"b{q}": float(2*beta) for q in range(n)}}


def ising_terms(n, h, pairs, offset):
    result = [([0]*n, offset)]
    for q in range(n):
        pauli = [0]*n; pauli[q] = 3; result.append((pauli, float(h[q])))
    for i, j, value in pairs:
        pauli = [0]*n; pauli[i] = pauli[j] = 3; result.append((pauli, value))
    return result


class Adapter:
    """Matched logical gates, explicit output contracts, canonical q0=LSB states."""

    def __init__(self, framework, n, operations, output="state", terms=None, weights=None):
        self.framework, self.n, self.operations, self.output = framework, n, operations, output
        self.optimized = framework == "superfermion_optimized"
        self.kind = "superfermion" if framework.startswith("superfermion") else ("cirq" if framework == "cirq_f64" else framework)
        self.terms, self.weights = terms, weights
        self.keys = list(dict.fromkeys(key for _, _, key in operations if key is not None))
        self.calls = 0; self.execute_s = 0.
        if self.kind == "superfermion":
            import superfermion as sf
            self.sf = sf
            self.template = sf.Circuit(n)
            for gate, wires, key in operations:
                method = getattr(self.template, gate.lower())
                method(sf.param(key), *wires) if key is not None else method(*wires)
            if self.optimized:
                self.prepared = sf.prepare(self.template)
                if output == "qubo": self.diagonal = sf.DiagonalObservable(np.asarray(weights).tolist())
            self.precision = "complex128"; self.backend = "superfermion-rust-cpu"
        elif self.kind == "qiskit":
            from qiskit import QuantumCircuit
            from qiskit.circuit import Parameter
            from qiskit.quantum_info import SparsePauliOp
            AerSimulator = aer_simulator_class()
            self.backend_object = AerSimulator(method="statevector", precision="double",
                max_parallel_threads=int(os.environ["SIM_THREADS"]), max_parallel_experiments=1,
                max_memory_mb=512)
            self.symbols = {key: Parameter(key) for key in self.keys}
            self.template = QuantumCircuit(n)
            for gate, wires, key in operations:
                method = getattr(self.template, gate.lower())
                method(self.symbols[key], *wires) if key is not None else method(*wires)
            if output == "state": self.template.save_statevector()
            else:
                strings = [("".join("IXYZ"[p] for p in reversed(pauli)), coefficient) for pauli, coefficient in terms]
                self.template.save_expectation_value(SparsePauliOp.from_list(strings), list(range(n)), label="readout")
            self.precision = "complex128"; self.backend = "qiskit-aer-statevector"
        elif self.kind == "cirq":
            import cirq
            import sympy
            self.cirq = cirq; self.qubits = cirq.LineQubit.range(n)
            self.symbols = {key: sympy.Symbol(key) for key in self.keys}
            gates = []
            for gate, wires, key in operations:
                q = [self.qubits[w] for w in wires]
                if gate == "CX": operation = cirq.CNOT(*q)
                elif gate == "H": operation = cirq.H(*q)
                elif gate == "X": operation = cirq.X(*q)
                elif gate == "P": operation = cirq.ZPowGate(exponent=self.symbols[key]/np.pi, global_shift=0).on(*q)
                else: operation = getattr(cirq, gate.lower())(self.symbols[key]).on(*q)
                gates.append(operation)
            self.template = cirq.Circuit(gates)
            if framework == "cirq_f64":
                self.simulator = cirq.Simulator(dtype=np.complex128)
            else:
                import qsimcirq
                self.simulator = qsimcirq.QSimSimulator(
                    qsim_options=qsimcirq.QSimOptions(cpu_threads=int(os.environ["SIM_THREADS"])))
            if output != "state":
                pauli_operators = [None, cirq.X, cirq.Y, cirq.Z]
                strings = [cirq.PauliString({self.qubits[q]: pauli_operators[p] for q, p in enumerate(pauli) if p},
                                            coefficient=coefficient) for pauli, coefficient in terms]
                self.observable = cirq.PauliSum.from_pauli_strings(strings)
            self.precision = "complex128" if framework == "cirq_f64" else "complex64"
            self.backend = "cirq-simulator-f64" if framework == "cirq_f64" else "cirq-qsim"
        elif self.kind == "pennylane":
            import pennylane as qml
            self.device = qml.device("lightning.qubit", wires=n, c_dtype=np.complex128)
            observable = None
            if output != "state":
                coefficients, operators = [], []
                for pauli, coefficient in terms:
                    factors = [getattr(qml, {1: "PauliX", 2: "PauliY", 3: "PauliZ"}[p])(wires=q)
                               for q, p in enumerate(pauli) if p]
                    operator = qml.prod(*factors) if len(factors) > 1 else (factors[0] if factors else qml.Identity(wires=0))
                    coefficients.append(coefficient); operators.append(operator)
                observable = qml.Hamiltonian(coefficients, operators)
            methods = {"H": qml.Hadamard, "X": qml.PauliX, "CX": qml.CNOT,
                       "RY": qml.RY, "RX": qml.RX, "RZ": qml.RZ, "P": qml.PhaseShift}
            @qml.qnode(self.device, interface=None, diff_method=None)
            def qnode(values):
                for gate, wires, key in operations:
                    methods[gate](values[key], wires=wires) if key is not None else methods[gate](wires=wires)
                return qml.state() if output == "state" else qml.expval(observable)
            self.qnode = qnode
            self.precision = "complex128"; self.backend = "pennylane-lightning.qubit"
        else:
            raise ValueError(framework)

    def reset(self):
        self.calls = 0; self.execute_s = 0.

    def state(self, values):
        if self.output != "state": raise ValueError("this adapter returns native readouts")
        start = clock()
        if self.kind == "superfermion":
            if self.optimized: result = np.asarray(self.prepared.states_batch([values])[0])
            else: result = np.asarray(self.template.bind(values).to_ir().simulate_to_state("statevector", "cpu").numpy())
        elif self.kind == "qiskit":
            circuit = self.template.assign_parameters({self.symbols[k]: v for k, v in values.items()})
            execution = self.backend_object.run(circuit).result()
            if not execution.success: raise RuntimeError(str(execution))
            result = canonical(execution.get_statevector(circuit), self.n)
        elif self.kind == "cirq":
            circuit = self.cirq.resolve_parameters(self.template, values)
            result = canonical(self.simulator.simulate(circuit, qubit_order=self.qubits).final_state_vector, self.n, msb=True)
        else: result = canonical(self.qnode(values), self.n, msb=True)
        self.calls += 1; self.execute_s += clock()-start
        return result

    def states(self, values):
        if self.optimized and len(values)*(1 << self.n)*16 <= (1 << 30):
            start = clock(); result = np.asarray(self.prepared.states_batch(values))
            self.calls += len(values); self.execute_s += clock()-start
            return result
        # All ordinary backends use the same preallocated bank, not list+vstack.
        result = np.empty((len(values), 1 << self.n), dtype=np.complex128)
        for row, parameters in enumerate(values): result[row] = self.state(parameters)
        return result

    def readouts(self, values):
        if self.output == "state": raise ValueError("use the common readout on returned vectors")
        start = clock()
        if self.optimized:
            result = (self.prepared.mean_z_batch(values) if self.output == "mean_z"
                      else self.prepared.diagonal_batch(values, self.diagonal))
        else:
            result = []
            for parameters in values:
                if self.kind == "superfermion":
                    state = self.template.bind(parameters).to_ir().simulate_to_state("statevector", "cpu")
                    value = state.expectation([(pauli, coefficient, 0.) for pauli, coefficient in self.terms])
                elif self.kind == "qiskit":
                    circuit = self.template.assign_parameters({self.symbols[k]: v for k, v in parameters.items()})
                    execution = self.backend_object.run(circuit).result()
                    if not execution.success: raise RuntimeError(str(execution))
                    value = execution.data(0)["readout"]
                elif self.kind == "cirq":
                    circuit = self.cirq.resolve_parameters(self.template, parameters)
                    value = self.simulator.simulate_expectation_values(circuit, observables=[self.observable], qubit_order=self.qubits)[0]
                else: value = self.qnode(parameters)
                result.append(float(np.real(value)))
        self.calls += len(values); self.execute_s += clock()-start
        return np.asarray(result, dtype=float)


def report_kernel(config, analytical=False):
    data = np.load(config["data_path"])
    task, n = config["task"], config["n_qubits"]
    if task == "qsvm": x, y = data["X_train"], data["X_test"]
    else: x, y = data["X_batch"], None
    adapter = None
    if analytical:
        from superfermion.kernels import angle_fidelity_kernel
        angle_fidelity_kernel(x[:1], x[:1])
    else:
        adapter = Adapter(config["framework"], n, feature_spec(n, "angle"))
        adapter.state(feature_values(x[0])); adapter.reset()
    baseline_rss = int(rss_mb()*2**20)
    start = clock()
    if analytical:
        ktrain = angle_fidelity_kernel(x)
        ktest = angle_fidelity_kernel(y, x) if y is not None else None
    else:
        sx = adapter.states([feature_values(row) for row in x])
        sy = adapter.states([feature_values(row) for row in y]) if y is not None else None
        ktrain = fidelity_matrix(sx, block_rows=config["kernel_block_rows"])
        ktest = fidelity_matrix(sy, sx, config["kernel_block_rows"]) if y is not None else None
    primary = clock()-start
    result = {"primary_runtime_s": primary, "kernel_probe": ktrain[:8, :8].tolist(),
              "kernel_trace": float(np.trace(ktrain)), "rss_postwarmup_mb": baseline_rss/2**20}
    if task == "qsvm":
        start = clock(); model = SVC(kernel="precomputed", random_state=config["seed"])
        model.fit(ktrain, data["y_train"])
        predictions = model.predict(ktest); scores = model.decision_function(ktest)
        result.update(metrics(data["y_test"], predictions, scores))
        result.update(end_to_end_s=primary+clock()-start, predictions=predictions.tolist(), scores=scores.tolist())
    else:
        start = clock(); count = len(x); centering = np.eye(count)-np.ones((count, count))/count
        eigenvalues = np.sort(np.linalg.eigvalsh(centering @ ktrain @ centering))[::-1]
        result.update(end_to_end_s=primary+clock()-start, eigenvalues=eigenvalues.tolist())
    if adapter: result.update(backend=adapter.backend, internal_precision=adapter.precision,
                              circuit_evaluations=adapter.calls, circuit_execution_s=adapter.execute_s)
    else: result.update(backend="analytical-angle", internal_precision="float64", circuit_evaluations=0)
    return result


def report_feature(config):
    n, name = config["n_qubits"], config["feature_map"]
    x = np.load(config["data_path"])["X_batch"]
    adapter = Adapter(config["framework"], n, feature_spec(n, name))
    adapter.state(feature_values(x[0], name)); adapter.reset()
    baseline = rss_mb()
    start = clock(); vectors = adapter.states([feature_values(row, name) for row in x]); total = clock()-start
    # Identical validation on every lane, explicitly outside primary timing.
    norms = np.sum(np.abs(vectors)**2, axis=1)
    return {"primary_runtime_s": total, "end_to_end_s": total, "backend": adapter.backend,
            "internal_precision": adapter.precision, "max_norm_error": float(np.max(np.abs(norms-1))),
            "circuit_evaluations": adapter.calls, "circuit_execution_s": adapter.execute_s,
            "rss_postwarmup_mb": baseline, "batch_size": len(x)}


def report_vqc(config):
    data = np.load(config["data_path"]); n = config["n_qubits"]
    x, y, xt, yt = data["X_train"], data["y_train"], data["X_test"], data["y_test"]
    native = config["track"] == "native_readout"
    weights = mean_z_weights(n)  # Cached once, for all common-readout lanes.
    adapter = Adapter(config["framework"], n, feature_spec(n, "angle", config["vqc_layers"]),
                      "mean_z" if native else "state", mean_z_terms(n))
    rng = np.random.default_rng(config["seed"]); theta = rng.normal(0, .05, 2*n*config["vqc_layers"])
    def predict(values, rows):
        parameters = [feature_values(row, theta=values) for row in rows]
        means = adapter.readouts(parameters) if native else np.abs(adapter.states(parameters))**2 @ weights
        return np.clip((1-means)/2, 1e-8, 1-1e-8)
    def loss(values, indices):
        probabilities = predict(values, x[indices]); labels = y[indices]
        return float(-np.mean(labels*np.log(probabilities)+(1-labels)*np.log(1-probabilities)))
    predict(theta, x[:1]); adapter.reset()
    baseline = rss_mb()
    batch = min(config["vqc_batch"], len(y)); fixed = np.arange(batch)
    start = clock(); initial = loss(theta, fixed)
    for iteration in range(config["vqc_iterations"]):
        indices = rng.choice(len(y), batch, replace=False); delta = rng.choice([-1., 1.], len(theta))
        rate = .08/(iteration+1)**.602; step = .05/(iteration+1)**.101
        plus, minus = loss(theta+step*delta, indices), loss(theta-step*delta, indices)
        theta -= rate*(plus-minus)/(2*step)*delta
    score = predict(theta, xt); final = loss(theta, fixed); total = clock()-start
    return {**metrics(yt, (score >= .5).astype(int), score), "primary_runtime_s": total,
            "end_to_end_s": total, "initial_loss": initial, "final_loss": final,
            "scores": score.tolist(), "parameters": theta.tolist(), "backend": adapter.backend,
            "internal_precision": adapter.precision, "circuit_evaluations": adapter.calls,
            "circuit_execution_s": adapter.execute_s, "rss_postwarmup_mb": baseline}


def report_qaoa(config):
    data = np.load(config["qubo_path"]); n = config["n_qubits"]
    h, pairs, offset = qubo_ising(data["a"], data["B"], float(data["const"]))
    cost, cards, k = data["costvec"], data["cards"].astype(np.int32), int(data["select_k"])
    native = config["track"] == "native_readout"
    operations = qaoa_spec(n, pairs)
    adapter = Adapter(config["framework"], n, operations, "qubo" if native else "state",
                      ising_terms(n, h, pairs, offset), cost)
    state_adapter = Adapter(config["framework"], n, operations) if native else adapter
    def energy(parameters):
        values = qaoa_values(n, h, pairs, *parameters)
        return float(adapter.readouts([values])[0] if native else np.dot(np.abs(adapter.state(values))**2, cost))
    parameters = np.array([.4, .3]); energy(parameters)
    state_adapter.state(qaoa_values(n, h, pairs, *parameters))
    adapter.reset(); state_adapter.reset()
    baseline = rss_mb()
    rng = np.random.default_rng(config["seed"])
    start = clock(); initial = energy(parameters)
    for iteration in range(config["qaoa_iterations"]):
        delta = rng.choice([-1., 1.], 2); rate = .12/(iteration+1)**.602; step = .08/(iteration+1)**.101
        plus, minus = energy(parameters+step*delta), energy(parameters-step*delta)
        parameters -= rate*(plus-minus)/(2*step)*delta
    probabilities = np.abs(state_adapter.state(qaoa_values(n, h, pairs, *parameters)))**2
    final = float(np.dot(probabilities, cost))
    primary = clock()-start
    # Feasible-mode selection and reference are application work, timed separately.
    start = clock(); feasible = np.flatnonzero(cards == k); raw = int(np.argmax(probabilities))
    selected = int(feasible[np.argmax(probabilities[feasible])]); optimal = float(cost[feasible].min())
    return {"primary_runtime_s": primary, "end_to_end_s": primary+clock()-start,
            "initial_energy": initial, "final_energy": final, "parameters": parameters.tolist(),
            "raw_cardinality_error": int(abs(int(cards[raw])-k)), "postselected_cost": float(cost[selected]),
            "optimal_feasible_energy": optimal, "postselected_gap": float(cost[selected]-optimal),
            "feasible_probability": float(probabilities[feasible].sum()), "backend": adapter.backend,
            "internal_precision": adapter.precision, "circuit_evaluations": adapter.calls+(state_adapter.calls if native else 0),
            "circuit_execution_s": adapter.execute_s+(state_adapter.execute_s if native else 0),
            "rss_postwarmup_mb": baseline}


def report_validation(config):
    n = config["n_qubits"]; rng = np.random.default_rng(2041+n)
    x = rng.uniform(-1.4, 1.4, n); theta = rng.normal(0, .2, 2*n)
    probes = np.unique(np.r_[0, 1, (1 << (n-1)), (1 << n)-1,
                             rng.integers(0, 1 << n, size=min(64, 1 << n))])
    if n <= 8: probes = np.arange(1 << n)
    weights = mean_z_weights(n); cases = []
    def record(name, vector, native=None, cost=None):
        values = vector[probes]
        row = {"case": name, "probe_indices": probes.tolist(), "vector_re": values.real.tolist(),
               "vector_im": values.imag.tolist(), "norm": float(np.vdot(vector, vector).real),
               "mean_z": float(np.dot(np.abs(vector)**2, weights))}
        if cost is not None:
            row.update(energy=float(np.dot(np.abs(vector)**2, cost)), energy_scale=float(max(1., np.max(np.abs(cost)))))
        if native is not None: row["native_value"] = float(native)
        cases.append(row)
    for name in ["angle", "phase", "pauli", "zz", "basis"]:
        adapter = Adapter(config["framework"], n, feature_spec(n, name))
        record(name, adapter.state(feature_values(x, name)))
    state_adapter = Adapter(config["framework"], n, feature_spec(n, "angle", 1))
    native_adapter = Adapter(config["framework"], n, feature_spec(n, "angle", 1), "mean_z", mean_z_terms(n))
    values = feature_values(x, theta=theta)
    record("vqc", state_adapter.state(values), native_adapter.readouts([values])[0])
    data = np.load(config["qubo_path"])
    h, pairs, offset = qubo_ising(data["a"], data["B"], float(data["const"]))
    operations = qaoa_spec(n, pairs)
    state_adapter = Adapter(config["framework"], n, operations)
    native_adapter = Adapter(config["framework"], n, operations, "qubo", ising_terms(n, h, pairs, offset), data["costvec"])
    for p, parameters in enumerate([(.11, .23), (.4, .3), (-.53, .17)]):
        values = qaoa_values(n, h, pairs, *parameters)
        record(f"qaoa_fixed_{p}", state_adapter.state(values), native_adapter.readouts([values])[0], data["costvec"])
    return {"cases": cases, "internal_precision": state_adapter.precision,
            "backend": state_adapter.backend, "primary_runtime_s": None}


def main(config_path, output_path):
    config = json.loads(Path(config_path).read_text())
    result = {key: config.get(key) for key in ["job_id", "cache_key", "task", "track", "framework", "dataset",
                                             "n_qubits", "feature_map", "repetition", "threads", "seed", "manifest_sha256"]}
    start = clock()
    try:
        if config["task"] == "validation": measurement = report_validation(config)
        elif config["task"] in ["qsvm", "qkpca"]: measurement = report_kernel(config, config["framework"] == "analytical_angle")
        elif config["task"] == "feature_map": measurement = report_feature(config)
        elif config["task"] == "vqc": measurement = report_vqc(config)
        elif config["task"] == "qaoa_fs": measurement = report_qaoa(config)
        else: raise ValueError(config["task"])
        result.update(measurement); result["status"] = "ok"
    except Exception as error:
        result.update(status="error", error_type=type(error).__name__, error=str(error), traceback=traceback.format_exc())
    result["worker_wall_s"] = clock()-start
    if resource is not None:
        result["process_peak_rss_mb"] = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss/1024
    else:
        import psutil
        result["process_peak_rss_mb"] = psutil.Process().memory_info().peak_wset/2**20
    result["rss_after_task_mb"] = rss_mb()
    result["current_rss_source"] = ("procfs" if Path("/proc/self/statm").exists()
        else ("psutil peak_wset (Windows)" if os.name == "nt" else "process high-water fallback"))
    result["backend_memory_discovery"] = MEMORY_DISCOVERY_SOURCE
    path = Path(output_path); temporary = path.with_suffix(".tmp")
    temporary.write_text(json.dumps(result, indent=2, allow_nan=False)); temporary.replace(path)
    print(json.dumps({"job_id": result["job_id"], "status": result["status"]}))
    if result["status"] != "ok": raise SystemExit(1)


if __name__ == "__main__": main(*sys.argv[1:])
'''
(EXPERIMENT_ROOT/'legacy_adapter.py').write_bytes(MODULE_SOURCE.lstrip("\n").encode("utf-8"))
print("Wrote legacy_adapter.py")

Wrote legacy_adapter.py


#### Native batch adapters and independent references

Open this source cell to inspect or modify `experiment_core.py`. Its SHA256 is recorded before timing.

In [8]:
MODULE_SOURCE = r'''
"""Matched CPU adapters and independent numerical references for the study."""
import os
import time
import numpy as np

from legacy_adapter import (Adapter as LegacyAdapter, feature_spec, feature_values,
                            mean_z_terms, mean_z_weights, fidelity_matrix,
                            ising_terms, canonical)


def canonical_bank(vectors, n, msb=False):
    bank = np.asarray(vectors, dtype=np.complex128).reshape(-1, 1 << n)
    if msb:
        bank = bank.reshape(len(bank), *([2] * n)).transpose(0, *range(n, 0, -1)).reshape(len(bank), -1)
    return np.ascontiguousarray(bank)


class Adapter(LegacyAdapter):
    """Native batching where supported; actual simulator precision is retained."""
    def __init__(self, framework, n, operations, output="state", terms=None, weights=None):
        if framework != "qulacs":
            super().__init__(framework, n, operations, output, terms, weights)
        else:
            from qulacs import ParametricQuantumCircuit, QuantumState, Observable
            self.framework, self.kind, self.n = framework, framework, n
            self.operations, self.output, self.terms, self.weights = operations, output, terms, weights
            self.optimized = False
            self.keys = list(dict.fromkeys(k for _, _, k in operations if k is not None))
            self.calls, self.execute_s = 0, 0.
            self.template = ParametricQuantumCircuit(n)
            self.parameter_slots, self.phase_keys = [], []
            for gate, wires, key in operations:
                if key is None:
                    getattr(self.template, "add_" + {"CX":"CNOT"}.get(gate, gate) + "_gate")(*wires)
                else:
                    # Qulacs rotates with exp(+i theta P/2). The common
                    # convention is exp(-i theta P/2); convert every angle.
                    axis = "RZ" if gate == "P" else gate
                    getattr(self.template, "add_parametric_" + axis + "_gate")(*wires, 0.)
                    self.parameter_slots.append(key)
                    if gate == "P": self.phase_keys.append(key)
            self.state_object = QuantumState(n)
            if output != "state":
                self.observable = Observable(n)
                for pauli, coefficient in terms:
                    text = " ".join(f'{"IXYZ"[p]} {q}' for q, p in enumerate(pauli) if p) or "I 0"
                    self.observable.add_operator(float(coefficient), text)
            self.precision, self.backend = "complex128", "qulacs-cpu-parametric"
        self.batch_strategy = {
            "superfermion_optimized":"PreparedCircuit.states_batch / native reductions",
            "superfermion_wheel":"existing bind/to_ir/simulate loop",
            "superfermion_source_baseline":"existing bind/to_ir/simulate loop",
            "qiskit":"Aer runtime parameter_binds; single experiment thread budget",
            "pennylane":"QNode broadcasting; Lightning expands broadcast tapes",
            "cirq":"qsim simulate_sweep / expectation sweep",
            "cirq_f64":"Cirq complex128 simulate_sweep / expectation sweep",
            "qulacs":"reused ParametricQuantumCircuit and QuantumState; parameter update loop",
        }[framework]

    def _qulacs_run(self, values):
        for i, key in enumerate(self.parameter_slots):
            self.template.set_parameter(i, -float(values[key]))
        self.state_object.set_zero_state()
        self.template.update_quantum_state(self.state_object)
        if self.phase_keys:
            # P(theta) = exp(i theta/2) RZ(theta); retain the global phase
            # for a literal, fully matched statevector output contract.
            self.state_object.multiply_coef(np.exp(.5j*sum(values[k] for k in self.phase_keys)))

    def _aer_execute(self, values):
        if self.keys:
            binds = [{self.symbols[k]: [float(row[k]) for row in values] for k in self.keys}]
            result = self.backend_object.run(self.template, parameter_binds=binds,
                runtime_parameter_bind_enable=True).result()
        else:
            result = self.backend_object.run([self.template for _ in values]).result()
        if not result.success or len(result.results) != len(values):
            raise RuntimeError("Aer returned an unsuccessful or incomplete batch")
        return result

    def states(self, values):
        values = list(values)
        if not values: return np.empty((0, 1 << self.n), dtype=np.complex128)
        if self.output != "state": raise ValueError("statevector output required")
        start = time.perf_counter()
        if self.kind == "superfermion":
            if self.optimized:
                result = np.asarray(self.prepared.states_batch(values))
            else:
                result = np.empty((len(values), 1 << self.n), dtype=np.complex128)
                for i, row in enumerate(values):
                    result[i] = self.template.bind(row).to_ir().simulate_to_state("statevector", "cpu").numpy()
        elif self.kind == "qiskit":
            execution = self._aer_execute(values)
            result = canonical_bank([execution.get_statevector(i) for i in range(len(values))], self.n)
        elif self.kind == "cirq":
            execution = self.simulator.simulate_sweep(self.template, params=values, qubit_order=self.qubits)
            result = canonical_bank([row.final_state_vector for row in execution], self.n, msb=True)
        elif self.kind == "pennylane":
            if self.keys:
                broadcast = {k: np.asarray([row[k] for row in values]) for k in self.keys}
                result = canonical_bank(self.qnode(broadcast), self.n, msb=True)
            else:
                result = canonical_bank([self.qnode(row) for row in values], self.n, msb=True)
        else:
            result = np.empty((len(values), 1 << self.n), dtype=np.complex128)
            for i, row in enumerate(values):
                self._qulacs_run(row)
                result[i] = self.state_object.get_vector()
        self.calls += len(values); self.execute_s += time.perf_counter()-start
        return result

    def state(self, values):
        return self.states([values])[0]

    def readouts(self, values):
        if self.output == "state": raise ValueError("native expectation output required")
        values = list(values)
        start = time.perf_counter()
        if self.kind == "superfermion":
            if self.optimized and self.output == "mean_z":
                result = self.prepared.mean_z_batch(values)
            elif self.optimized and self.output == "qubo":
                result = self.prepared.diagonal_batch(values, self.diagonal)
            else:
                result = []
                for row in values:
                    state = (self.prepared.simulate(row) if self.optimized else
                             self.template.bind(row).to_ir().simulate_to_state("statevector", "cpu"))
                    result.append(state.expectation([(p, c, 0.) for p, c in self.terms]).real)
        elif self.kind == "qiskit":
            execution=self._aer_execute(values)
            result = [execution.data(i)["readout"] for i in range(len(values))]
        elif self.kind == "cirq":
            result = [row[0] for row in self.simulator.simulate_expectation_values_sweep(
                self.template, observables=[self.observable], params=values, qubit_order=self.qubits)]
        elif self.kind == "pennylane":
            if self.keys:
                result = self.qnode({k: np.asarray([row[k] for row in values]) for k in self.keys})
            else:
                result = [self.qnode(row) for row in values]
        else:
            result = []
            for row in values:
                self._qulacs_run(row)
                result.append(self.observable.get_expectation_value(self.state_object))
        self.calls += len(values); self.execute_s += time.perf_counter()-start
        return np.real(np.asarray(result)).astype(float).reshape(-1)


def reference_state(n, operations, values):
    """Independent NumPy state propagation, canonical qubit q = bit q."""
    state = np.zeros(1 << n, dtype=np.complex128); state[0] = 1
    indices = np.arange(1 << n, dtype=np.uint64)
    for gate, wires, key in operations:
        if gate == "CX":
            c, t = wires
            state = state[indices ^ (((indices >> c) & 1) << t)]
            continue
        if gate == "H": matrix = np.array([[1,1],[1,-1]], complex)/np.sqrt(2)
        elif gate == "X": matrix = np.array([[0,1],[1,0]], complex)
        else:
            theta = float(values[key]); c, s = np.cos(theta/2), np.sin(theta/2)
            if gate == "RY": matrix = np.array([[c,-s],[s,c]], complex)
            elif gate == "RX": matrix = np.array([[c,-1j*s],[-1j*s,c]], complex)
            elif gate == "RZ": matrix = np.diag([np.exp(-.5j*theta), np.exp(.5j*theta)])
            elif gate == "P": matrix = np.diag([1,np.exp(1j*theta)])
            else: raise ValueError(gate)
        q = wires[0]; stride = 1 << q
        bank = state.reshape(-1, 2, stride)
        lo, hi = bank[:,0,:].copy(), bank[:,1,:].copy()
        bank[:,0,:] = matrix[0,0]*lo+matrix[0,1]*hi
        bank[:,1,:] = matrix[1,0]*lo+matrix[1,1]*hi
    return state


def align_error(actual, expected):
    overlap = np.vdot(expected, actual)
    phase = overlap/abs(overlap) if abs(overlap) > 1e-20 else 1.
    maximum = 0.
    for start in range(0, len(actual), 1 << 18):
        stop = start + (1 << 18)
        maximum = max(maximum, float(np.max(np.abs(actual[start:stop]/phase-expected[start:stop]))))
    return maximum


def pauli_expectations(bank, terms, n):
    """Exact Pauli contractions with bounded basis-index temporaries."""
    bank = np.asarray(bank, dtype=np.complex128).reshape(-1, 1 << n)
    result = np.zeros(len(bank), dtype=np.complex128)
    for pauli, coefficient in terms:
        flip = sum(1 << q for q,p in enumerate(pauli) if p in [1,2])
        ys = sum(p == 2 for p in pauli)
        sign_qubits = [q for q,p in enumerate(pauli) if p in [2,3]]
        for start in range(0, 1 << n, 1 << 18):
            stop = min(start + (1 << 18), 1 << n)
            indices = np.arange(start, stop, dtype=np.uint64)
            parity = np.zeros(len(indices), dtype=np.uint8)
            for q in sign_qubits: parity ^= ((indices >> q) & 1).astype(np.uint8)
            sign = 1. - 2.*parity
            for row, vector in enumerate(bank):
                value = np.vdot(vector[start:stop], vector[indices ^ flip]*sign)
                result[row] += coefficient*((-1j)**ys)*value
    if np.max(np.abs(result.imag)) > 1e-8: raise RuntimeError("observable is not Hermitian")
    return result.real


def state_chunk_rows(n, budget_bytes, simultaneous_banks=1, row_cap=None):
    """Bound exported complex128 banks; backend workspaces are additional."""
    required = simultaneous_banks*(1 << n)*16
    rows = int(budget_bytes)//required
    if rows < 1:
        raise MemoryError(f"{simultaneous_banks} state(s) at {n} qubits need {required/2**20:.1f} MiB; output budget is {budget_bytes/2**20:.1f} MiB")
    return min(rows, int(row_cap)) if row_cap else rows


def quantum_kernel_matrices(adapter, train_rows, test_rows, budget_bytes, block_rows=128):
    """Dense when affordable; otherwise exact kernels by re-encoding blocks.

    Every simulator uses the same mode/block size for a condition. No state
    bank is written to disk. The bounded mode trades additional simulation for
    memory; its timing is not interchangeable with a retained-bank experiment.
    """
    train_rows, test_rows = list(train_rows), list(test_rows)
    nt, nv = len(train_rows), len(test_rows)
    kt, kv = np.empty((nt, nt)), np.empty((nv, nt))
    encoding, algebra, norm_error = 0., 0., 0.
    def states(rows):
        nonlocal encoding, norm_error
        before = time.perf_counter(); bank = adapter.states(rows)
        encoding += time.perf_counter()-before
        if bank.shape != (len(rows), 1 << adapter.n) or not np.isfinite(bank).all():
            raise ValueError("Invalid kernel state block")
        for vector in bank:
            norm_error = max(norm_error, abs(float(np.vdot(vector, vector).real)-1.))
        return bank
    retained_bytes = (nt+nv)*(1 << adapter.n)*16
    if retained_bytes <= budget_bytes:
        left, right = states(train_rows), states(test_rows)
        before = time.perf_counter()
        kt[:] = fidelity_matrix(left, block_rows=8)
        kv[:] = fidelity_matrix(right, left, block_rows=8)
        algebra += time.perf_counter()-before
        strategy, rows, peak_bytes = "retained_state_bank", nt+nv, retained_bytes
    else:
        rows = state_chunk_rows(adapter.n, budget_bytes, simultaneous_banks=2, row_cap=block_rows)
        for i in range(0, nt, rows):
            stop_i = min(i+rows, nt); left = states(train_rows[i:stop_i])
            for j in range(i, nt, rows):
                stop_j = min(j+rows, nt)
                right = left if i == j else states(train_rows[j:stop_j])
                before = time.perf_counter(); tile = np.abs(left.conj() @ right.T)**2
                kt[i:stop_i,j:stop_j] = tile
                if i != j: kt[j:stop_j,i:stop_i] = tile.T
                algebra += time.perf_counter()-before
                del right
            for j in range(0, nv, rows):
                stop_j = min(j+rows, nv); right = states(test_rows[j:stop_j])
                before = time.perf_counter()
                kv[j:stop_j,i:stop_i] = np.abs(right.conj() @ left.T)**2
                algebra += time.perf_counter()-before
                del right
            del left
        strategy, peak_bytes = "bounded_reencoding_blocks", min(nt+nv, 2*rows)*(1 << adapter.n)*16
    return kt, kv, dict(encoding_runtime_s=encoding, kernel_algebra_s=algebra,
        kernel_strategy=strategy, kernel_block_rows=rows,
        logical_statebank_bytes=retained_bytes, exported_bank_peak_bytes=peak_bytes,
        all_encoded_state_norm_error=norm_error)


def tfim_ground_energy(n, coupling=1., field=.7):
    """Exact open-chain TFIM reference, with no 2**n by 2**n matrix.

    After a global spin rotation and Jordan-Wigner transformation, the
    positive single-particle energies are twice the singular values of the
    bidiagonal field/coupling matrix. E0 is minus their singular-value sum.
    This applies only to tfim_terms(n, coupling, field), with open boundaries.
    Independent dense checks for small n are performed in offline_checks.py.
    """
    matrix = np.diag(np.full(n, float(field)))
    if n > 1: matrix += np.diag(np.full(n-1, float(coupling)), 1)
    return -float(np.linalg.svd(matrix, compute_uv=False).sum())


def circuit_spec(n, family, depth, seed):
    rng = np.random.default_rng(seed)
    if family in ["angle","phase","pauli","zz"]:
        op = feature_spec(n,family)
        return op, feature_values(rng.uniform(-1.5,1.5,n), family)
    if family == "ghz": return [("H",[0],None)]+[("CX",[0,q],None) for q in range(1,n)], {}
    op, values = [], {}
    def rotation(gate, q, value):
        key = "r"+str(len(values)); values[key] = float(value); op.append((gate,[q],key))
    if family == "random":
        for layer in range(depth):
            for q in range(n):
                rotation("RY",q,rng.uniform(-np.pi,np.pi))
                rotation("RZ",q,rng.uniform(-np.pi,np.pi))
            for q in range(layer%2,n-1,2): op.append(("CX",[q,q+1],None))
            for q in range(1-layer%2,n-1,2): op.append(("CX",[q,q+1],None))
    elif family == "qft":
        # Nontrivial product input, then QFT with controlled phase gates
        # decomposed into common P/CNOT operations and final bit swaps.
        for q in range(n): rotation("RY",q,rng.uniform(-1.5,1.5))
        for target in reversed(range(n)):
            op.append(("H",[target],None))
            for control in reversed(range(target)):
                phi = np.pi/2**(target-control)
                rotation("P",control,phi/2); op.append(("CX",[control,target],None))
                rotation("P",target,-phi/2); op.append(("CX",[control,target],None))
                rotation("P",target,phi/2)
        for q in range(n//2):
            op.extend([("CX",[q,n-1-q],None),("CX",[n-1-q,q],None),("CX",[q,n-1-q],None)])
    else: raise ValueError(family)
    return op, values


def qaoa_spec_values(n, h, pairs, p, theta):
    op, values = [("H",[q],None) for q in range(n)], {}
    for layer in range(p):
        gamma, beta = theta[2*layer:2*layer+2]
        for q in range(n):
            key = f"z{layer}_{q}"; values[key] = float(2*gamma*h[q]); op.append(("RZ",[q],key))
        for j,(a,b,c) in enumerate(pairs):
            key = f"j{layer}_{j}"; values[key] = float(2*gamma*c)
            op.extend([("CX",[a,b],None),("RZ",[b],key),("CX",[a,b],None)])
        for q in range(n):
            key = f"b{layer}_{q}"; values[key] = float(2*beta); op.append(("RX",[q],key))
    return op, values


def qubo_to_ising(a, b, constant):
    """C(z)=constant+a.z+sum_{i<j} b[i,j]z_i z_j; z=(1-Z)/2."""
    b=np.triu(b,1)
    h=-np.asarray(a)/2-(b.sum(axis=0)+b.sum(axis=1))/4
    pairs=[(i,j,float(b[i,j]/4)) for i in range(len(a)) for j in range(i+1,len(a)) if b[i,j]!=0]
    offset=float(constant+np.asarray(a).sum()/2+b.sum()/4)
    return h,pairs,offset


def tfim_terms(n, coupling=1., field=.7):
    terms = []
    for q in range(n-1):
        p = [0]*n; p[q]=p[q+1]=3; terms.append((p,-coupling))
    for q in range(n):
        p = [0]*n; p[q]=1; terms.append((p,-field))
    return terms


def hamiltonian_matrix(n, terms):
    matrices = [np.eye(2),np.array([[0,1],[1,0]]),np.array([[0,-1j],[1j,0]]),np.diag([1,-1])]
    result = np.zeros((1 << n,1 << n), complex)
    for pauli, coefficient in terms:
        matrix = np.array([[1.]])
        for q in reversed(range(n)): matrix = np.kron(matrix,matrices[pauli[q]])
        result += coefficient*matrix
    return result


def angle_kernel(x, y=None):
    y = x if y is None else y
    return np.prod(np.cos((x[:,None,:]-y[None,:,:])/2)**2, axis=-1)


def spsa(initial, objective, iterations, seed, batch_indices=None, trace_objective=None):
    rng = np.random.default_rng(seed)
    theta = np.asarray(initial).copy(); trace = []
    for k in range(iterations):
        delta = rng.choice([-1.,1.],len(theta)); a=.12/(k+1)**.602; c=.1/(k+1)**.101
        indices = None if batch_indices is None else batch_indices[k]
        plus, minus = objective(theta+c*delta,indices), objective(theta-c*delta,indices)
        theta -= a*(plus-minus)/(2*c)*delta
        value = float(objective(theta,indices) if trace_objective is None else trace_objective(theta))
        trace.append(value)
    return theta, trace


def parameter_shift_values(values, keys):
    rows = []
    for key in keys:
        plus, minus = dict(values), dict(values)
        plus[key] += np.pi/2; minus[key] -= np.pi/2
        rows.extend([plus,minus])
    return rows
'''
(EXPERIMENT_ROOT/'experiment_core.py').write_bytes(MODULE_SOURCE.lstrip("\n").encode("utf-8"))
print("Wrote experiment_core.py")

Wrote experiment_core.py


#### Train-only data preparation

Open this source cell to inspect or modify `prepare_study_data.py`. Its SHA256 is recorded before timing.

In [9]:
MODULE_SOURCE = r'''
"""Shared train-only preprocessing, immutable dataset splits, and QUBO inputs."""
import hashlib
import json
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from sklearn.datasets import (load_breast_cancer, load_iris, load_wine,
                              load_digits, load_diabetes, make_moons, make_friedman1)
from sklearn.decomposition import PCA
from sklearn.feature_selection import mutual_info_classif
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler


def sha(path): return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def load_dataset(name, seed, custom):
    sources = {
        "breast_cancer": "sklearn.datasets.load_breast_cancer (Wisconsin diagnostic)",
        "iris": "sklearn.datasets.load_iris (all three classes)",
        "wine": "sklearn.datasets.load_wine (all three classes)",
        "digits_01": "sklearn.datasets.load_digits, digits 0 and 1 only",
        "moons": "sklearn.datasets.make_moons(n_samples=400, noise=0.20, random_state=seed)",
        "diabetes": "sklearn.datasets.load_diabetes (regression)",
        "friedman1": "sklearn.datasets.make_friedman1(n_samples=400, n_features=10, noise=1, random_state=seed)",
    }
    kind = "regression" if name in ["diabetes","friedman1"] else "classification"
    if name in custom:
        spec = custom[name]; frame = pd.read_csv(spec["path"])
        if spec["target"] not in frame: raise ValueError("custom CSV target is missing")
        y = frame.pop(spec["target"]).to_numpy()
        x = frame.to_numpy(dtype=float); kind = spec["kind"]
        source = f'custom CSV {spec["path"]}; SHA256={sha(spec["path"])}'
    elif name == "moons": x,y = make_moons(400,noise=.2,random_state=seed); source=sources[name]
    elif name == "friedman1": x,y = make_friedman1(400,n_features=10,noise=1,random_state=seed); source=sources[name]
    else:
        loader = {"breast_cancer":load_breast_cancer,"iris":load_iris,"wine":load_wine,
                  "digits_01":load_digits,"diabetes":load_diabetes}[name]
        x,y = loader(return_X_y=True); source=sources[name]
        if name == "digits_01": mask=y<2; x,y=x[mask],y[mask]
    x = np.asarray(x,dtype=np.float64)
    if not np.isfinite(x).all() or pd.isna(y).any(): raise ValueError(f"non-finite/missing data in {name}")
    classes = []
    if kind == "classification":
        encoder = LabelEncoder(); y = encoder.fit_transform(y); classes = encoder.classes_.astype(str).tolist()
    else: y = np.asarray(y,dtype=float)
    return x,np.asarray(y),kind,source,classes


def preprocess(x, train, test, n):
    scaler = StandardScaler().fit(x[train])
    xt, xv = scaler.transform(x[train]), scaler.transform(x[test])
    components = min(n,x.shape[1],len(train)-1)
    pca = PCA(n_components=components,svd_solver="full").fit(xt)
    a,b = pca.transform(xt),pca.transform(xv)
    limits = np.maximum(np.quantile(np.abs(a),.95,axis=0),1e-10)
    a,b = np.clip(a/limits,-1,1)*np.pi,np.clip(b/limits,-1,1)*np.pi
    if components < n:
        a,b = np.pad(a,((0,0),(0,n-components))),np.pad(b,((0,0),(0,n-components)))
    arrays = dict(X_train=a,X_test=b,raw_train=x[train],raw_test=x[test],train_ids=train,test_ids=test,
                  scaler_mean=scaler.mean_,scaler_scale=scaler.scale_,pca_mean=pca.mean_,
                  pca_components=pca.components_,angle_limits=limits)
    audit = dict(active_components=components,padded_qubits=n-components,
                 pca_explained_variance=float(pca.explained_variance_ratio_.sum()),
                 train_clipped_fraction=float(np.mean(np.abs(a[:,:components])>=np.pi)),
                 test_clipped_fraction=float(np.mean(np.abs(b[:,:components])>=np.pi)))
    return arrays,audit


def qubo_inputs(xtrain, ytrain, n, seed, allow_ancillas=False):
    mi = mutual_info_classif(xtrain,ytrain,random_state=seed)
    features = np.argsort(-mi,kind="stable")[:min(n,xtrain.shape[1])]
    active=len(features)
    if active < n and not allow_ancillas: raise ValueError("feature-selection qubits exceed available original features")
    relevance = mi[features]; relevance /= max(float(relevance.max()),1e-12)
    correlation = np.nan_to_num(np.abs(np.corrcoef(xtrain[:,features],rowvar=False)))
    k = max(1,active//2); penalty = 1.5
    a = np.zeros(n);a[:active]=-relevance+penalty*(1-2*k)
    b = np.zeros((n,n));b[:active,:active]=np.triu(.25*correlation+2*penalty,1)
    constant = penalty*k*k
    # Retain the 2**n diagonal but avoid a 2**n by n bit matrix at 24 qubits.
    logical = np.empty(1 << active, dtype=np.float64)
    for start in range(0, 1 << active, 1 << 14):
        stop = min(start+(1 << 14), 1 << active)
        indices = np.arange(start, stop, dtype=np.uint64)
        bits = ((indices[:,None] >> np.arange(active,dtype=np.uint64)) & 1).astype(float)
        logical[start:stop] = constant+bits@a[:active]+np.einsum("bi,ij,bj->b",bits,b[:active,:active],bits,optimize=True)
    weights=np.tile(logical,1 << (n-active)) if active<n else logical
    return dict(qubo_a=a,qubo_b=b,qubo_constant=np.array(constant),qubo_cost=weights,
                selected_original_features=features,cardinality=np.array(k),feature_relevance=relevance,
                feature_correlation=correlation,active_feature_variables=np.array(active),
                feature_selection_ancillas=np.array(n-active))


def prepare(config):
    root = Path(config["root"]); directory=root/"data"; directory.mkdir(parents=True,exist_ok=True)
    profile=config["profile"]; manifest=dict(paths={},sha256={},datasets=[]); unsupported=[]
    train_ratio=float(profile["train_ratio"]); test_ratio=float(profile["test_ratio"])
    if not (0 < train_ratio < 1 and 0 < test_ratio < 1
            and np.isclose(train_ratio+test_ratio,1.0,rtol=0,atol=1e-12)):
        raise ValueError("Train and test ratios must be positive and sum to 1.")
    for seed in profile["seeds"]:
        for name in profile["classification_datasets"]+profile["regression_datasets"]:
            x,y,kind,source,classes=load_dataset(name,seed,config.get("custom_datasets",{}))
            train,test=train_test_split(np.arange(len(x)),test_size=test_ratio,random_state=seed,
                stratify=y if kind=="classification" else None)
            if set(train)&set(test): raise AssertionError("split leakage")
            if len(train)+len(test)!=len(x): raise AssertionError("split omitted dataset rows")
            for n in sorted(set(profile["qml_qubits"]+profile.get("variational_qubits",profile["qml_qubits"]))):
                arrays,audit=preprocess(x,train,test,n)
                ym,ys=(0.,1.) if kind=="classification" else (float(y[train].mean()),float(max(y[train].std(),1e-12)))
                arrays.update(y_train=y[train],y_test=y[test],y_train_scaled=(y[train]-ym)/ys,
                    y_test_scaled=(y[test]-ym)/ys,target_mean=np.array(ym),target_scale=np.array(ys))
                requested=name in profile["feature_selection_datasets"]
                allow_ancillas=bool(profile.get("feature_selection_ancillas",False))
                available=requested and (n<=x.shape[1] or allow_ancillas)
                audit.update(feature_selection_requested=requested,feature_selection_available=available,
                             feature_selection_max_qubits=x.shape[1],
                             feature_selection_active_features=min(n,x.shape[1]) if requested else 0,
                             feature_selection_ancilla_qubits=max(0,n-x.shape[1]) if requested and allow_ancillas else 0)
                if available:
                    arrays.update(qubo_inputs(x[train],y[train],n,seed,allow_ancillas=allow_ancillas))
                elif requested:
                    unsupported.append(dict(dataset=name,n_qubits=n,seed=seed,
                        original_features=x.shape[1],reason="Feature-selection qubits exceed the number of original features"))
                key=f"{name}:{n}:{seed}"; path=directory/(key.replace(":","_")+".npz")
                np.savez_compressed(path,**arrays)
                manifest["paths"][key]=str(path); manifest["sha256"][key]=sha(path)
                manifest["datasets"].append(dict(dataset=name,kind=kind,source=source,classes=classes,
                    seed=seed,n_qubits=n,original_samples=len(x),original_features=x.shape[1],
                    train_samples=len(train),test_samples=len(test),
                    requested_train_ratio=train_ratio,requested_test_ratio=test_ratio,
                    actual_train_ratio=len(train)/len(x),actual_test_ratio=len(test)/len(x),
                    class_counts_train={str(v):int((y[train]==v).sum()) for v in np.unique(y[train])} if kind=="classification" else {},
                    raw_values_sha256=hashlib.sha256(x.tobytes()+y.tobytes()).hexdigest(),**audit))
    (root/"data_manifest.json").write_text(json.dumps(manifest,indent=2))
    pd.DataFrame(manifest["datasets"]).to_csv(root/"dataset_audit.csv",index=False)
    pd.DataFrame(unsupported,columns=["dataset","n_qubits","seed","original_features","reason"]).to_csv(
        root/"unsupported_feature_selection.csv",index=False)
    print(f'Prepared {len(manifest["paths"])} dataset/qubit/seed inputs; '
          f'{train_ratio:.0%}/{test_ratio:.0%} split using all rows; train-only scaling, PCA and angle limits.')
    print("Bundled datasets require no external dataset downloads. Padded components are explicitly recorded.")
    if unsupported:
        print(f"Recorded {len(unsupported)} unsupported feature-selection dataset/qubit/seed cases in unsupported_feature_selection.csv; other QML tasks retain these qubit settings.")


if __name__=="__main__": prepare(json.loads(Path(sys.argv[1]).read_text()))
'''
(EXPERIMENT_ROOT/'prepare_study_data.py').write_bytes(MODULE_SOURCE.lstrip("\n").encode("utf-8"))
print("Wrote prepare_study_data.py")

Wrote prepare_study_data.py


#### QML, optimization, gradients and circuit workloads

Open this source cell to inspect or modify `study_worker.py`. Its SHA256 is recorded before timing.

In [10]:
MODULE_SOURCE = r'''
"""A single real benchmark job; errors are retained and never replaced by fallbacks."""
import json
import importlib
import importlib.metadata
import os
from pathlib import Path
try:
    import resource
except ImportError:  # Windows: no resource module; psutil fallbacks below
    resource = None
import sys
import time
import traceback

import numpy as np
from scipy.special import expit, softmax
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.kernel_ridge import KernelRidge
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
                             roc_auc_score, mean_squared_error, mean_absolute_error, r2_score)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

from experiment_core import (Adapter, feature_spec, feature_values, mean_z_terms,
    mean_z_weights, fidelity_matrix, reference_state, align_error, pauli_expectations,
    circuit_spec, qaoa_spec_values, qubo_to_ising, ising_terms, tfim_terms,
    hamiltonian_matrix, angle_kernel, spsa, parameter_shift_values,
    quantum_kernel_matrices, state_chunk_rows, tfim_ground_energy)
import legacy_adapter


def peak_mb():
    if resource is not None:
        return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss/1024
    import psutil
    return psutil.Process().memory_info().peak_wset/2**20


def current_rss_mib():
    """Resident RAM, including Python and native allocations; no sampler."""
    if os.name == "nt":
        import psutil
        return psutil.Process().memory_info().rss/2**20, "Windows psutil current RSS"
    try:
        pages=int(Path("/proc/self/statm").read_text().split()[1])
        return pages*os.sysconf("SC_PAGE_SIZE")/2**20, "Linux /proc/self/statm"
    except (OSError, ValueError, IndexError):
        # A high-water mark is not a current-RSS measurement.
        return None, "unavailable: procfs resident-page counter could not be read"


def classification_metrics(y, prediction, scores=None):
    result = dict(accuracy=float(accuracy_score(y,prediction)),
                  balanced_accuracy=float(balanced_accuracy_score(y,prediction)),
                  f1_macro=float(f1_score(y,prediction,average="macro",zero_division=0)))
    if len(np.unique(y))==2 and scores is not None:
        scores=np.asarray(scores)
        result["roc_auc"] = float(roc_auc_score(y,scores[:,1] if scores.ndim==2 else scores))
    return result


def regression_metrics(y,prediction):
    return dict(rmse=float(np.sqrt(mean_squared_error(y,prediction))),
                mae=float(mean_absolute_error(y,prediction)),r2=float(r2_score(y,prediction)))


def tolerance(job): return 3e-5 if job["framework"]=="cirq" else 3e-9


def bank_audit(job, bank, operations, values):
    """All norms, plus full amplitude vectors for first and last batch rows."""
    n=job["n_qubits"]; tol=tolerance(job)
    if bank.shape!=(len(values),1 << n) or not np.isfinite(bank).all(): raise ValueError("invalid state bank")
    norm_error=max(abs(float(np.vdot(vector,vector).real)-1) for vector in bank)
    selected=sorted(set([0,len(values)-1])); error=0.
    for i in selected:
        error=max(error,align_error(bank[i],reference_state(n,operations,values[i])))
    if norm_error>tol or error>tol: raise RuntimeError(f"state audit failed: amplitude={error}, norm={norm_error}")
    return dict(max_amplitude_error=error,max_norm_error=norm_error,tolerance=tol,
                vector_audit="full amplitude vectors for first/last rows; all batch norms")


def adapter_metadata(adapter):
    return dict(internal_precision=adapter.precision,backend=adapter.backend,
                batch_strategy=adapter.batch_strategy,circuit_calls=adapter.calls,
                backend_execute_total_s=adapter.execute_s)


def timed_repeat(function, job):
    for _ in range(job["warmups"]): function()
    times=[]
    for _ in range(job["inner_repetitions"]):
        start=time.perf_counter(); result=function(); times.append(time.perf_counter()-start)
    return result,dict(primary_runtime_s=float(np.median(times)),inner_runtime_s=times,
                       peak_measured_stage_mb=peak_mb())


def kernel_job(job):
    data=np.load(job["data_path"]); n=job["n_qubits"]; name=job["feature_map"]
    x,y=data["X_train"],data["X_test"]
    row_cap=job.get("kernel_row_cap")
    if row_cap: x=x[:int(row_cap)]; y=y[:int(row_cap)]
    rows_x=[feature_values(row,name) for row in x]; rows_y=[feature_values(row,name) for row in y]
    analytical=job["framework"]=="analytical_angle"
    prep=time.perf_counter()
    adapter=None if analytical else Adapter(job["framework"],n,feature_spec(n,name))
    prepare_s=time.perf_counter()-prep
    if adapter:
        adapter.states(rows_x[:1]); adapter.reset()
    else: angle_kernel(x[:1])
    start=time.perf_counter()
    if analytical:
        kt,kv=angle_kernel(x),angle_kernel(y,x); encode_s=0.; kernel_s=time.perf_counter()-start
    else:
        kt,kv,kernel_info=quantum_kernel_matrices(adapter,rows_x,rows_y,
            job["max_statebank_bytes"],job.get("kernel_block_rows",128))
        encode_s=kernel_info["encoding_runtime_s"];kernel_s=kernel_info["kernel_algebra_s"]
    primary=time.perf_counter()-start; peak=peak_mb()
    result=dict(primary_runtime_s=primary,encoding_runtime_s=encode_s,kernel_algebra_s=kernel_s,
                train_samples=len(x),test_samples=len(y),kernel_row_cap=(int(row_cap) if row_cap else None),
                circuit_prepare_s=prepare_s,peak_measured_stage_mb=peak,
                kernel_probe=kt[:6,:6].tolist(),test_kernel_probe=kv[:3,:6].tolist(),
                kernel_diagonal_error=float(np.max(np.abs(np.diag(kt)-1))),
                kernel_min_eigenvalue=float(np.linalg.eigvalsh((kt+kt.T)/2).min()))
    if result["kernel_diagonal_error"]>5*tolerance(job) or result["kernel_min_eigenvalue"] < -5*tolerance(job):
        raise RuntimeError("kernel normalization or PSD check failed")
    if adapter:
        result.update(adapter_metadata(adapter));result.update(kernel_info)
        if kernel_info["all_encoded_state_norm_error"]>tolerance(job):
            raise RuntimeError("Kernel state-block normalization failed")
        error=0.;norm=kernel_info["all_encoded_state_norm_error"]
        for rows in [rows_x,rows_y]:
            for index in sorted(set([0,len(rows)-1])):
                probe=adapter.states([rows[index]])
                audit=bank_audit(job,probe,adapter.operations,[rows[index]])
                error=max(error,audit["max_amplitude_error"]);norm=max(norm,audit["max_norm_error"])
                del probe
        result.update(max_amplitude_error=error,max_norm_error=norm,tolerance=tolerance(job),
            vector_audit="All encoded-state norms; independently replayed full first/last train/test vectors after timing")
    else:
        # Every analytical value is audited against an independent product
        # RY/CNOT state construction, not against this closed-form function.
        class IndependentKernel:
            def __init__(self): self.n=n
            def states(self,rows):
                return np.stack([reference_state(n,feature_spec(n,"angle"),row) for row in rows]) if rows else np.empty((0,1 << n),complex)
        reference,_,_=quantum_kernel_matrices(IndependentKernel(),rows_x[:6],[],
            job["max_statebank_bytes"],job.get("kernel_block_rows",128))
        error=float(np.max(np.abs(kt[:6,:6]-reference)))
        if error>3e-9: raise RuntimeError("analytical Angle kernel mismatch")
        result.update(internal_precision="float64 analytical",backend="exact product-cosine Angle kernel",
                      batch_strategy="separate classical algorithm",circuit_calls=0,max_kernel_error=error,
                      kernel_strategy="analytical_with_blocked_reference_audit")
    before=time.perf_counter()
    if job["task"]=="qsvm":
        model=SVC(C=1,kernel="precomputed",decision_function_shape="ovr",random_state=job["seed"])
        model.fit(kt,data["y_train"]); prediction=model.predict(kv); scores=model.decision_function(kv)
        result.update(classification_metrics(data["y_test"],prediction,scores))
    elif job["task"]=="qkrr":
        model=KernelRidge(alpha=.1,kernel="precomputed").fit(kt,data["y_train_scaled"])
        prediction=model.predict(kv)*float(data["target_scale"])+float(data["target_mean"])
        result.update(regression_metrics(data["y_test"],prediction))
    else:
        mean=kt.mean(axis=0); grand=float(kt.mean())
        centered=kt-mean[None,:]-mean[:,None]+grand
        centered_test=kv-mean[None,:]-kv.mean(axis=1)[:,None]+grand
        eigenvalues,eigenvectors=np.linalg.eigh((centered+centered.T)/2)
        order=np.argsort(eigenvalues)[::-1]; order=order[eigenvalues[order]>1e-8][:3]
        if not len(order): raise RuntimeError("QKPCA has no positive components")
        vectors=eigenvectors[:,order]; roots=np.sqrt(eigenvalues[order])
        train_embedding=vectors*roots; test_embedding=centered_test@vectors/roots
        model=LogisticRegression(C=1,max_iter=1000).fit(train_embedding,data["y_train"])
        prediction=model.predict(test_embedding); scores=model.predict_proba(test_embedding)
        result.update(classification_metrics(data["y_test"],prediction,scores),
                      qkpca_components=len(order),qkpca_top_eigenvalues=eigenvalues[order].tolist(),
                      qkpca_variance_fraction=float(eigenvalues[order].sum()/np.maximum(eigenvalues,0).sum()))
    fit_s=time.perf_counter()-before
    result.update(classical_stage_s=fit_s,end_to_end_s=primary+fit_s,predictions=prediction.tolist())
    return result


def classical_job(job):
    data=np.load(job["data_path"]); x,y=data["X_train"],data["y_train"]
    kind=job["kind"]; name=job["model"]
    constructors={"rbf_svc":lambda:SVC(C=1,kernel="rbf",gamma="scale",random_state=job["seed"]),
                  "logistic":lambda:LogisticRegression(C=1,max_iter=1000),
                  "rf_classifier":lambda:RandomForestClassifier(n_estimators=100,n_jobs=1,random_state=job["seed"]),
                  "ridge":lambda:Ridge(alpha=1.),
                  "rbf_krr":lambda:KernelRidge(alpha=.1,kernel="rbf",gamma=1/x.shape[1]),
                  "rf_regressor":lambda:RandomForestRegressor(n_estimators=100,n_jobs=1,random_state=job["seed"])}
    start=time.perf_counter(); model=constructors[name]().fit(x,y); prediction=model.predict(data["X_test"])
    result=dict(predictions=prediction.tolist(),internal_precision="classical estimator",
                train_samples=len(x),test_samples=len(data["X_test"]))
    if kind=="classification":
        score=model.decision_function(data["X_test"]) if hasattr(model,"decision_function") else model.predict_proba(data["X_test"])
        result.update(classification_metrics(data["y_test"],prediction,score))
    else: result.update(regression_metrics(data["y_test"],prediction))
    elapsed=time.perf_counter()-start
    result.update(primary_runtime_s=elapsed,end_to_end_s=elapsed,peak_measured_stage_mb=peak_mb())
    if kind=="classification":
        pca=data["pca_components"]; raw=StandardScaler().fit(data["raw_train"])
        # A second classical control retains every original feature. It is
        # an information-content comparison, not the matched-input ranking.
        full_model=constructors[name](); before=time.perf_counter()
        full_model.fit(raw.transform(data["raw_train"]),y)
        full_prediction=full_model.predict(raw.transform(data["raw_test"]))
        result.update(full_feature_runtime_s=time.perf_counter()-before,
                      full_feature_accuracy=float(accuracy_score(data["y_test"],full_prediction)))
    return result


def evaluator(job, operations, output, terms, weights=None):
    native=job["track"]=="native_readout"
    before=time.perf_counter()
    adapter=Adapter(job["framework"],job["n_qubits"],operations,
                    output=output if native else "state",terms=terms,weights=weights)
    prepare_s=time.perf_counter()-before
    if not native and weights is None and output=="mean_z": weights=mean_z_weights(job["n_qubits"])
    chunk=state_chunk_rows(job["n_qubits"],job["max_statebank_bytes"],simultaneous_banks=2)
    def evaluate(rows):
        rows=list(rows);outputs=[]
        for start in range(0,len(rows),chunk):
            block=rows[start:start+chunk]
            if native: values=adapter.readouts(block)
            else:
                bank=adapter.states(block)
                values=np.abs(bank)**2@weights if weights is not None else pauli_expectations(bank,terms,job["n_qubits"])
                del bank
            outputs.extend(np.asarray(values,dtype=float))
        return np.asarray(outputs,dtype=float)
    return adapter,evaluate,prepare_s


def variational_ml(job):
    data=np.load(job["data_path"]); n=job["n_qubits"]; layers=job["depth"]
    x,xt=data["X_train"],data["X_test"]; regression=job["task"]=="vqr"
    y=data["y_train_scaled"] if regression else data["y_train"]
    classes=len(np.unique(y)) if not regression else 1
    units=1 if classes<=2 else classes; width=2*n*layers
    rng=np.random.default_rng(job["seed"])
    initial=rng.normal(0,.15,units*(width+1)+(1 if regression else 0))
    if regression: initial[-1]=.5
    adapter,evaluate,prep=evaluator(job,feature_spec(n,job["feature_map"],layers),"mean_z",mean_z_terms(n))
    def scores(theta,inputs):
        result=[]
        for unit in range(units):
            offset=unit*(width+1); angles=theta[offset:offset+width]
            rows=[feature_values(row,job["feature_map"],angles) for row in inputs]
            z=evaluate(rows)
            result.append((theta[-1] if regression else 2.)*z+theta[offset+width])
        values=np.column_stack(result)
        if regression: return values[:,0]
        return np.column_stack([1-expit(values[:,0]),expit(values[:,0])]) if units==1 else softmax(values,axis=1)
    def objective(theta,indices):
        indices=np.arange(len(x)) if indices is None else indices
        predicted=scores(theta,x[indices])
        if regression: return float(np.mean((predicted-y[indices])**2))
        return float(-np.mean(np.log(np.clip(predicted[np.arange(len(indices)),y[indices].astype(int)],1e-9,1))))
    scores(initial,x[:1]); adapter.reset()
    batches=[rng.choice(len(x),min(job["batch_size"],len(x)),replace=False) for _ in range(job["iterations"])]
    start=time.perf_counter(); initial_loss=objective(initial,None)
    final,trace=spsa(initial,objective,job["iterations"],job["seed"]+501,batches,
                     trace_objective=lambda theta:objective(theta,None))
    final_loss=trace[-1] if trace else initial_loss; training=time.perf_counter()-start; peak=peak_mb()
    before=time.perf_counter(); predicted=scores(final,xt); inference=time.perf_counter()-before
    if regression:
        prediction=predicted*float(data["target_scale"])+float(data["target_mean"])
        quality=regression_metrics(data["y_test"],prediction)
    else:
        prediction=predicted.argmax(axis=1); quality=classification_metrics(data["y_test"],prediction,predicted)
    # Native reductions are checked against independently propagated states
    # at the final parameters, outside the measured training and inference.
    rows=[feature_values(x[0],job["feature_map"],final[:width])]
    expected=pauli_expectations([reference_state(n,adapter.operations,rows[0])],mean_z_terms(n),n)[0]
    actual=evaluate(rows)[0]; error=float(abs(actual-expected))
    if error>tolerance(job): raise RuntimeError("variational final expectation mismatch")
    return dict(primary_runtime_s=training,inference_runtime_s=inference,end_to_end_s=training+inference,
        circuit_prepare_s=prep,peak_measured_stage_mb=peak,initial_loss=initial_loss,final_loss=final_loss,
        optimization_trace=[initial_loss]+trace,trace_kind="full training loss at each iterate; "+
            ("full-training-partition" if job.get("training_batch_mode")=="full_train_partition" else "mini-batch")+" SPSA perturbations",
        predictions=prediction.tolist(),learned_parameters=final.tolist(),native_readout_error=error,
        **quality,**adapter_metadata(adapter))


def optimization(job):
    n=job["n_qubits"]; rng=np.random.default_rng(job["seed"])
    if job["task"]=="vqe":
        terms=tfim_terms(n); op=feature_spec(n,"angle",job["depth"]); width=2*n*job["depth"]
        initial=rng.normal(0,.2,width)
        reference_start=time.perf_counter();ground=tfim_ground_energy(n)
        ground_reference_s=time.perf_counter()-reference_start
        adapter,evaluate,prep=evaluator(job,op,"pauli",terms)
        values=lambda theta:feature_values(np.zeros(n),"angle",theta)
        weights=None
    else:
        if job["task"]=="qaoa_fs":
            data=np.load(job["data_path"]); a,b,constant=data["qubo_a"],data["qubo_b"],float(data["qubo_constant"])
            h,pairs,offset=qubo_to_ising(a,b,constant); weights=data["qubo_cost"]
        else:
            edges=[(q,(q+1)%n,float(rng.uniform(.5,1.5))) for q in range(n)]
            if job["dataset"]=="erdos_renyi":
                edges=[(i,j,float(rng.uniform(.5,1.5))) for i in range(n) for j in range(i+1,n) if rng.random()<.6]
                if not edges: edges=[(0,1,1.)]
            h=np.zeros(n); pairs=[(i,j,w/2) for i,j,w in edges]; offset=-sum(w for _,_,w in edges)/2
            weights=np.empty(1 << n,dtype=float)
            for start in range(0,1 << n,1 << 18):
                stop=min(start+(1 << 18),1 << n);indices=np.arange(start,stop,dtype=np.uint64)
                cost=np.zeros(len(indices))
                for i,j,w in edges: cost-=w*(((indices >> i)^(indices >> j))&1)
                weights[start:stop]=cost
        terms=ising_terms(n,h,pairs,offset); initial=rng.uniform(.2,.7,2*job["depth"])
        op,_=qaoa_spec_values(n,h,pairs,job["depth"],initial)
        values=lambda theta:qaoa_spec_values(n,h,pairs,job["depth"],theta)[1]
        adapter,evaluate,prep=evaluator(job,op,"qubo",terms,weights)
        ground=float(weights.min())
    def objective(theta,unused=None): return float(evaluate([values(theta)])[0])
    objective(initial); adapter.reset()
    start=time.perf_counter(); first=objective(initial)
    final,trace=spsa(initial,objective,job["iterations"],job["seed"]+700)
    energy=trace[-1] if trace else first; elapsed=time.perf_counter()-start; peak=peak_mb()
    # The same final state export is outside primary native-readout training
    # for every backend, so solution diagnostics do not change that contract.
    before=time.perf_counter(); export=Adapter(job["framework"],n,op)
    bank=export.states([values(final)]); diagnostics_s=time.perf_counter()-before
    audit=bank_audit(job,bank,op,[values(final)])
    expected=float(pauli_expectations(bank,terms,n)[0]); native_error=abs(expected-energy)
    scale=max(1.,sum(abs(c) for _,c in terms))
    if native_error>tolerance(job)*scale or energy<ground-tolerance(job)*scale: raise RuntimeError("optimization energy audit failed")
    result=dict(primary_runtime_s=elapsed,end_to_end_s=elapsed+diagnostics_s,circuit_prepare_s=prep,
        solution_diagnostics_s=diagnostics_s,peak_measured_stage_mb=peak,initial_energy=first,
        final_energy=energy,ground_energy=ground,energy_gap=energy-ground,
        optimization_trace=[first]+trace,learned_parameters=final.tolist(),
        native_readout_error=float(native_error),energy_scale=scale,**audit,**adapter_metadata(adapter))
    if job["task"]=="vqe":
        result.update(ground_reference_method="exact open-chain Jordan-Wigner singular values (n by n)",
                      ground_reference_s=ground_reference_s)
    if weights is not None:
        probabilities=np.abs(bank[0])**2; modal=int(probabilities.argmax())
        result.update(modal_bitstring=modal,modal_cost=float(weights[modal]),
            optimum_probability=float(probabilities[np.isclose(weights,ground,atol=1e-10)].sum()))
        if job["task"]=="maxcut": result["approximation_ratio"]=float(energy/ground) if ground<0 else None
        else:
            features=data["selected_original_features"]
            selected=[int(features[q]) for q in range(len(features)) if (modal>>q)&1]
            result.update(selected_features=selected,selected_feature_count=len(selected),
                target_cardinality=int(data["cardinality"]),active_feature_variables=len(features),
                feature_selection_ancillas=n-len(features))
            before=time.perf_counter()
            if selected:
                model=make_pipeline(StandardScaler(),LogisticRegression(C=1,max_iter=1000))
                model.fit(data["raw_train"][:,selected],data["y_train"])
                prediction=model.predict(data["raw_test"][:,selected])
            else:
                model=DummyClassifier(strategy="most_frequent").fit(data["raw_train"],data["y_train"])
                prediction=model.predict(data["raw_test"])
            result.update(fs_downstream_accuracy=float(accuracy_score(data["y_test"],prediction)),
                          fs_downstream_runtime_s=time.perf_counter()-before)
    return result


def gradient_job(job):
    n=job["n_qubits"]; rng=np.random.default_rng(job["seed"])
    x=rng.uniform(-1,1,n); theta=rng.normal(0,.2,2*n*job["depth"])
    op=feature_spec(n,"angle",job["depth"]); values=feature_values(x,"angle",theta)
    keys=["t"+str(i) for i in range(len(theta))]
    shifted=parameter_shift_values(values,keys)
    adapter,evaluate,prep=evaluator(job,op,"mean_z",mean_z_terms(n))
    def execute():
        outcomes=evaluate(shifted); return (outcomes[::2]-outcomes[1::2])/2
    gradient,timing=timed_repeat(execute,job)
    finite=[]; epsilon=1e-6
    for key in keys:
        plus,minus=dict(values),dict(values); plus[key]+=epsilon; minus[key]-=epsilon
        a=pauli_expectations([reference_state(n,op,plus)],mean_z_terms(n),n)[0]
        b=pauli_expectations([reference_state(n,op,minus)],mean_z_terms(n),n)[0]
        finite.append((a-b)/(2*epsilon))
    error=float(np.max(np.abs(gradient-np.array(finite))))
    if error>tolerance(job)*3: raise RuntimeError(f"parameter-shift gradient mismatch: {error}")
    return dict(**timing,circuit_prepare_s=prep,gradient=gradient.tolist(),max_gradient_error=error,
                shifted_circuits=2*len(keys),**adapter_metadata(adapter))


def circuit_job(job):
    n=job["n_qubits"]; op,base=circuit_spec(n,job["family"],job["depth"],job["seed"])
    rng=np.random.default_rng(job["seed"]+111)
    rows=[{k:(float(v+rng.normal(0,.05)) if job["family"] not in ["qft","ghz"] else v)
           for k,v in base.items()} for _ in range(job["batch_size"])]
    before=time.perf_counter(); adapter=Adapter(job["framework"],n,op); prep=time.perf_counter()-before
    chunk=state_chunk_rows(n,job["max_statebank_bytes"])
    def execute():
        norm=0.
        for start in range(0,len(rows),chunk):
            bank=adapter.states(rows[start:start+chunk])
            if not np.isfinite(bank).all():raise RuntimeError("Nonfinite circuit state block")
            for vector in bank:norm=max(norm,abs(float(np.vdot(vector,vector).real)-1.))
            del bank
        return norm
    norm,timing=timed_repeat(execute,job);metadata=adapter_metadata(adapter)
    error=0.
    for index in sorted(set([0,len(rows)-1])):
        bank=adapter.states([rows[index]])
        audit=bank_audit(job,bank,op,[rows[index]])
        error=max(error,audit["max_amplitude_error"]);norm=max(norm,audit["max_norm_error"])
        del bank
    if norm>tolerance(job):raise RuntimeError("Circuit stream normalization failed")
    return dict(**timing,circuit_prepare_s=prep,gate_count=len(op),
                statebank_bytes=min(chunk,len(rows))*(1 << n)*16,
                logical_statebank_bytes=len(rows)*(1 << n)*16,state_chunk_rows=min(chunk,len(rows)),
                output_contract="complex128 state exports in bounded chunks; all-state norm reduction inside timing",
                throughput_circuits_s=len(rows)/timing["primary_runtime_s"],
                max_amplitude_error=error,max_norm_error=norm,tolerance=tolerance(job),
                vector_audit="Every measured state norm; first/last full vectors independently replayed after timing",**metadata)


def validation_job(job):
    n=job["n_qubits"]; rng=np.random.default_rng(12345); checks=[]
    for family in ["angle","phase","pauli","zz","ghz","random","qft"]:
        op,base=circuit_spec(n,family,2,12345)
        rows=[base,{k:float(v+.17) for k,v in base.items()}]
        adapter=Adapter(job["framework"],n,op); bank=adapter.states(rows)
        checks.append(dict(case=family,passed=True,**bank_audit(job,bank,op,rows)))
    op=[("X",[0],None)]; bank=Adapter(job["framework"],n,op).states([{}])
    if np.argmax(np.abs(bank[0]))!=1: raise RuntimeError("q0=LSB convention failed")
    checks.append(dict(case="asymmetric_q0_basis",passed=True,**bank_audit(job,bank,op,[{}])))
    terms=tfim_terms(n)+[([2]+[0]*(n-1),.13)]
    op,base=circuit_spec(n,"pauli",1,555)
    reference=reference_state(n,op,base); expected=float(pauli_expectations([reference],terms,n)[0])
    native=Adapter(job["framework"],n,op,output="pauli",terms=terms)
    actual=float(native.readouts([base])[0]); error=abs(actual-expected)
    if error>tolerance(job)*sum(abs(c) for _,c in terms): raise RuntimeError("mixed X/Y/Z native observable mismatch")
    checks.append(dict(case="mixed_XYZ_observable",passed=True,native_readout_error=error,tolerance=tolerance(job)))
    mz=Adapter(job["framework"],n,op,output="mean_z",terms=mean_z_terms(n))
    rows=[base,{k:float(v+.13) for k,v in base.items()}]
    expected=pauli_expectations([reference_state(n,op,row) for row in rows],mean_z_terms(n),n)
    error=float(np.max(np.abs(mz.readouts(rows)-expected)))
    if error>tolerance(job): raise RuntimeError("native mean-Z batch mismatch")
    checks.append(dict(case="native_mean_z_batch",passed=True,native_readout_error=error,tolerance=tolerance(job)))
    diagonal=mean_z_weights(n); native=Adapter(job["framework"],n,op,output="qubo",terms=mean_z_terms(n),weights=diagonal)
    error=float(np.max(np.abs(native.readouts(rows)-expected)))
    if error>tolerance(job): raise RuntimeError("native diagonal batch mismatch")
    checks.append(dict(case="native_diagonal_batch",passed=True,native_readout_error=error,tolerance=tolerance(job)))
    gradients=gradient_job({**job,"track":"native_readout","depth":1})
    checks.append(dict(case="parameter_shift_vs_finite_difference",passed=True,max_gradient_error=gradients["max_gradient_error"]))
    return dict(checks=checks,internal_precision="complex64" if job["framework"]=="cirq" else "complex128")


def preflight_job(job):
    """Exercise real native extensions before launching the larger campaign."""
    n=job["n_qubits"]
    operations=[("RY",[0],"input"),("H",[0],None),("CX",[0,1],None),("RY",[1],"a"),("RZ",[0],"b")]
    rows=[dict(input=.61,a=.31,b=-.27),dict(input=-.33,a=-.46,b=.19)]
    state_adapter=Adapter(job["framework"],n,operations)
    bank=state_adapter.states(rows)
    audit=bank_audit(job,bank,operations,rows)
    reference=np.asarray([reference_state(n,operations,row) for row in rows])
    native=Adapter(job["framework"],n,operations,output="mean_z",terms=mean_z_terms(n),weights=mean_z_weights(n))
    error=float(np.max(np.abs(native.readouts(rows)-pauli_expectations(reference,mean_z_terms(n),n))))
    if error>tolerance(job): raise RuntimeError(f"native readout smoke mismatch: {error}")
    modules={
        "qiskit":["qiskit","qiskit_aer"],"pennylane":["pennylane","pennylane_lightning"],
        "cirq":["cirq","qsimcirq"],"cirq_f64":["cirq"],"qulacs":["qulacs"],
    }.get(job["framework"],["superfermion","superfermion._sf_core"])
    return dict(**audit,native_readout_error=error,**adapter_metadata(state_adapter),
        python_executable=sys.executable,python_version=sys.version,
        isolated_python=bool(sys.flags.isolated),user_site_disabled=bool(sys.flags.no_user_site),
        imported_module_files={name:str(getattr(importlib.import_module(name),"__file__",None)) for name in modules},
        package_versions={package:importlib.metadata.version(package) for package in
            ["numpy","scipy","sympy","autoray","cirq-core","qsimcirq","pennylane","pennylane-lightning"]})


def run(job):
    task=job["task"]
    if task=="preflight": return preflight_job(job)
    if task=="validation": return validation_job(job)
    if task in ["qsvm","qkrr","qkpca"]: return kernel_job(job)
    if task=="classical": return classical_job(job)
    if task in ["vqc","vqr"]: return variational_ml(job)
    if task in ["qaoa_fs","maxcut","vqe"]: return optimization(job)
    if task=="gradient": return gradient_job(job)
    if task=="circuit": return circuit_job(job)
    raise ValueError(task)


if __name__=="__main__":
    job=json.loads(Path(sys.argv[1]).read_text()); result=dict(job)
    entry_rss,entry_source=current_rss_mib()
    try:
        result.update(run(job),status="ok",memory_metric=("Windows psutil peak_wset at measured-stage end" if os.name=="nt" else "Linux process HWM at measured-stage end")+"; includes imports/preparation/warmup; not statevector-only memory",
                      memory_discovery_source=legacy_adapter.MEMORY_DISCOVERY_SOURCE)
    except Exception as error:
        result.update(status="error",error=str(error),traceback=traceback.format_exc())
    exit_rss,exit_source=current_rss_mib()
    result.update(
        worker_entry_rss_mib=entry_rss,
        worker_exit_rss_mib=exit_rss,
        worker_peak_rss_mib=peak_mb(),
        primary_stage_peak_rss_mib=result.get("peak_measured_stage_mb"),
        worker_rss_change_mib=(exit_rss-entry_rss if entry_rss is not None and exit_rss is not None else None),
        worker_entry_rss_source=entry_source,
        worker_exit_rss_source=exit_source,
        worker_memory_metric=("Windows psutil peak_wset at job end" if os.name=="nt" else "Linux process high-water RSS at job end")+", before JSON serialization; includes imports, preparation, warmup, fitting, inference, diagnostics and audits that the job performs",
        ram_units="MiB (2**20 bytes)",
    )
    path=Path(sys.argv[2]); temporary=path.with_suffix(".tmp"); temporary.write_text(json.dumps(result,indent=2,allow_nan=False)); temporary.replace(path)
    print(json.dumps({"task":job["task"],"framework":job["framework"],"status":result["status"]}))
    if result["status"]!="ok": raise SystemExit(1)
'''
(EXPERIMENT_ROOT/'study_worker.py').write_bytes(MODULE_SOURCE.lstrip("\n").encode("utf-8"))
print("Wrote study_worker.py")

Wrote study_worker.py


#### Fresh-process pairing, fingerprints and checkpoints

Open this source cell to inspect or modify `study_driver.py`. Its SHA256 is recorded before timing.

In [11]:
MODULE_SOURCE = r'''
"""Paired fresh-process campaigns, resumable fingerprints, and strict validity gates."""
import hashlib
import itertools
import json
import os
from pathlib import Path
import random
import subprocess
import sys
import time

import pandas as pd
from experiment_core import state_chunk_rows


SCRIPT_NAMES=["legacy_adapter.py","experiment_core.py","prepare_study_data.py",
              "study_worker.py","study_driver.py","analyze_study.py","offline_checks.py"]

# -I ignores the notebook's PYTHONPATH/PYTHONHOME/user site and warning overrides.
# Only the explicitly selected library site and the benchmark scripts are added.
WORKER_BOOTSTRAP = """
import pathlib, runpy, sys
worker, job, output, site = sys.argv[1:]
sys.path.insert(0, str(pathlib.Path(worker).resolve().parent))
if site:
    sys.path.insert(0, site)
sys.argv = [worker, job, output]
runpy.run_path(worker, run_name="__main__")
"""
REPORT_BOOTSTRAP = """
import pathlib, runpy, sys
script, config = sys.argv[1:]
sys.path.insert(0, str(pathlib.Path(script).resolve().parent))
sys.argv = [script, config]
runpy.run_path(script, run_name="__main__")
"""


def digest(value):
    return hashlib.sha256(json.dumps(value,sort_keys=True,separators=(",",":"),allow_nan=False).encode()).hexdigest()


def environment(config,framework,threads):
    env=os.environ.copy()
    for key in ["PYTHONPATH", "PYTHONHOME", "PYTHONSTARTUP", "PYTHONWARNINGS"]:
        env.pop(key, None)
    env.update(PYTHONNOUSERSITE="1", PYTHONUNBUFFERED="1")
    if framework=="classical": threads=1
    for k in ["OPENBLAS_NUM_THREADS","MKL_NUM_THREADS","NUMEXPR_NUM_THREADS","VECLIB_MAXIMUM_THREADS"]: env[k]="1"
    env.update(OMP_NUM_THREADS=str(threads),OMP_DYNAMIC="FALSE",RAYON_NUM_THREADS=str(threads),
        QULACS_NUM_THREADS=str(threads),SIM_THREADS=str(threads),SF_F32="0",PYTHONHASHSEED="0")
    for key in list(env):
        if key.startswith("SF_") and key!="SF_F32": env.pop(key)
    return env


def invoke(config,job,force=False):
    root=Path(config["root"]); env=environment(config,job["framework"],job["threads"])
    job={**job,"warmups":config["profile"]["warmups"],"inner_repetitions":config["profile"]["inner_repetitions"],
         "max_statebank_bytes":config["max_statebank_bytes"],
         "kernel_block_rows":config["profile"].get("kernel_block_rows",128),
         "kernel_row_cap":(config.get("kernel_row_caps") or {}).get(job.get("n_qubits")),
         "manifest_sha256":config["manifest_sha256"]}
    fingerprint=dict(job=job,script_sha256={name:hashlib.sha256((root/name).read_bytes()).hexdigest() for name in SCRIPT_NAMES},
        data_sha256=hashlib.sha256(Path(job["data_path"]).read_bytes()).hexdigest() if "data_path" in job else None,
        thread_environment={k:env.get(k) for k in ["OMP_NUM_THREADS","OMP_DYNAMIC","RAYON_NUM_THREADS","QULACS_NUM_THREADS","SIM_THREADS","SF_F32","OPENBLAS_NUM_THREADS"]})
    job["cache_key"]=digest(fingerprint); job["job_id"]=job["cache_key"][:24]
    directory=root/"raw"; directory.mkdir(exist_ok=True)
    path=directory/(job["job_id"]+".json"); job_path=directory/(job["job_id"]+".config.json")
    log_path=directory/(job["job_id"]+".log")
    if path.exists() and not force:
        old=json.loads(path.read_text())
        if old.get("status")=="ok" and old.get("cache_key")==job["cache_key"]: return old
    job_path.write_text(json.dumps(job,indent=2)); path.unlink(missing_ok=True)
    before = time.monotonic()
    with log_path.open("w") as log:
        try:
            execution=subprocess.run([config["python"],"-I","-c",WORKER_BOOTSTRAP,
                str(root/"study_worker.py"),str(job_path),str(path),config["sites"].get(job["framework"],"")],cwd=root,env=env,
                stdout=log,stderr=subprocess.STDOUT,timeout=config["profile"]["timeout_s"])
        except subprocess.TimeoutExpired: execution=None
    if path.exists():
        result=json.loads(path.read_text())
        result.update(worker_log=str(log_path), worker_result=str(path),
                      worker_wall_time_s=time.monotonic()-before,
                      worker_returncode=execution.returncode if execution is not None else None)
        if execution is None:
            result.update(status="error", error="timeout")
        if execution is not None and execution.returncode!=0 and result.get("status")=="ok":
            result.update(status="error",error="worker reported success but exited nonzero")
        path.write_text(json.dumps(result,indent=2,allow_nan=False))
        return result
    result={**job,"status":"error","error":"timeout" if execution is None else f"worker exited {execution.returncode}",
            "log_tail":log_path.read_text(errors="replace")[-6000:],
            "worker_log":str(log_path),"worker_result":str(path),
            "worker_wall_time_s":time.monotonic()-before,
            "worker_returncode":execution.returncode if execution is not None else None}
    path.write_text(json.dumps(result,indent=2)); return result


def verify_manifest(config):
    root=Path(config["root"]); path=root/"manifest.json"
    if hashlib.sha256(path.read_bytes()).hexdigest()!=config["manifest_sha256"]: raise RuntimeError("manifest changed")
    manifest=json.loads(path.read_text())
    for name,expected in manifest["script_sha256"].items():
        if hashlib.sha256((root/name).read_bytes()).hexdigest()!=expected: raise RuntimeError(f"script changed: {name}; regenerate the manifest and rerun checks")
    inputs=json.loads((root/"data_manifest.json").read_text())
    if digest(inputs)!=digest(manifest["data"]): raise RuntimeError("dataset manifest changed")
    for key,path in inputs["paths"].items():
        if hashlib.sha256(Path(path).read_bytes()).hexdigest()!=inputs["sha256"][key]: raise RuntimeError(f"data changed: {key}")
    return inputs


def report_failures(root, results, name):
    failed = [result for result in results if result["status"] != "ok"]
    path = root/(name+"_errors.json")
    if not failed:
        path.unlink(missing_ok=True)
        return
    path.write_text(json.dumps(failed, indent=2))
    for result in failed:
        print(f"\nFAILED {result['framework']} / {result['n_qubits']} qubits: "
              f"{result.get('error', 'unknown error')}", flush=True)
        print(result.get("traceback", result.get("log_tail", "No traceback; inspect the worker log."))[-4000:], flush=True)
        print("Worker log:", result.get("worker_log", "unavailable"), flush=True)


def preflight(config):
    root=Path(config["root"]); results=[]
    for framework in config["frameworks"]:
        result=invoke(config,dict(task="preflight",framework=framework,dataset="backend_smoke",
            n_qubits=2,threads=1,repetition=0,seed=42,track="environment",depth=1,
            batch_size=2,iterations=1,feature_map="angle",family="preflight"))
        results.append(result)
        print(f"Import/native smoke {framework}: {result['status']}",flush=True)
    (root/"backend_preflight.json").write_text(json.dumps(results,indent=2))
    report_failures(root,results,"backend_preflight")
    if any(result["status"]!="ok" for result in results):
        raise RuntimeError("Backend preflight failed. The actual traceback is printed above and saved in backend_preflight_errors.json.")
    print("All eight installed backends passed isolated native-state/native-readout smoke checks.",flush=True)


def validate(config):
    root=Path(config["root"]); results=[]
    (root/"correctness_passed.json").unlink(missing_ok=True)
    (root/"correctness.csv").unlink(missing_ok=True)
    for n,framework in itertools.product(config["profile"]["validation_qubits"],config["frameworks"]):
        job=dict(task="validation",framework=framework,dataset="independent_reference",n_qubits=n,
            threads=1,repetition=0,seed=42,track="correctness",depth=1,batch_size=2,iterations=1,
            feature_map="angle",family="validation")
        result=invoke(config,job); results.append(result)
        print(f'Checked {framework}, {n} qubits: {result["status"]}',flush=True)
    (root/"validation_results.json").write_text(json.dumps(results,indent=2))
    failed=[r for r in results if r["status"]!="ok"]
    report_failures(root,results,"validation")
    if failed:
        raise RuntimeError("Independent correctness checks failed. The actual worker tracebacks are printed above and saved in validation_errors.json; no performance ranking will run.")
    checks=[dict(framework=r["framework"],n_qubits=r["n_qubits"],**case) for r in results for case in r["checks"]]
    pd.DataFrame(checks).to_csv(root/"correctness.csv",index=False)
    (root/"correctness_passed.json").write_text(json.dumps(dict(manifest_sha256=config["manifest_sha256"],
        configuration_sha256=digest(config),checks=len(checks))))
    print(f'Passed {len(checks)} independent full-state/native-observable/gradient checks.',flush=True)


def groups(config,inputs):
    p=config["profile"]; result=[]
    dataset_audit={(row["dataset"],row["n_qubits"],row["seed"]):row for row in inputs["datasets"]}
    optimization_steps=p.get("optimization_iterations",p["iterations"])
    ml_steps=p.get("variational_iterations",p["iterations"])
    def append(base,analytical=False):
        base={"dataset":"synthetic","n_qubits":p["qml_qubits"][0],
              "feature_map":"angle","family":"application","track":"statevector",
              "depth":1,"batch_size":p["training_batch"],"iterations":optimization_steps,
              "max_statebank_bytes":config["max_statebank_bytes"],
              "kernel_block_rows":p.get("kernel_block_rows",128),**base}
        problem={k:v for k,v in base.items() if k!="framework"}
        base["problem_id"]=digest(problem)[:20]
        frameworks=["classical"] if base.get("framework")=="classical" else list(config["frameworks"])
        jobs=[dict(base,framework=f) for f in frameworks]
        if analytical: jobs.append(dict(base,framework="analytical_angle",track="analytical"))
        result.append(jobs)
    outer=itertools.product(config["threads"],range(p["repetitions"]),p["seeds"])
    for threads,repetition,seed in outer:
        common=dict(threads=threads,repetition=repetition,seed=seed)
        for n in p["qml_qubits"]:
            condition=dict(common,n_qubits=n)
            for name in p["classification_datasets"]+p["regression_datasets"]:
                path=inputs["paths"][f"{name}:{n}:{seed}"]
                regression=name in p["regression_datasets"]
                for feature_map in p["kernel_maps"]:
                    append(dict(condition,task="qkrr" if regression else "qsvm",dataset=name,
                                feature_map=feature_map,data_path=path),analytical=feature_map=="angle")
                if name in p["qkpca_datasets"]:
                    append(dict(condition,task="qkpca",dataset=name,feature_map="zz",data_path=path))
                if name in p["feature_selection_datasets"] and dataset_audit[(name,n,seed)].get("feature_selection_available",True):
                    for track in ["statevector","native_readout"]:
                        append(dict(condition,task="qaoa_fs",dataset=name,track=track,
                                    depth=p["qaoa_depth"],data_path=path))
                models=["ridge","rbf_krr","rf_regressor"] if regression else ["rbf_svc","logistic","rf_classifier"]
                for model in models:
                    append(dict(condition,task="classical",dataset=name,framework="classical",model=model,
                                kind="regression" if regression else "classification",data_path=path))
            for graph,track in itertools.product(["ring","erdos_renyi"],["statevector","native_readout"]):
                append(dict(condition,task="maxcut",dataset=graph,track=track,depth=p["qaoa_depth"]))
        # VQC and VQR have a separate qubit grid and a separate iteration budget.
        for n in p.get("variational_qubits",p["qml_qubits"]):
            condition=dict(common,n_qubits=n)
            for name in p["vqc_datasets"]+p["vqr_datasets"]:
                path=inputs["paths"][f"{name}:{n}:{seed}"]
                for track,depth in itertools.product(["statevector","native_readout"],p["variational_depths"]):
                    batch=dataset_audit[(name,n,seed)]["train_samples"] if p.get("variational_full_batch",False) else p["training_batch"]
                    append(dict(condition,task="vqr" if name in p["vqr_datasets"] else "vqc",dataset=name,
                                track=track,depth=depth,data_path=path,iterations=ml_steps,batch_size=batch,
                                training_batch_mode="full_train_partition" if p.get("variational_full_batch",False) else "minibatch"))
        for n,track in itertools.product(p["vqe_qubits"],["statevector","native_readout"]):
            append(dict(common,task="vqe",dataset="TFIM_open_chain",n_qubits=n,track=track,depth=2))
        for n,track in itertools.product(p["gradient_qubits"],["statevector","native_readout"]):
            append(dict(common,task="gradient",n_qubits=n,track=track,depth=1))
        for n,family in itertools.product(p["state_qubits"],["ghz","qft","random"]):
            append(dict(common,task="circuit",family=family,n_qubits=n,batch_size=1,depth=2))
        for n,family in itertools.product(p["map_qubits"],["angle","phase","pauli","zz"]):
            append(dict(common,task="circuit",family=family,n_qubits=n,batch_size=p["map_batch"],depth=1))
        for n,batch in itertools.product(p.get("sweep_qubits",[p["batch_qubits"]]),p["batch_sizes"]):
            append(dict(common,task="circuit",family="angle",dataset="batch_sweep",n_qubits=n,batch_size=batch,depth=1))
        for n,depth in itertools.product(p.get("sweep_qubits",[p["batch_qubits"]]),p["circuit_depths"]):
            append(dict(common,task="circuit",family="random",dataset="depth_sweep",n_qubits=n,batch_size=1,depth=depth))
    return result



def preview(config,inputs):
    campaign=groups(config,inputs); jobs=[j for group in campaign for j in group]
    table=pd.DataFrame(jobs)
    dataset_sizes={(row["dataset"],row["n_qubits"],row["seed"]):
        row["train_samples"]+row["test_samples"] for row in inputs["datasets"]}
    maximum_split_sizes={(row["dataset"],row["n_qubits"],row["seed"]):
        max(row["train_samples"],row["test_samples"]) for row in inputs["datasets"]}
    counts=table.groupby(["task","framework"]).size().unstack(fill_value=0)
    counts.to_csv(Path(config["root"])/"planned_jobs.csv")
    # Exported-bank estimates follow the actual bounded execution strategy.
    # Backend workspaces, reductions, audits and other copies need additional RAM.
    budget_rows=[]
    for job in jobs:
        if job["task"]=="circuit":
            batch=job["batch_size"]
        elif job["task"] in ["qsvm","qkrr","qkpca"]:
            batch=dataset_sizes[(job["dataset"],job["n_qubits"],job["seed"])]
            kernel_cap=(config.get("kernel_row_caps") or {}).get(job["n_qubits"])
            if kernel_cap: batch=min(batch,2*int(kernel_cap))
        elif job["task"] in ["vqc","vqr"]:
            batch=maximum_split_sizes[(job["dataset"],job["n_qubits"],job["seed"])]
        else:
            batch=1
        logical_bank=batch*(1 << job["n_qubits"])*16
        if job["task"]=="classical":
            logical_bank=0;bank=0;strategy="classical_no_state_bank"
        elif job["task"] in ["qsvm","qkrr","qkpca"]:
            if job["framework"]=="analytical_angle":
                chunk=state_chunk_rows(job["n_qubits"],config["max_statebank_bytes"],2,config["profile"].get("kernel_block_rows",128))
                bank=min(6,2*chunk)*(1 << job["n_qubits"])*16
                strategy="analytical_with_blocked_reference_audit"
            elif logical_bank<=config["max_statebank_bytes"]:
                bank=logical_bank;strategy="retained_state_bank"
            else:
                chunk=state_chunk_rows(job["n_qubits"],config["max_statebank_bytes"],2,config["profile"].get("kernel_block_rows",128))
                bank=min(batch,2*chunk)*(1 << job["n_qubits"])*16
                strategy="bounded_reencoding_blocks"
        else:
            count=4*job["n_qubits"]*job["depth"] if job["task"]=="gradient" else batch
            chunk=state_chunk_rows(job["n_qubits"],config["max_statebank_bytes"],1 if job["task"]=="circuit" else 2)
            bank=min(count,chunk)*(1 << job["n_qubits"])*16
            strategy="chunked_state_exports" if job["task"]=="circuit" else "chunked_readout_evaluations"
        budget_rows.append(dict(task=job["task"],dataset=job["dataset"],framework=job["framework"],
            n_qubits=job["n_qubits"],track=job["track"],seed=job["seed"],repetition=job["repetition"],
            execution_strategy=strategy,logical_statebank_mib=logical_bank/2**20,
            estimated_statebank_mib=bank/2**20,limit_mib=config["max_statebank_bytes"]/2**20,
            limit_exceeded=bank>config["max_statebank_bytes"]))
    budgets=pd.DataFrame(budget_rows)
    budgets.to_csv(Path(config["root"])/"statebank_budget.csv",index=False)
    report=dict(jobs=len(jobs),paired_groups=len(campaign),profile=config["profile_name"],
                frameworks=config["frameworks"],fresh_process_repetitions=config["profile"]["repetitions"],
                split_seeds=config["profile"]["seeds"],max_statebank_bytes=config["max_statebank_bytes"],
                max_jobs=config["max_jobs"],job_limit_exceeded=len(jobs)>config["max_jobs"],
                statebank_limit_exceeded_jobs=int(budgets.limit_exceeded.sum()),
                kernel_row_caps=config.get("kernel_row_caps") or {},
                largest_estimated_statebank_mib=float(budgets.estimated_statebank_mib.max()))
    (Path(config["root"])/"campaign_plan.json").write_text(json.dumps(report,indent=2))
    print(json.dumps(report,indent=2)); print(counts.to_string())
    if budgets.limit_exceeded.any():
        print(budgets[budgets.limit_exceeded].head(8).to_string(index=False),flush=True)
        raise RuntimeError("Requested state banks exceed MAX_STATEBANK_MIB; campaign_plan.json and statebank_budget.csv contain the requirements. Review the qubit/sample/batch settings or memory budget before launching.")
    return campaign


def campaign(config,inputs,family=None):
    root=Path(config["root"]); gate=json.loads((root/"correctness_passed.json").read_text())
    if gate["manifest_sha256"]!=config["manifest_sha256"] or gate["configuration_sha256"]!=digest(config):
        raise RuntimeError("correctness gate belongs to a different build/configuration")
    campaign=groups(config,inputs)
    if family: campaign=[g for g in campaign if g[0]["task"] in family.split(",")]
    count=sum(len(g) for g in campaign)
    if count>config["max_jobs"]: raise RuntimeError(f"{count} jobs exceed MAX_JOBS={config['max_jobs']}; inspect the plan and raise the visible budget to run this profile")
    rng=random.Random(96331); rng.shuffle(campaign)
    if config.get("order_qubits_first",False):campaign.sort(key=lambda group:group[0]["n_qubits"])
    for group in campaign: rng.shuffle(group)
    rows=[]; before=time.monotonic()
    path=root/("results_"+(family.replace(",","_") if family else "all")+".json")
    path.write_text("[]")
    report_after=float(config.get("report_refresh_seconds",0))
    last_report=time.monotonic()
    print(f"Executing {count} jobs sequentially in fresh processes. Cached successes may resume.",flush=True)
    for i,job in enumerate([j for group in campaign for j in group],1):
        result=invoke(config,job); rows.append(result)
        if i%8==0 or result["status"]!="ok" or i==count:
            print(f'{i}/{count}: {job["task"]} {job["dataset"]} {job["framework"]}: {result["status"]}; elapsed {time.monotonic()-before:.1f}s',flush=True)
            path.write_text(json.dumps(rows,indent=2))
        if report_after>0 and time.monotonic()-last_report>=report_after:
            # Run between workers so report generation does not compete with a timed job.
            report_config=root/"periodic_report_config.json"
            report_config.write_text(json.dumps(config,indent=2))
            report_env=environment(config,"classical",1);report_env["MPLBACKEND"]="Agg"
            report_log=root/"periodic_reports.log"
            with report_log.open("a") as log:
                refresh=subprocess.run([config["python"],"-I","-c",REPORT_BOOTSTRAP,
                    str(root/"analyze_study.py"),str(report_config)],cwd=root,env=report_env,
                    stdout=log,stderr=subprocess.STDOUT)
            print("Time/RAM figures refreshed from checkpoints." if refresh.returncode==0 else
                  f"Report refresh failed ({refresh.returncode}); inspect {report_log.name}.",flush=True)
            last_report=time.monotonic()
        if result["status"]!="ok" and config.get("fail_fast",True):
            (root/"latest_failure.json").write_text(json.dumps(result,indent=2))
            report_failures(root,[result],"benchmark")
            raise RuntimeError("Benchmark failed; inspect latest_failure.json. Successful checkpoints are retained.")
    path.write_text(json.dumps(rows,indent=2))
    (root/("campaign_receipt_"+path.stem+".json")).write_text(json.dumps(dict(jobs=count,
        success=sum(r["status"]=="ok" for r in rows),failed=sum(r["status"]!="ok" for r in rows),
        manifest_sha256=config["manifest_sha256"],wall_time_s=time.monotonic()-before),indent=2))
    print(f"Saved {path.name}: {len(rows)} measured jobs.",flush=True)


if __name__=="__main__":
    config=json.loads(Path(sys.argv[1]).read_text()); inputs=verify_manifest(config)
    action=sys.argv[2]
    if action=="preflight": preflight(config)
    elif action=="validate": validate(config)
    elif action=="preview": preview(config,inputs)
    elif action=="run": campaign(config,inputs,sys.argv[3] if len(sys.argv)>3 else None)
    else: raise ValueError(action)
'''
(EXPERIMENT_ROOT/'study_driver.py').write_bytes(MODULE_SOURCE.lstrip("\n").encode("utf-8"))
print("Wrote study_driver.py")

Wrote study_driver.py


#### Editable statistics and plotting code

Open this source cell to inspect or modify `analyze_study.py`. Its SHA256 is recorded before timing.

In [12]:
MODULE_SOURCE = r'''
"""Inspectable medians, paired comparisons, outcome audits, and scientific figures."""
import json
import hashlib
import os
from pathlib import Path
import sys

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from study_driver import groups, verify_manifest

LABELS={"superfermion_optimized":"SF modified", "superfermion_source_baseline":"SF source",
        "superfermion_wheel":"SF published", "qiskit":"Aer f64", "pennylane":"Lightning f64",
        "cirq":"qsim f32", "cirq_f64":"Cirq f64", "qulacs":"Qulacs f64", "analytical_angle":"Analytical Angle"}
ORDER=["superfermion_optimized","superfermion_source_baseline","superfermion_wheel",
       "qiskit","pennylane","qulacs","cirq","cirq_f64"]
COLORS={"superfermion_optimized":"#174c78","superfermion_source_baseline":"#668aa8",
        "superfermion_wheel":"#a5b9c9","qiskit":"#a38120","pennylane":"#b96121",
        "qulacs":"#984d72","cirq":"#5c6269","cirq_f64":"#949ba1","analytical_angle":"#23272b"}
MARKERS=dict(zip(ORDER,["o","s","^","D","v","P","x","*"]))
CONDITION=["task","dataset","n_qubits","feature_map","family","track","depth","batch_size",
           "iterations","threads","seed","model"]
METRICS=["accuracy","balanced_accuracy","f1_macro","roc_auc","rmse","mae","r2","final_loss",
         "final_energy","ground_energy","energy_gap","approximation_ratio","optimum_probability",
         "fs_downstream_accuracy","qkpca_variance_fraction","max_gradient_error"]
RAM_METRICS={"worker_peak":"worker_peak_rss_mib",
             "primary_stage_peak":"primary_stage_peak_rss_mib",
             "entry_rss":"worker_entry_rss_mib", "exit_rss":"worker_exit_rss_mib",
             "rss_change":"worker_rss_change_mib"}
RAM_TASKS=["qsvm","qkrr","qkpca","vqc","vqr","qaoa_fs","maxcut","vqe","gradient","circuit"]
RAM_TASK_LABELS={"qsvm":"QSVM","qkrr":"QKRR","qkpca":"QKPCA","vqc":"VQC","vqr":"VQR",
                 "qaoa_fs":"QAOA FS","maxcut":"MaxCut","vqe":"VQE","gradient":"Gradient","circuit":"Circuit"}


def style():
    plt.rcParams.update({"figure.dpi":110,"savefig.dpi":150,"font.size":11,"axes.titlesize":12,
        "axes.labelsize":11,"xtick.labelsize":10,"ytick.labelsize":10,"figure.facecolor":"white",
        "axes.spines.top":False,"axes.spines.right":False,"axes.axisbelow":True,
        "grid.color":"#e2e5e8","grid.linewidth":.6,"lines.linewidth":1.6})


def load_results(root,manifest_sha):
    indexed={}
    for path in root.glob("results_*.json"):
        for row in json.loads(path.read_text()):
            if row.get("manifest_sha256")==manifest_sha: indexed[row["job_id"]]=row
    # Include per-job checkpoints when a campaign has not finished its section.
    for path in (root/"raw").glob("*.json"):
        if path.name.endswith(".config.json"):continue
        row=json.loads(path.read_text())
        if row.get("manifest_sha256")==manifest_sha and row.get("task") not in ["preflight","validation"] and "status" in row:
            indexed[row["job_id"]]=row
    return list(indexed.values())


def html_table(root,name,table,limit=100):
    table.to_csv(root/(name+".csv"),index=False)
    css="""<style>table {border-collapse:collapse; font:13px system-ui,sans-serif;} th,td{padding:6px 9px;text-align:left;border-bottom:1px solid #ddd;} th{background:#eef3f7;} </style>"""
    (root/(name+".html")).write_text(css+table.head(limit).to_html(index=False,float_format=lambda x:f"{x:.4g}"))


def aggregate(frame):
    result=[]
    for key,subset in frame.groupby(CONDITION+["framework"],dropna=False,sort=False):
        row=dict(zip(CONDITION+["framework"],key))
        x=subset["primary_runtime_s"].to_numpy()*1000
        row.update(median_ms=float(np.median(x)),q25_ms=float(np.quantile(x,.25)),q75_ms=float(np.quantile(x,.75)),
            repetitions=len(x),end_to_end_median_ms=float(subset.get("end_to_end_s",subset.primary_runtime_s).median()*1000),
            peak_process_hwm_mib=float(subset.peak_measured_stage_mb.median()))
        for metric in METRICS:
            if metric in subset: row[metric]=float(subset[metric].median()) if subset[metric].notna().any() else np.nan
        result.append(row)
    return pd.DataFrame(result,columns=CONDITION+["framework","median_ms","q25_ms","q75_ms",
        "repetitions","end_to_end_median_ms","peak_process_hwm_mib"]+METRICS)


def ram_aggregate(frame, keys):
    """Summarize per-job RSS measurements; never sum process peaks."""
    records=[]
    for key,part in frame.groupby(keys,dropna=False,sort=False):
        if not isinstance(key,tuple): key=(key,)
        record=dict(zip(keys,key))
        ok=part[part.status=="ok"]
        available=ok[np.isfinite(ok.worker_peak_rss_mib)&(ok.worker_peak_rss_mib>0)]
        record.update(jobs=len(part),successful_jobs=len(ok),failed_jobs=len(part)-len(ok),
            measured_jobs=len(available),missing_peak_jobs=len(ok)-len(available),
            conditions=int(available[CONDITION].drop_duplicates().shape[0]),
            datasets=int(available.dataset.nunique()),tasks=int(available.task.nunique()))
        for prefix,column in RAM_METRICS.items():
            values=available[column].dropna().to_numpy(dtype=float)
            values=values[np.isfinite(values)]
            record[prefix+"_measured_jobs"]=len(values)
            for suffix,quantile in [("min",0),("q25",.25),("median",.5),("q75",.75),("p95",.95),("max",1)]:
                record[f"{prefix}_{suffix}_mib"]=float(np.quantile(values,quantile)) if len(values) else np.nan
        records.append(record)
    columns=keys+["jobs","successful_jobs","failed_jobs","measured_jobs","missing_peak_jobs",
                  "conditions","datasets","tasks"]
    for prefix in RAM_METRICS:
        columns += [prefix+"_measured_jobs"]+[f"{prefix}_{s}_mib" for s in ["min","q25","median","q75","p95","max"]]
    return pd.DataFrame(records,columns=columns)


def ram_reports(root, rows, config):
    """Raw, condition-level, framework/task and matched RAM reports."""
    collected={row["job_id"]:row for row in rows}
    for name in ["backend_preflight.json","validation_results.json"]:
        path=root/name
        if path.exists():
            for row in json.loads(path.read_text()):
                if row.get("manifest_sha256")==config["manifest_sha256"]:
                    collected[row["job_id"]]=row
    frame=pd.DataFrame([{k:v for k,v in row.items() if not isinstance(v,(list,dict))}
                        for row in collected.values()])
    if frame.empty:frame=pd.DataFrame(columns=list(dict.fromkeys(["job_id","framework","status"]+CONDITION)))
    for column in CONDITION:
        if column not in frame: frame[column]="none"
    frame["model"]=frame.model.fillna("none")
    if "repetition" not in frame:frame["repetition"]=np.nan
    for column in RAM_METRICS.values():
        if column not in frame: frame[column]=np.nan
        frame[column]=pd.to_numeric(frame[column],errors="coerce")
    frame["lane_kind"]=frame.framework.map(lambda f:
        "simulator" if f in config["frameworks"] else "analytical" if f=="analytical_angle" else "classical")
    ordered=["job_id","framework","lane_kind","backend","internal_precision"]+CONDITION+[
        "repetition","status","train_samples","test_samples"]+list(RAM_METRICS.values())+[
        "worker_entry_rss_source","worker_exit_rss_source","error"]
    html_table(root,"ram_per_job",frame[[c for c in ordered if c in frame]],80)
    diagnostics=frame[frame.task.isin(["preflight","validation"])].copy()
    html_table(root,"ram_backend_checks",diagnostics[[c for c in ordered if c in frame]],40)
    benchmark=frame[~frame.task.isin(["preflight","validation"])].copy()
    successful=benchmark[benchmark.status=="ok"]
    valid=np.isfinite(successful.worker_peak_rss_mib)&(successful.worker_peak_rss_mib>0)
    if not valid.all():
        raise RuntimeError("Successful jobs are missing worker peak RAM. Inspect ram_per_job.csv and rerun those jobs.")
    stages=successful.primary_stage_peak_rss_mib.dropna()
    if (successful.loc[stages.index,"worker_peak_rss_mib"]+1e-9<stages).any():
        raise RuntimeError("A worker peak is below its earlier primary-stage high-water mark.")
    tables={
        "ram_by_condition":ram_aggregate(benchmark,CONDITION+["framework"]),
        "ram_by_framework_task":ram_aggregate(benchmark,["framework","task","track","threads"]),
        "ram_framework_summary":ram_aggregate(benchmark,["framework","threads"]),
    }
    for name,table in tables.items(): html_table(root,name,table,100)
    pairs=[]
    simulators=successful[successful.framework.isin(config["frameworks"])]
    for key,part in simulators.groupby(CONDITION+["repetition"],dropna=False,sort=False):
        if len(part)!=len(config["frameworks"]) or set(part.framework)!=set(config["frameworks"]): continue
        modified=float(part.loc[part.framework=="superfermion_optimized","worker_peak_rss_mib"].iloc[0])
        for _,row in part.iterrows():
            if row.framework=="superfermion_optimized": continue
            record=dict(zip(CONDITION+["repetition"],key))
            record.update(competitor=row.framework,competitor_peak_mib=float(row.worker_peak_rss_mib),
                modified_peak_mib=modified,competitor_over_modified_ram=float(row.worker_peak_rss_mib/modified),
                competitor_minus_modified_mib=float(row.worker_peak_rss_mib-modified))
            pairs.append(record)
    pair_columns=CONDITION+["repetition","competitor","competitor_peak_mib","modified_peak_mib",
                            "competitor_over_modified_ram","competitor_minus_modified_mib"]
    html_table(root,"ram_paired_comparisons",pd.DataFrame(pairs,columns=pair_columns),100)
    methodology=dict(units="MiB (2**20 bytes)",scope="CPU worker process resident RAM, including native allocations",
        worker_peak=("Windows psutil peak_wset at job end" if os.name=="nt" else "OS ru_maxrss/1024 at job end")+"; includes imports, setup, warmup and all later stages/audits performed by that job; sampled before JSON serialization",
        primary_stage_peak=("Windows psutil peak_wset" if os.name=="nt" else "Existing OS high-water mark")+" at the primary stage endpoint; includes earlier imports/setup/warmup and is not a resettable stage-only allocation counter",
        entry_exit=("Windows psutil current RSS before/after run(job)" if os.name=="nt" else "Current resident pages from /proc/self/statm before/after run(job)")+"; missing measurements stay missing, never a substituted high-water mark",
        rss_change="Exit minus entry resident RSS; retained change, not peak allocation; can be negative",
        aggregation="Median, IQR, P95 and maximum of per-job measurements, using successful jobs; not time-averaged RSS; failed and missing counts retained",
        overview="Framework and task summaries pool campaign settings; conditions and job counts are recorded; they are descriptive, not universal framework rankings",
        comparisons="Use ram_by_condition.csv or complete matched groups in ram_paired_comparisons.csv; compare the same dataset, seed, qubits, track, depth, batch, iterations, threads and model",
        limitations=["Does not measure GPU memory, virtual address size, whole-machine RAM or child-process memory",
                     "Shared resident pages mean process RSS values must not be summed as physical-machine RAM",
                     "Current RSS snapshots can miss transient peaks; OS worker high-water marks retain those peaks",
                     "Preflight/validation footprints are exported separately from benchmark summaries"])
    (root/"ram_methodology.json").write_text(json.dumps(methodology,indent=2))
    tables["ram_per_job"]=frame
    return tables


def task_cost_reports(root,rows,config,inputs):
    """Every planned condition/qubit, including failures and pending jobs."""
    planned=[job for group in groups(config,inputs) for job in group]
    key_fields=CONDITION+["repetition","framework"]
    def key(row):return tuple(row.get(k,"none") for k in key_fields)
    indexed={key(row):row for row in rows}
    records=[]
    for job in planned:
        result=indexed.get(key(job))
        record={**job,**(result or {})}
        record["status"]=result["status"] if result else "not_started"
        record["model"]=record.get("model","none")
        for metric in ["primary_runtime_s","worker_wall_time_s","worker_peak_rss_mib"]:
            record[metric]=record.get(metric,np.nan)
        records.append({k:v for k,v in record.items() if not isinstance(v,(dict,list))})
    frame=pd.DataFrame(records)
    html_table(root,"experiment_job_status",frame,100)
    result=[]
    for k,part in frame.groupby(CONDITION+["framework"],dropna=False,sort=False):
        record=dict(zip(CONDITION+["framework"],k))
        ok=part[part.status=="ok"]
        record.update(expected_jobs=len(part),successful_jobs=len(ok),
            failed_jobs=int((part.status=="error").sum()),pending_jobs=int((part.status=="not_started").sum()))
        for source,prefix in [("primary_runtime_s","primary_runtime"),
                              ("worker_wall_time_s","worker_wall_time"),
                              ("worker_peak_rss_mib","worker_peak")]:
            values=pd.to_numeric(ok[source],errors="coerce").dropna().to_numpy(dtype=float)
            values=values[np.isfinite(values)&(values>0)]
            for suffix,q in [("q25",.25),("median",.5),("q75",.75)]:
                units="mib" if source=="worker_peak_rss_mib" else "s"
                record[f"{prefix}_{suffix}_{units}"]=float(np.quantile(values,q)) if len(values) else np.nan
        result.append(record)
    summary=pd.DataFrame(result)
    html_table(root,"all_experiment_time_ram",summary,100)
    return summary,frame


def summarize_pairs(frame,frameworks):
    pairs=[]; coverage=[]
    simulators=frame[frame.framework.isin(frameworks)]
    for key,subset in simulators.groupby(CONDITION+["repetition"],dropna=False,sort=False):
        base=dict(zip(CONDITION+["repetition"],key)); present=set(subset.framework)
        complete=present==set(frameworks) and len(subset)==len(frameworks)
        coverage.append(dict(base,complete=complete,present=";".join(sorted(present))))
        if not complete: continue
        modified=float(subset[subset.framework=="superfermion_optimized"].primary_runtime_s.iloc[0])
        for _,row in subset.iterrows():
            if row.framework=="superfermion_optimized": continue
            pairs.append(dict(base,competitor=row.framework,competitor_over_modified=float(row.primary_runtime_s/modified)))
    return pd.DataFrame(pairs,columns=CONDITION+["repetition","competitor","competitor_over_modified"]),pd.DataFrame(coverage,columns=CONDITION+["repetition","complete","present"])


def audit_outcomes(rows,frameworks):
    buckets={}
    for row in rows:
        if row.get("status")!="ok" or row["framework"] not in frameworks: continue
        key=tuple(row.get(k,"none") for k in CONDITION+["repetition"])
        buckets.setdefault(key,[]).append(row)
    result=[]
    for key,group in buckets.items():
        ref=next((r for r in group if r["framework"]=="superfermion_source_baseline"),None)
        if not ref: continue
        for row in group:
            record=dict(zip(CONDITION+["repetition"],key)); record.update(framework=row["framework"])
            if "predictions" in ref and "predictions" in row:
                a,b=np.asarray(row["predictions"]),np.asarray(ref["predictions"])
                record["prediction_max_difference"]=float(np.max(np.abs(a-b)))
                record["prediction_mismatches"]=int(np.sum(a!=b)) if row["task"] in ["qsvm","qkpca","vqc"] else None
            for metric in ["final_energy","final_loss","balanced_accuracy","rmse"]:
                if metric in ref and metric in row: record[metric+"_difference"]=abs(row[metric]-ref[metric])
            for field in ["kernel_probe","test_kernel_probe","gradient","optimization_trace"]:
                if field in ref and field in row:
                    record[field+"_max_difference"]=float(np.max(np.abs(np.asarray(row[field])-np.asarray(ref[field]))))
            result.append(record)
    return pd.DataFrame(result)


class Figures:
    def __init__(self,root): self.root=root; self.paths={}; (root/"figures").mkdir(exist_ok=True)
    def save(self,key,fig):
        fig.savefig(self.root/"figures"/(key+".png"),bbox_inches="tight")
        fig.savefig(self.root/"figures"/(key+".svg"),bbox_inches="tight")
        plt.close(fig); self.paths[key]=str(self.root/"figures"/(key+".png"))

    def ram(self,tables):
        overview=tables["ram_framework_summary"]
        overview=overview[overview.framework.isin(ORDER)&(overview.measured_jobs>0)]
        if overview.empty: return
        threads=int(overview.threads.min())
        overview=overview[overview.threads==threads]
        order=[f for f in ORDER if f in set(overview.framework)]
        fig,ax=plt.subplots(figsize=(10.5,6))
        for index,framework in enumerate(order):
            row=overview[overview.framework==framework].iloc[0]
            ax.errorbar(row.worker_peak_median_mib,index,
                xerr=[[row.worker_peak_median_mib-row.worker_peak_q25_mib],
                      [row.worker_peak_q75_mib-row.worker_peak_median_mib]],
                fmt=MARKERS[framework],color=COLORS[framework],capsize=3,markersize=7)
        ax.set(yticks=range(len(order)),yticklabels=[LABELS[f] for f in order],
               xlabel="Worker peak resident RAM (MiB)",xlim=(0,None),
               title=f"RAM per simulator framework · {threads} thread budget")
        ax.set_ylim(len(order)-.5,-.5);ax.grid(axis="x")
        fig.text(.5,.015,"Median / IQR of per-job peaks across benchmark tasks and settings; includes setup and audits.\nPooled description, not a matched ranking. See condition and paired RAM tables.",
                 ha="center",fontsize=10)
        fig.tight_layout(rect=(0,.09,1,1));self.save("ram_framework_overview",fig)

        by_task=tables["ram_by_framework_task"]
        by_task=by_task[by_task.framework.isin(ORDER)&(by_task.threads==threads)&(by_task.measured_jobs>0)]
        if by_task.empty: return
        maximum=max(float(by_task.worker_peak_median_mib.max()),1.)
        cmap=plt.get_cmap("Blues").copy();cmap.set_bad("#eef1f3")
        for track in ["statevector","native_readout"]:
            part=by_task[by_task.track==track]
            if part.empty: continue
            tasks=[task for task in RAM_TASKS if task in set(part.task)]
            matrix=part.pivot(index="framework",columns="task",values="worker_peak_median_mib")
            matrix=matrix.reindex(index=order,columns=tasks)
            values=matrix.to_numpy(dtype=float)
            fig,ax=plt.subplots(figsize=(max(8.5,len(tasks)*1.05+3),6))
            image=ax.imshow(np.ma.masked_invalid(values),cmap=cmap,vmin=0,vmax=maximum,aspect="auto")
            for i in range(len(order)):
                for j in range(len(tasks)):
                    value=values[i,j]
                    ax.text(j,i,f"{value:.0f}" if np.isfinite(value) else "—",ha="center",va="center",
                        fontsize=10,color="white" if np.isfinite(value) and value>maximum*.65 else "#23272b")
            ax.set(xticks=range(len(tasks)),xticklabels=[RAM_TASK_LABELS[t] for t in tasks],
                   yticks=range(len(order)),yticklabels=[LABELS[f] for f in order],
                   title=f"RAM per framework and task · {track} · {threads} thread budget")
            plt.setp(ax.get_xticklabels(),rotation=35,ha="right")
            fig.colorbar(image,ax=ax,pad=.025,label="Median worker peak RAM (MiB)")
            fig.text(.5,.015,"Cells pool dataset/qubit/depth/seed settings within each task and track; job counts and IQR are in CSVs.\nGrey / — means not run; both tracks use the same colour scale. Includes setup and job diagnostics.",
                     ha="center",fontsize=10)
            fig.tight_layout(rect=(0,.1,1,1));self.save("ram_task_"+track,fig)

    def all_task_costs(self,summary,config):
        """Three panels per exact experiment condition; qubit grids never pooled."""
        cases=[c for c in CONDITION if c!="n_qubits"]
        index=[]
        for k,part in summary.groupby(cases,dropna=False,sort=True):
            case={field:(value.item() if isinstance(value,np.generic) else value) for field,value in zip(cases,k)};task=case["task"]
            profile=config["profile"]
            if task in ["vqc","vqr"]:requested=profile.get("variational_qubits",profile["qml_qubits"])
            elif task=="vqe":requested=profile["vqe_qubits"]
            elif task=="gradient":requested=profile["gradient_qubits"]
            elif task=="circuit":
                if case["dataset"] in ["batch_sweep","depth_sweep"]:requested=profile.get("sweep_qubits",[profile["batch_qubits"]])
                elif case["family"] in ["ghz","qft","random"]:requested=profile["state_qubits"]
                else:requested=profile["map_qubits"]
            else:requested=profile["qml_qubits"]
            framework_order=[f for f in ORDER+["analytical_angle","classical"] if f in set(part.framework)]
            fig,axes=plt.subplots(1,3,figsize=(16,4.8))
            panels=[("primary_runtime","s","Primary computation time (s)",True),
                    ("worker_wall_time","s","Full worker elapsed time (s)",True),
                    ("worker_peak","mib","Worker peak resident RAM (MiB)",False)]
            for ax,(prefix,units,label,logarithmic) in zip(axes,panels):
                measured=False
                for framework in framework_order:
                    values=part[part.framework==framework].set_index("n_qubits").reindex(requested)
                    y=values[f"{prefix}_median_{units}"].to_numpy(dtype=float)
                    lo=values[f"{prefix}_q25_{units}"].to_numpy(dtype=float)
                    hi=values[f"{prefix}_q75_{units}"].to_numpy(dtype=float)
                    measured |= bool(np.isfinite(y).any())
                    ax.plot(requested,y,color=COLORS.get(framework,"#23272b"),marker=MARKERS.get(framework,"o"),
                        label=LABELS.get(framework,framework),linestyle="--" if framework in ["cirq_f64","superfermion_source_baseline"] else "-")
                    ax.fill_between(requested,lo,hi,color=COLORS.get(framework,"#23272b"),alpha=.10)
                ax.set(xlabel="Qubits / projected input dimensions",ylabel=label,xticks=requested)
                ax.set_xlim(min(requested)-.5,max(requested)+.5)
                ax.grid(axis="y")
                if measured:
                    if logarithmic:ax.set_yscale("log")
                    else:ax.set_ylim(bottom=0)
                else:
                    ax.set_yticks([]);ax.text(.5,.5,"No successful measurements",transform=ax.transAxes,ha="center",va="center")
            title=f'{task.upper()} · {case["dataset"]} · {case["track"]}'
            subtitle=f'seed {case["seed"]}; threads {case["threads"]}'
            if task in ["qsvm","qkrr","qkpca"]:
                title+=' · '+str(case["feature_map"])+" map"
                kernel_cap=(config.get("kernel_row_caps") or {}).get(case["n_qubits"])
                subtitle+=('; kernel rows capped to %d train / %d test (exact same jobs in every lane); matched kernel strategy by qubit count'%(kernel_cap,kernel_cap)
                    if kernel_cap else '; full 80/20 dataset; matched kernel strategy by qubit count')
            elif task=="circuit":
                title+=' · '+str(case["family"])
                subtitle+=f'; depth {case["depth"]}; logical output batch {case["batch_size"]}; warmups {profile["warmups"]} + measured passes {profile["inner_repetitions"]}'
            elif task=="gradient":
                subtitle+=f'; depth {case["depth"]}; 2P shifts per gradient; warmups {profile["warmups"]} + measured passes {profile["inner_repetitions"]}'
            elif task in ["vqc","vqr"]:
                batch_label="full training batch" if profile.get("variational_full_batch",False) else "minibatch"
                subtitle+=f'; depth {case["depth"]}; {batch_label} {case["batch_size"]}; SPSA updates {case["iterations"]}; full training-loss trace'
            elif task=="classical":subtitle+='; model '+str(case["model"])+"; full 80/20 dataset"
            else:
                subtitle+=f'; depth {case["depth"]}; SPSA updates {case["iterations"]}'
                if task=="qaoa_fs":subtitle+='; real-feature/ancilla counts in dataset audit'
            success=int(part.successful_jobs.sum());expected=int(part.expected_jobs.sum())
            failures=int(part.failed_jobs.sum());pending=int(part.pending_jobs.sum())
            fig.suptitle(title+"\n"+subtitle,fontsize=12)
            handles,labels=axes[0].get_legend_handles_labels()
            fig.legend(handles,labels,loc="lower center",bbox_to_anchor=(.5,.115),ncol=5,frameon=False,fontsize=9)
            footer=f'{success}/{expected} planned jobs successful; {failures} failed; {pending} pending. Lines/bands show median/process IQR, not confidence intervals.\n'
            footer+='Gaps are missing measurements, never zero. RAM includes setup and audits. Full worker time includes imports/warmup/reference checks.\n'
            footer+='Kernel/state exports use the documented bounded strategy; backend workspaces need extra RAM. Feature-selection ancillas never create dataset features.'
            fig.text(.5,.005,footer,ha="center",fontsize=8.5)
            fig.tight_layout(rect=(0,.28,1,.89))
            token=hashlib.sha256(json.dumps(case,sort_keys=True,default=str).encode()).hexdigest()[:16]
            key="time_ram_"+task+"_"+token;self.save(key,fig)
            index.append(dict(case,key=key,figure_title=title,successful_jobs=success,expected_jobs=expected,
                              failed_jobs=failures,pending_jobs=pending,
                              png=self.paths[key],svg=str(self.root/"figures"/(key+".svg"))))
        html_table(self.root,"all_experiment_figures",pd.DataFrame(index),100)
        (self.root/"all_experiment_figures.json").write_text(json.dumps(index,indent=2))

    def latency(self,key,summary,task,facet="dataset",track="native_readout",feature_map=None):
        subset=summary[(summary.task==task)&(summary.track==track)&summary.framework.isin(ORDER)]
        if feature_map is not None: subset=subset[subset.feature_map==feature_map]
        if subset.empty: return
        # A figure shows one explicit n/depth/seed/thread configuration. The
        # complete tables retain every other configuration; no max-speedup
        # selection is used to choose the displayed slice.
        for field in ["n_qubits","depth","seed","threads"]: subset=subset[subset[field]==subset[field].min()]
        facets=list(dict.fromkeys(subset[facet])); cols=min(2,len(facets)); r=(len(facets)+cols-1)//cols
        fig,axes=plt.subplots(r,cols,figsize=(6.4*cols,4.7*r),squeeze=False,sharex=True,sharey=True)
        order=[f for f in ORDER if f in set(subset.framework)]
        low=max(float(subset.q25_ms.min())*.65,1e-4); high=float(subset.q75_ms.max())*1.6
        for ax,label in zip(axes.flat,facets):
            part=subset[subset[facet]==label]
            for index,f in enumerate(order):
                available=part[part.framework==f]
                if available.empty:continue
                row=available.iloc[0]
                ax.errorbar(row.median_ms,index,xerr=[[row.median_ms-row.q25_ms],[row.q75_ms-row.median_ms]],
                    color=COLORS[f],fmt=MARKERS[f],capsize=3,markersize=6)
            ax.set(yticks=range(len(order)),yticklabels=[LABELS[f] for f in order],xscale="log",
                   xlim=(low,high),title=str(label),xlabel="Runtime (ms, log scale)")
            ax.grid(axis="x"); ax.set_ylim(len(order)-.5,-.5)
        for ax in list(axes.flat)[len(facets):]: ax.set_visible(False)
        row=subset.iloc[0]
        fig.suptitle(f'{task.upper()} · {track} · {int(row.n_qubits)} qubits · depth {int(row.depth)} · {int(row.threads)} thread(s)',y=1.01,fontsize=14)
        fig.text(.5,-.01,"Dots: median of fresh processes. Whiskers: process IQR, not a confidence interval. Fixed seed shown.",ha="center",fontsize=10)
        fig.tight_layout(); self.save(key,fig)

    def circuit_scaling(self,summary):
        subset=summary[(summary.task=="circuit")&(summary.dataset=="synthetic")&summary.family.isin(["ghz","qft","random"])]
        if subset.empty: return
        subset=subset[(subset.seed==subset.seed.min())&(subset.threads==subset.threads.min())]
        fig,axes=plt.subplots(1,3,figsize=(16,5),sharey=True)
        for ax,family in zip(axes,["ghz","qft","random"]):
            for f in ORDER:
                part=subset[(subset.framework==f)&(subset.family==family)].sort_values("n_qubits")
                if part.empty: continue
                ax.plot(part.n_qubits,part.median_ms,marker=MARKERS[f],color=COLORS[f],label=LABELS[f],
                        linestyle="--" if f in ["superfermion_source_baseline","cirq_f64"] else "-")
                ax.fill_between(part.n_qubits,part.q25_ms,part.q75_ms,color=COLORS[f],alpha=.12)
            ax.set(title=family.upper(),xlabel="Qubits",yscale="log",xticks=sorted(subset.n_qubits.unique()));ax.grid(axis="y")
        axes[0].set_ylabel("Statevector API runtime (ms, log scale)")
        handles,labels=axes[0].get_legend_handles_labels();fig.legend(handles,labels,loc="lower center",bbox_to_anchor=(.5,-.15),ncol=4,frameon=False)
        fig.suptitle("CPU statevector scaling · batch 1 · one fixed seed and thread budget; shaded process IQR",fontsize=14)
        fig.tight_layout();self.save("circuit_scaling",fig)

    def sweep(self,key,summary,dataset,xfield,title,per_circuit=False):
        subset=summary[(summary.task=="circuit")&(summary.dataset==dataset)]
        if subset.empty: return
        for field in ["n_qubits","seed","threads"]: subset=subset[subset[field]==subset[field].min()]
        fig,ax=plt.subplots(figsize=(9,5))
        for f in ORDER:
            part=subset[subset.framework==f].sort_values(xfield)
            divisor=part.batch_size if per_circuit else 1
            ax.plot(part[xfield],part.median_ms/divisor,color=COLORS[f],marker=MARKERS[f],label=LABELS[f])
            ax.fill_between(part[xfield],part.q25_ms/divisor,part.q75_ms/divisor,color=COLORS[f],alpha=.10)
        row=subset.iloc[0]
        ax.set(title=f'{title} · {int(row.n_qubits)} qubits · {int(row.threads)} thread(s)',xlabel="Batch size" if xfield=="batch_size" else "Circuit layers",
               ylabel="ms per circuit" if per_circuit else "Batch runtime (ms)",yscale="log",xticks=sorted(subset[xfield].unique()))
        ax.grid(axis="y");ax.legend(ncol=2,loc="upper left",bbox_to_anchor=(1,1),frameon=False)
        fig.text(.5,-.02,"Same complex128 output contract; qsim computes internally in complex64. Bands show process IQR.",ha="center",fontsize=10)
        fig.tight_layout();self.save(key,fig)

    def speedup_distribution(self,pairs):
        if pairs.empty:return
        order=[f for f in ORDER if f!="superfermion_optimized" and f in set(pairs.competitor)]
        arrays=[pairs[pairs.competitor==f].competitor_over_modified.to_numpy() for f in order]
        fig,ax=plt.subplots(figsize=(10,6));box=ax.boxplot(arrays,vert=False,tick_labels=[LABELS[f] for f in order],patch_artist=True,showfliers=False)
        for patch,f in zip(box["boxes"],order):patch.set_facecolor(COLORS[f]);patch.set_alpha(.55)
        ax.axvline(1,color="#272c31",linestyle="--",linewidth=1)
        ax.set(xscale="log",xlabel="Competitor runtime / modified SF runtime (log scale)",title="Paired runtime ratios across the measured configurations")
        ax.invert_yaxis();ax.grid(axis="x")
        fig.text(.5,-.01,"Below 1: competitor faster. Boxes show workload/repetition heterogeneity, not statistical confidence.",ha="center",fontsize=10)
        fig.tight_layout();self.save("paired_speedup_distribution",fig)

    def readout(self,frame):
        tasks=["vqc","vqr","qaoa_fs","maxcut","vqe","gradient"]
        part=frame[frame.task.isin(tasks)&frame.framework.isin(ORDER)]
        key=[k for k in CONDITION if k!="track"]+["repetition","framework"]
        wide=part.pivot_table(index=key,columns="track",values="primary_runtime_s",aggfunc="first").reset_index()
        if not {"statevector","native_readout"}<=set(wide):return
        wide=wide.dropna(subset=["statevector","native_readout"])
        wide["statevector_over_native"]=wide.statevector/wide.native_readout
        wide.to_csv(self.root/"native_readout_ratios.csv",index=False)
        fig,axes=plt.subplots(2,3,figsize=(15,10),sharex=True,sharey=True)
        for ax,task in zip(axes.flat,tasks):
            for index,f in enumerate(ORDER):
                values=wide[(wide.task==task)&(wide.framework==f)].statevector_over_native
                if values.empty:continue
                mid,q25,q75=float(values.median()),float(values.quantile(.25)),float(values.quantile(.75))
                ax.errorbar(mid,index,xerr=[[mid-q25],[q75-mid]],fmt=MARKERS[f],color=COLORS[f],capsize=3)
            ax.axvline(1,color="#333",linestyle="--");ax.set(title=task.upper(),xscale="log",yticks=range(len(ORDER)),yticklabels=[LABELS[f] for f in ORDER],xlabel="State export / native readout runtime")
            ax.grid(axis="x");ax.set_ylim(len(ORDER)-.5,-.5)
        fig.suptitle("Output-contract sensitivity · above 1 means native readout is faster",fontsize=14)
        fig.text(.5,-.01,"Dots: median paired ratio. Whiskers: IQR across recorded cases/repeats; equivalent observables, different APIs.",ha="center",fontsize=10)
        fig.tight_layout();self.save("readout_sensitivity",fig)

    def quality(self,frame):
        ref=frame[(frame.framework.isin(["superfermion_optimized","classical"]))&(frame.repetition==0)]
        ref=ref[(ref.seed==ref.seed.min())&(ref.n_qubits==ref.n_qubits.min())&(ref.threads==ref.threads.min())]
        series={"RBF SVC":ref[(ref.task=="classical")&(ref.model=="rbf_svc")],
                "Logistic":ref[(ref.task=="classical")&(ref.model=="logistic")],
                "Random forest":ref[(ref.task=="classical")&(ref.model=="rf_classifier")],
                "QSVM (ZZ)":ref[(ref.task=="qsvm")&(ref.feature_map=="zz")],
                "QKPCA + logistic":ref[ref.task=="qkpca"],
                "VQC":ref[(ref.task=="vqc")&(ref.track=="native_readout")]}
        datasets=list(dict.fromkeys(ref[ref.task=="qsvm"].dataset));fig,ax=plt.subplots(figsize=(10,6))
        colors=["#656b71","#92999f","#ac8a29","#174c78","#668aa8","#984d72"]
        for j,((label,part),color) in enumerate(zip(series.items(),colors)):
            values=[float(part[part.dataset==d].balanced_accuracy.iloc[0]) if len(part[part.dataset==d]) else np.nan for d in datasets]
            ax.plot(values,np.arange(len(datasets))+(j-2.5)*.09,linestyle="none",marker=["o","s","^","D","v","P"][j],color=color,label=label)
        ax.set(yticks=range(len(datasets)),yticklabels=datasets,xlim=(0,1.03),xlabel="Held-out balanced accuracy",title="QML and classical model quality · same projected input and split")
        ax.invert_yaxis();ax.grid(axis="x");ax.legend(loc="upper left",bbox_to_anchor=(1,1),frameon=False)
        fig.text(.5,-.02,"One small validation split shown; missing models are omitted. No confidence interval or quantum-advantage claim.",ha="center",fontsize=10)
        fig.tight_layout();self.save("classification_quality",fig)
        regression=ref[ref.task.isin(["qkrr","vqr","classical"])&ref.rmse.notna()]
        if regression.empty:return
        names=list(dict.fromkeys(regression.dataset));fig,axes=plt.subplots(1,len(names),figsize=(6*len(names),4),squeeze=False)
        for ax,d in zip(axes.flat,names):
            part=regression[regression.dataset==d]
            records=[]
            for _,row in part.iterrows():
                if row.task=="classical":label=row.model
                elif row.task=="vqr":
                    if row.track!="native_readout":continue
                    label="VQR"
                else:label="QKRR "+row.feature_map
                records.append((label,row.rmse))
            ax.barh([r[0] for r in records],[r[1] for r in records],color="#668aa8")
            ax.set(title=d,xlabel="Held-out RMSE (original target units)");ax.invert_yaxis();ax.grid(axis="x")
        fig.tight_layout();self.save("regression_quality",fig)

    def traces(self,rows):
        selected=[r for r in rows if r.get("status")=="ok" and r["framework"] in ["superfermion_optimized","qulacs","cirq"]
                  and r["track"]=="native_readout" and r["repetition"]==0 and "optimization_trace" in r]
        if not selected:return
        first_seed=min(r["seed"] for r in selected);selected=[r for r in selected if r["seed"]==first_seed]
        cases=list(dict.fromkeys((r["task"],r["dataset"],r["n_qubits"],r["depth"]) for r in selected))[:8]
        fig,axes=plt.subplots((len(cases)+1)//2,2,figsize=(12,4*((len(cases)+1)//2)),squeeze=False)
        for ax,case in zip(axes.flat,cases):
            part=[r for r in selected if (r["task"],r["dataset"],r["n_qubits"],r["depth"])==case]
            for row in part:
                f=row["framework"];values=row["optimization_trace"]
                ax.plot(range(len(values)),values,color=COLORS[f],marker=MARKERS[f],label=LABELS[f],linestyle="--" if f=="qulacs" else "-")
            ax.set(title=f'{case[0].upper()} · {case[1]} · {case[2]}q',xlabel="SPSA update",ylabel="Full training loss" if case[0] in ["vqc","vqr"] else "Energy")
            ax.grid(axis="y");ax.legend(frameon=False)
        for ax in list(axes.flat)[len(cases):]:ax.set_visible(False)
        fig.suptitle("Matched optimizer traces · overlap indicates numerical agreement, not better learning",fontsize=14,y=1.01)
        fig.tight_layout();self.save("optimization_traces",fig)

    def memory(self,summary):
        part=summary[(summary.task=="circuit")&(summary.family=="random")&(summary.dataset=="synthetic")]
        if part.empty:return
        part=part[(part.n_qubits==part.n_qubits.max())&(part.seed==part.seed.min())&(part.threads==part.threads.min())]
        fig,ax=plt.subplots(figsize=(9,5))
        for i,f in enumerate(ORDER):
            row=part[part.framework==f]
            if row.empty:continue
            ax.plot(float(row.peak_process_hwm_mib.iloc[0]),i,marker=MARKERS[f],color=COLORS[f],linestyle="none")
        ax.set(yticks=range(len(ORDER)),yticklabels=[LABELS[f] for f in ORDER],xlabel="Process high-water RSS (MiB)",title=f'Process memory footprint · random circuit · {int(part.n_qubits.max())} qubits')
        ax.set_xlim(left=0);ax.invert_yaxis();ax.grid(axis="x")
        fig.text(.5,-.01,"Includes imports, preparation and warmup; captured before numerical audit. This is not statevector-only memory.",ha="center",fontsize=10)
        fig.tight_layout();self.save("process_memory",fig)


def analyze(config):
    style();root=Path(config["root"]);inputs=verify_manifest(config)
    rows=load_results(root,config["manifest_sha256"])
    frame=pd.DataFrame([{k:v for k,v in row.items() if not isinstance(v,(list,dict))} for row in rows])
    for column in CONDITION:
        if column not in frame:frame[column]="none"
    frame["model"]=frame.model.fillna("none")
    for column in ["status","framework","repetition","primary_runtime_s","peak_measured_stage_mb"]+METRICS:
        if column not in frame:frame[column]=np.nan
    frame.to_csv(root/"all_results.csv",index=False);(root/"all_results.json").write_text(json.dumps(rows,indent=2))
    successful=frame[frame.status=="ok"].copy();summary=aggregate(successful)
    html_table(root,"timing_summary",summary,80)
    ram_tables=ram_reports(root,rows,config)
    pairs,coverage=summarize_pairs(successful,config["frameworks"])
    html_table(root,"paired_speedups",pairs,80);html_table(root,"group_coverage",coverage,60)
    agreement=audit_outcomes(rows,config["frameworks"]);html_table(root,"outcome_agreement",agreement,60)
    simulator_summary=summary[summary.framework.isin(config["frameworks"])]
    leaders=[]
    for key,part in simulator_summary.groupby(CONDITION,dropna=False,sort=False):
        if set(part.framework)!=set(config["frameworks"]):continue
        best=part.loc[part.median_ms.idxmin()];f64=part[part.framework!="cirq"];best64=f64.loc[f64.median_ms.idxmin()]
        modified=part[part.framework=="superfermion_optimized"].iloc[0]
        source=part[part.framework=="superfermion_source_baseline"].iloc[0]
        ql=part[part.framework=="qulacs"].iloc[0]
        leaders.append(dict(zip(CONDITION,key),fastest_all=best.framework,fastest_f64=best64.framework,
            modified_ms=modified.median_ms,source_over_modified=source.median_ms/modified.median_ms,
            qulacs_over_modified=ql.median_ms/modified.median_ms))
    leaders=pd.DataFrame(leaders,columns=CONDITION+["fastest_all","fastest_f64","modified_ms",
        "source_over_modified","qulacs_over_modified"]);html_table(root,"condition_leaders",leaders,100)
    quality_columns=[c for c in CONDITION+["framework","repetitions"]+METRICS if c in summary]
    quality=summary[quality_columns].copy()
    quality=quality[quality[[c for c in METRICS if c in quality]].notna().any(axis=1)]
    html_table(root,"model_and_solver_quality",quality,100)
    expected=sum(len(g) for g in groups(config,inputs))
    overview=dict(profile=config["profile_name"],expected_jobs=expected,completed_jobs=len(rows),
        successful_jobs=int((frame.status=="ok").sum()),failed_jobs=int((frame.status!="ok").sum()),
        dataset_names=[r["dataset"] for r in inputs["datasets"]],
        simulator_lanes=config["frameworks"],complete_process_groups=int(coverage.complete.sum()),
        measured_conditions=len(leaders),modified_fastest_all=int((leaders.fastest_all=="superfermion_optimized").sum()),
        modified_fastest_f64=int((leaders.fastest_f64=="superfermion_optimized").sum()),
        source_over_modified_median=float(leaders.source_over_modified.median()) if len(leaders) else None,
        source_over_modified_min=float(leaders.source_over_modified.min()) if len(leaders) else None,
        source_over_modified_max=float(leaders.source_over_modified.max()) if len(leaders) else None,
        qulacs_over_modified_median=float(leaders.qulacs_over_modified.median()) if len(leaders) else None,
        fresh_process_repetitions=config["profile"]["repetitions"],split_seeds=config["profile"]["seeds"],
        limitations=["CPU ideal statevector experiments only; no GPU, noise or hardware results",
            "Runtime IQR is descriptive; validation profile is not a publication-quality statistical study",
            "Small held-out splits and short optimizer budgets do not establish quantum ML advantage",
            "Process memory includes imports/preparation/warmup, not statevector-only memory"],
        attempted_complete=bool(len(rows)==expected),pending_jobs=max(0,expected-len(rows)),
        complete=bool(len(rows)==expected and (frame.status=="ok").all() and coverage.complete.all()))
    overview["dataset_names"]=list(dict.fromkeys(overview["dataset_names"]))
    (root/"overview.json").write_text(json.dumps(overview,indent=2))
    figures=Figures(root)
    cost_summary,cost_jobs=task_cost_reports(root,rows,config,inputs)
    figures.all_task_costs(cost_summary,config)
    for key,task,track,feature in [("qsvm_latency","qsvm","statevector","zz"),("qkrr_latency","qkrr","statevector","zz"),
        ("qkpca_latency","qkpca","statevector","zz"),("vqc_latency","vqc","native_readout",None),
        ("vqr_latency","vqr","native_readout",None),("qaoa_fs_latency","qaoa_fs","native_readout",None),
        ("maxcut_latency","maxcut","native_readout",None),("vqe_latency","vqe","native_readout",None),
        ("gradient_latency","gradient","native_readout",None)]:
        figures.latency(key,summary,task,track=track,feature_map=feature)
    figures.latency("feature_maps_latency",summary[summary.family.isin(["angle","phase","pauli","zz"])&(summary.dataset=="synthetic")],"circuit",facet="family",track="statevector")
    figures.circuit_scaling(summary)
    figures.sweep("batch_runtime",summary,"batch_sweep","batch_size","Native batch scaling")
    figures.sweep("batch_per_circuit",summary,"batch_sweep","batch_size","Native batch amortization",per_circuit=True)
    figures.sweep("depth_scaling",summary,"depth_sweep","depth","Random entangling circuit depth scaling")
    figures.speedup_distribution(pairs);figures.readout(successful);figures.quality(successful)
    figures.traces(rows);figures.memory(summary)
    figures.ram(ram_tables)
    (root/"figure_index.json").write_text(json.dumps(figures.paths,indent=2))
    print(json.dumps(overview,indent=2));print(f"Rendered {len(figures.paths)} figure views with PNG and editable SVG exports.")


if __name__=="__main__":analyze(json.loads(Path(sys.argv[1]).read_text()))
'''
(EXPERIMENT_ROOT/'analyze_study.py').write_bytes(MODULE_SOURCE.lstrip("\n").encode("utf-8"))
print("Wrote analyze_study.py")

Wrote analyze_study.py


#### Independent dense/Fourier/Pauli math checks

Open this source cell to inspect or modify `offline_checks.py`. Its SHA256 is recorded before timing.

In [13]:
MODULE_SOURCE = r'''
"""Independent NumPy checks; these are not compiled-framework benchmarks."""

import importlib.util
import itertools
import json
from pathlib import Path
import sys
import tempfile
import time

import numpy as np


def load(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec); spec.loader.exec_module(module)
    return module


def single(state, matrix, q):
    stride = 1 << q; pairs = state.reshape(-1, 2*stride)
    left, right = pairs[:, :stride].copy(), pairs[:, stride:].copy()
    pairs[:, :stride] = matrix[0, 0]*left+matrix[0, 1]*right
    pairs[:, stride:] = matrix[1, 0]*left+matrix[1, 1]*right


def simulate(n, operations, values):
    state = np.zeros(1 << n, complex); state[0] = 1
    for gate, wires, key in operations:
        angle = values[key] if key is not None else 0.
        c, s = np.cos(angle/2), np.sin(angle/2)
        if gate == "CX":
            indices = np.arange(len(state)); mask = ((indices >> wires[0]) & 1)*(1 << wires[1])
            state = state[indices ^ mask]
            continue
        matrices = {"H": np.array([[1, 1], [1, -1]])/np.sqrt(2), "X": np.array([[0, 1], [1, 0]]),
                    "RX": np.array([[c, -1j*s], [-1j*s, c]]), "RY": np.array([[c, -s], [s, c]]),
                    "RZ": np.diag([np.exp(-1j*angle/2), np.exp(1j*angle/2)]), "P": np.diag([1., np.exp(1j*angle)])}
        single(state, matrices[gate], wires[0])
    return state


PAULIS = [np.eye(2), np.array([[0, 1], [1, 0]]), np.array([[0, -1j], [1j, 0]]), np.diag([1, -1])]


def dense_pauli(codes):
    result = np.ones((1, 1), complex)
    for code in reversed(codes): result = np.kron(result, PAULIS[code])
    return result


def bitmask_expectation(state, codes):
    flip = sum(1 << q for q, p in enumerate(codes) if p in [1, 2])
    sign = sum(1 << q for q, p in enumerate(codes) if p in [2, 3])
    phase = (-1j)**codes.count(2)
    return sum(np.conj(z)*phase*((-1)**((i & sign).bit_count()))*state[i ^ flip] for i, z in enumerate(state))


def main(source_dir=None):
    here = Path(__file__).resolve().parent
    source = Path(source_dir) if source_dir else here.parent/"superfermion-optimized"
    worker = load("benchmark_math", here/"legacy_adapter.py")
    kernel = load("analytical_kernel", source/"superfermion/kernels.py")
    rng = np.random.default_rng(331)
    cases = 0; maximum = 0.
    for n in range(1, 6):
        state = rng.normal(size=1 << n)+1j*rng.normal(size=1 << n); state /= np.linalg.norm(state)
        for codes in itertools.product(range(4), repeat=n):
            expected = np.vdot(state, dense_pauli(codes) @ state)
            actual = bitmask_expectation(state, list(codes))
            maximum = max(maximum, abs(expected-actual)); cases += 1
            np.testing.assert_allclose(actual, expected, atol=2e-12)
        x, y = rng.normal(size=(5, n)), rng.normal(size=(3, n))
        encode = lambda rows: np.stack([simulate(n, worker.feature_spec(n, "angle"), worker.feature_values(row)) for row in rows])
        actual = kernel.angle_fidelity_kernel(x, y, block_rows=2)
        np.testing.assert_allclose(actual, np.abs(encode(x).conj() @ encode(y).T)**2, atol=2e-12)
        np.testing.assert_allclose(worker.mean_z_weights(n), [(n-2*i.bit_count())/n for i in range(1 << n)])
    n = 4; a = rng.normal(size=n); b = rng.normal(size=(n, n)); b = (b+b.T)/2; np.fill_diagonal(b, 0)
    constant = .71; h, pairs, offset = worker.qubo_ising(a, b, constant)
    indices = np.arange(1 << n); bits = ((indices[:, None] >> np.arange(n)) & 1).astype(float)
    costs = constant+bits @ a+.5*np.einsum("bi,ij,bj->b", bits, b, bits)
    spins = 1-2*bits
    ising_costs = offset+spins @ h+sum(value*spins[:, i]*spins[:, j] for i, j, value in pairs)
    np.testing.assert_allclose(costs, ising_costs, atol=1e-12)
    for gamma, beta in [(.11, .23), (.4, .3), (-.53, .17)]:
        actual = simulate(n, worker.qaoa_spec(n, pairs), worker.qaoa_values(n, h, pairs, gamma, beta))
        expected = np.exp(-1j*gamma*costs)/np.sqrt(1 << n)
        rx = np.array([[np.cos(beta), -1j*np.sin(beta)], [-1j*np.sin(beta), np.cos(beta)]])
        for q in range(n): single(expected, rx, q)
        phase = np.vdot(expected, actual); phase /= abs(phase)
        np.testing.assert_allclose(actual/phase, expected, atol=1e-12)

    class ReferenceAdapter:
        def __init__(self, framework, n, operations, output="state", terms=None, weights=None):
            self.n, self.operations, self.output = n, operations, output
            self.backend = "NumPy reference, offline harness check only"; self.precision = "complex128"
            self.calls = 0; self.execute_s = 0.
            self.observable = sum(coefficient*dense_pauli(pauli) for pauli, coefficient in terms) if terms else None
        def reset(self): self.calls = 0; self.execute_s = 0.
        def state(self, values):
            start = time.perf_counter(); result = simulate(self.n, self.operations, values)
            self.calls += 1; self.execute_s += time.perf_counter()-start; return result
        def states(self, values): return np.stack([self.state(row) for row in values])
        def readouts(self, values):
            return np.array([np.vdot(vector, self.observable @ vector).real for vector in self.states(values)])
    worker.Adapter = ReferenceAdapter
    tasks = 0
    with tempfile.TemporaryDirectory() as temporary:
        directory = Path(temporary)
        x = rng.normal(size=(24, n)); y = np.array([0, 1]*12)
        main_path = directory/"main.npz"; batch_path = directory/"batch.npz"; qubo_path = directory/"qubo.npz"
        np.savez(main_path, X_train=x[:16], y_train=y[:16], X_test=x[16:], y_test=y[16:])
        np.savez(batch_path, X_batch=x[:6])
        np.savez(qubo_path, a=a, B=b, const=constant, costvec=costs, cards=bits.sum(axis=1).astype(int), select_k=2)
        config = dict(framework="numpy_reference", n_qubits=n, seed=42, kernel_block_rows=2, vqc_layers=1,
                      vqc_batch=3, vqc_iterations=2, qaoa_iterations=2, data_path=str(main_path), qubo_path=str(qubo_path))
        for task in ["qsvm", "qkpca"]:
            job = {**config, "task": task}
            if task == "qkpca": job["data_path"] = str(batch_path)
            result = worker.report_kernel(job); json.dumps(result, allow_nan=False); tasks += 1
        for name in ["angle", "phase", "pauli", "zz"]:
            result = worker.report_feature({**config, "feature_map": name, "data_path": str(batch_path)})
            assert result["max_norm_error"] < 1e-12; tasks += 1
        for track in ["common_statevector", "native_readout"]:
            for function in [worker.report_vqc, worker.report_qaoa]:
                result = function({**config, "track": track}); json.dumps(result, allow_nan=False); tasks += 1
        common = worker.report_qaoa({**config, "track": "common_statevector"})
        native = worker.report_qaoa({**config, "track": "native_readout"})
        np.testing.assert_allclose(common["initial_energy"], native["initial_energy"], atol=1e-12)
        np.testing.assert_allclose(common["final_energy"], native["final_energy"], atol=1e-9)
        assert common["circuit_evaluations"] == native["circuit_evaluations"]
        result = worker.report_validation(config)
        assert all(abs(case["norm"]-1) < 1e-12 for case in result["cases"])
    report = {"status": "passed", "pauli_strings_checked": cases, "max_pauli_error": maximum,
              "harness_tasks_checked": tasks, "analytical_kernel": "matches independently simulated states",
              "qubo_ising": "matches all basis-state costs", "qaoa": "matches direct diagonal phase evolution",
              "compiled_rust_and_external_frameworks": "not tested by this offline checker"}
    import experiment_core as study
    qft_errors=[]
    for qubits in range(2,7):
        op,values=study.circuit_spec(qubits,"qft",1,42)
        input_state=simulate(qubits,op[:qubits],values)
        expected=np.fft.ifft(input_state)*np.sqrt(1 << qubits)
        actual=study.reference_state(qubits,op,values)
        qft_errors.append(float(np.max(np.abs(actual-expected))))
        np.testing.assert_allclose(actual,expected,atol=2e-12)
    upper=np.triu(b,1); h,pairs,offset=study.qubo_to_ising(a,upper,constant)
    np.testing.assert_allclose(costs,offset+spins@h+sum(c*spins[:,i]*spins[:,j] for i,j,c in pairs),atol=2e-12)
    terms=study.tfim_terms(n)+[([2,1,0,3],.17)]
    state=rng.normal(size=1 << n)+1j*rng.normal(size=1 << n); state/=np.linalg.norm(state)
    expected=np.vdot(state,study.hamiltonian_matrix(n,terms)@state).real
    np.testing.assert_allclose(study.pauli_expectations([state],terms,n)[0],expected,atol=2e-12)
    report.update(study_qft="agrees with dense Fourier transform for 2..6 qubits",
                  study_qft_max_error=max(qft_errors),study_upper_qubo="all basis costs agree",
                  study_mixed_pauli_observable="agrees with independently formed dense matrix")
    ground_errors=[]
    for qubits in [1,2,4,6]:
        for coupling,field in [(1.,.7),(.3,1.2),(0.,.7),(1.,0.)]:
            dense=study.hamiltonian_matrix(qubits,study.tfim_terms(qubits,coupling,field))
            exact=float(np.linalg.eigvalsh(dense)[0])
            reduced=study.tfim_ground_energy(qubits,coupling,field)
            ground_errors.append(abs(exact-reduced))
            np.testing.assert_allclose(reduced,exact,rtol=1e-12,atol=2e-12)
    kernel_errors=[]
    for family in ["angle","zz"]:
        operations=study.feature_spec(n,family)
        train=[study.feature_values(row,family) for row in rng.uniform(-1,1,(7,n))]
        test=[study.feature_values(row,family) for row in rng.uniform(-1,1,(3,n))]
        class IndependentStates:
            def __init__(self):self.n=n
            def states(self,rows):return np.stack([simulate(n,operations,row) for row in rows]) if rows else np.empty((0,1 << n),complex)
        adapter=IndependentStates()
        dense_train,dense_test=adapter.states(train),adapter.states(test)
        expected_train=np.abs(dense_train.conj()@dense_train.T)**2
        expected_test=np.abs(dense_test.conj()@dense_train.T)**2
        for budget,cap in [(512,1),(1024,2),(10_000,128)]:
            actual_train,actual_test,info=study.quantum_kernel_matrices(adapter,train,test,budget,cap)
            np.testing.assert_allclose(actual_train,expected_train,rtol=1e-12,atol=2e-12)
            np.testing.assert_allclose(actual_test,expected_test,rtol=1e-12,atol=2e-12)
            assert info["exported_bank_peak_bytes"]<=budget
            kernel_errors.append(max(float(np.max(np.abs(actual_train-expected_train))),float(np.max(np.abs(actual_test-expected_test)))))
    assert study.state_chunk_rows(24,512*2**20,2)==1
    try:study.state_chunk_rows(24,256*2**20,2)
    except MemoryError:pass
    else:raise AssertionError("An impossible two-bank budget was accepted")
    report.update(open_tfim_reference="16 small-chain dense-eigenvalue comparisons passed",
                  open_tfim_max_error=max(ground_errors),
                  bounded_kernel="Angle/ZZ kernels agree with independently retained states in dense and blocked modes",
                  bounded_kernel_max_error=max(kernel_errors),bounded_state_budget="24-qubit two-bank budget boundaries checked")
    from sklearn.datasets import load_iris
    from prepare_study_data import qubo_inputs
    iris=load_iris()
    logical=qubo_inputs(iris.data,iris.target,4,42)
    embedded=qubo_inputs(iris.data,iris.target,8,42,allow_ancillas=True)
    np.testing.assert_allclose(embedded["qubo_cost"],np.tile(logical["qubo_cost"],16),atol=2e-12)
    assert len(embedded["selected_original_features"])==4 and int(embedded["feature_selection_ancillas"])==4
    energies=[]
    for physical,qubo in [(4,logical),(8,embedded)]:
        h,pairs,offset=study.qubo_to_ising(qubo["qubo_a"],qubo["qubo_b"],float(qubo["qubo_constant"]))
        op,values=study.qaoa_spec_values(physical,h,pairs,1,[.31,.23])
        state=simulate(physical,op,values)
        energies.append(float(np.dot(np.abs(state)**2,qubo["qubo_cost"])))
    np.testing.assert_allclose(energies[0],energies[1],atol=2e-12)
    report["feature_selection_ancillas"]="8-qubit physical embedding preserves the 4-real-feature cost and QAOA energy"
    print(json.dumps(report, indent=2))


if __name__ == "__main__": main(sys.argv[1] if len(sys.argv) > 1 else None)
'''
(EXPERIMENT_ROOT/'offline_checks.py').write_bytes(MODULE_SOURCE.lstrip("\n").encode("utf-8"))
print("Wrote offline_checks.py")

Wrote offline_checks.py


### 7. Verify math independently of all simulator adapters

These checks compare bitmask Pauli values with dense matrices, the Angle
formula with explicit states, QUBO and Ising costs on every small basis
state, and QFT with a dense Fourier transform. They check the reference math;
the independent cross-framework gate below checks the compiled simulators.

The new scaling helpers also check bounded Angle/ZZ kernels against independently
retained states, exported-bank limits at 24 qubits, and the reduced open-TFIM
ground reference against 16 small-chain dense eigensolves. These checks do not
substitute for compiled-framework validation or constitute performance benchmarks.


In [14]:
run_logged([PYTHON,EXPERIMENT_ROOT/"offline_checks.py",OPT],"independent_math",timeout=600)
print((LOGS/"independent_math.log").read_text())

Finished independent_math in 3.1 seconds
{
  "status": "passed",
  "pauli_strings_checked": 1364,
  "max_pauli_error": 2.220513810852149e-16,
  "harness_tasks_checked": 10,
  "analytical_kernel": "matches independently simulated states",
  "qubo_ising": "matches all basis-state costs",
  "qaoa": "matches direct diagonal phase evolution",
  "compiled_rust_and_external_frameworks": "not tested by this offline checker",
  "study_qft": "agrees with dense Fourier transform for 2..6 qubits",
  "study_qft_max_error": 2.8305244335018383e-16,
  "study_upper_qubo": "all basis costs agree",
  "study_mixed_pauli_observable": "agrees with independently formed dense matrix",
  "open_tfim_reference": "16 small-chain dense-eigenvalue comparisons passed",
  "open_tfim_max_error": 9.769962616701378e-15,
  "bounded_kernel": "Angle/ZZ kernels agree with independently retained states in dense and blocked modes",
  "bounded_kernel_max_error": 6.661338147750939e-16,
  "bounded_state_budget": "24-qubit two-ba

## Data

### 8. Prepare shared training and test inputs

| Dataset | Kind | Labels / target | Origin |
|---|---|---|---|
| Breast Cancer | Binary classification | Original scikit-learn labels | Wisconsin diagnostic data |
| Iris | Multiclass classification | All three species | Bundled real dataset |
| Wine | Multiclass classification | All three classes | Bundled real dataset |
| Digits 0/1 | Binary classification | Only handwritten digits 0 and 1 | Subset of bundled digits |
| Moons | Binary classification | Two noisy crescents | Seeded synthetic generator |
| Diabetes | Regression | Continuous original response | Bundled real dataset |
| Friedman1 | Regression | Nonlinear continuous response | Seeded synthetic generator |

Use all available rows in an **80% training / 20% testing** split.
Classification splits are stratified and each split uses its configured seed.
There are no fixed sample caps; integer rounding is recorded in the audit. Fit `StandardScaler`, full-SVD PCA, and the 95th-percentile
angle limit **on training rows only**. Transform held-out rows with those
fitted objects, clip angles to \([-\pi,\pi]\), and save identical inputs for
every backend. Regression target mean/std also come from training only.
No test label enters feature selection, PCA, angle scaling or model fitting.

If qubits exceed available PCA components, extra angles are zero-padded.
The audit reports active components and padded qubits; padding adds no new
dataset information. Original rows, indices and preprocessing arrays are
retained in each NPZ. QAOA feature selection uses original features rather
than PCA coordinates, with train-only relevance/correlation coefficients.

Feature-selection logical variables represent original features. A physical
register may add explicitly audited ancillas. Wine's 16/20/24-qubit registers embed its 13-feature
problem with 3/7/11 ancillas. Ancillary coefficients are zero, the cardinality
penalty acts only on real feature bits and downstream models ignore ancillary
bits. The dataset audit records both counts. No feature values are fabricated.
The requested preset supports every feature-selection grid point this way;
disabling `feature_selection_ancillas` restores the explicit unsupported-case guard.
QUBO diagonal costs are constructed in small bit-index blocks, avoiding a
2**24-by-24 temporary bit matrix. The full cost vector is still retained.


In [15]:
CONFIG_PATH = EXPERIMENT_ROOT/"run_config.json"
CONFIG_PATH.write_text(json.dumps(CONF,indent=2))
run_logged([PYTHON,EXPERIMENT_ROOT/"prepare_study_data.py",CONFIG_PATH],"prepare_data",timeout=600)
INPUTS = json.loads((EXPERIMENT_ROOT/"data_manifest.json").read_text())
from IPython.display import HTML, Image, display

def show_table(name,fields=None,filters=None,limit=25):
    path=EXPERIMENT_ROOT/(name+".csv")
    with path.open() as handle: rows=list(csv.DictReader(handle))
    if filters: rows=[r for r in rows if all(r.get(k)==str(v) for k,v in filters.items())]
    fields=fields or (list(rows[0]) if rows else [])
    def fmt(value):
        if value in [None,"","nan"]:return "—"
        try:return f"{float(value):.5g}"
        except (ValueError,TypeError):return str(value)
    table='<table><thead><tr>'+''.join('<th>'+html.escape(k)+'</th>' for k in fields)+'</tr></thead><tbody>'
    table+=''.join('<tr>'+''.join('<td>'+html.escape(fmt(r.get(k)))+'</td>' for k in fields)+'</tr>' for r in rows[:limit])
    table+='</tbody></table>'
    display(HTML('<style>th,td{padding:6px 9px;border-bottom:1px solid #ddd;text-align:left}table{border-collapse:collapse}</style>'+table))
    print(f"Showing {min(limit,len(rows))}/{len(rows)} rows; full table: {path.name}")

show_table("dataset_audit",["dataset","kind","original_samples","original_features","n_qubits",
    "active_components","padded_qubits","train_samples","test_samples",
    "actual_train_ratio","actual_test_ratio","feature_selection_available",
    "feature_selection_active_features","feature_selection_ancilla_qubits","pca_explained_variance"])

Finished prepare_data in 10.5 seconds


dataset,kind,original_samples,original_features,n_qubits,active_components,padded_qubits,train_samples,test_samples,actual_train_ratio,actual_test_ratio,feature_selection_available,feature_selection_active_features,feature_selection_ancilla_qubits,pca_explained_variance
breast_cancer,classification,569,30,4,4,0,455,114,0.79965,0.20035,True,4,0,0.79626
breast_cancer,classification,569,30,8,8,0,455,114,0.79965,0.20035,True,8,0,0.92843
breast_cancer,classification,569,30,12,12,0,455,114,0.79965,0.20035,True,12,0,0.97041
breast_cancer,classification,569,30,16,16,0,455,114,0.79965,0.20035,True,16,0,0.98941
breast_cancer,classification,569,30,20,20,0,455,114,0.79965,0.20035,True,20,0,0.99585
iris,classification,150,4,4,4,0,120,30,0.8,0.2,False,0,0,1
iris,classification,150,4,8,4,4,120,30,0.8,0.2,False,0,0,1
iris,classification,150,4,12,4,8,120,30,0.8,0.2,False,0,0,1
iris,classification,150,4,16,4,12,120,30,0.8,0.2,False,0,0,1
iris,classification,150,4,20,4,16,120,30,0.8,0.2,False,0,0,1


Showing 25/35 rows; full table: dataset_audit.csv


### 9. Record source, wheels, native extensions, packages and data

The manifest records actual extension and wheel hashes, compiler version and
flags, machine/platform information, script hashes, and input bytes. The
published wheel's build choices can differ from the source builds; use
**SF source vs SF modified** to attribute changes to this patch.

In [16]:
PROVENANCE = []
probe = r"""
import hashlib, json, os, pathlib, sys
site = pathlib.Path(sys.argv[1]).resolve()
sys.path.insert(0,str(site))
import superfermion as sf
import superfermion._sf_core as core
assert pathlib.Path(sf.__file__).resolve().is_relative_to(site)
print(json.dumps({'version': sf.__version__, 'package_file': sf.__file__,
                  'extension_file': core.__file__, 'extension_sha256': hashlib.sha256(pathlib.Path(core.__file__).read_bytes()).hexdigest(),
                  'has_prepared_api': hasattr(sf, 'prepare')}))
"""
for framework, site in SITES.items():
    env = os.environ.copy(); env.update(PYTHONPATH=site, EXPECTED_SITE=site, SF_F32="0")
    result = json.loads(subprocess.check_output([PYTHON, "-I", "-c", probe, site], cwd=ROOT, env=env, text=True))
    assert result["version"] == "0.1.14"
    assert result["has_prepared_api"] == (framework == "superfermion_optimized")
    result.update(framework=framework, wheel=SITE_WHEELS[framework],
                  wheel_sha256=hashlib.sha256(Path(SITE_WHEELS[framework]).read_bytes()).hexdigest())
    PROVENANCE.append(result)
packages = json.loads(subprocess.check_output([PYTHON, "-I", "-m", "pip", "list", "--format=json"], text=True))
cpu_probe = subprocess.run(["lscpu"], text=True, capture_output=True) if shutil.which("lscpu") else None
cpu = cpu_probe.stdout if cpu_probe is not None and cpu_probe.returncode == 0 else json.dumps({
    "machine": platform.machine(), "logical_cpus": os.cpu_count(), "processor": platform.processor(),
    "detail_unavailable": cpu_probe.stderr.strip() if cpu_probe else "lscpu not installed"})
scripts = ["legacy_adapter.py","experiment_core.py","prepare_study_data.py","study_worker.py","study_driver.py","analyze_study.py","offline_checks.py"]
MANIFEST = dict(base_commit=BASE_COMMIT, patch_sha256=PATCH_SHA256, source_hashes=EXPECTED_SOURCE_SHA256,
                installations=PROVENANCE, packages=packages, cpu=cpu, platform=platform.platform(),
                python=subprocess.check_output([PYTHON, "--version"], text=True).strip(),
                rust=subprocess.check_output([CARGO, "--version"], env=BUILD_ENV, text=True).strip(),
                build_flags="--release --locked; repository portable CPU flags; no native/LTO overrides",
                rustflags=BUILD_ENV.get("RUSTFLAGS", "repository .cargo/config.toml"),
                script_sha256={name: hashlib.sha256((EXPERIMENT_ROOT/name).read_bytes()).hexdigest() for name in scripts},
                data=INPUTS, profile=RUN_PROFILE, repetitions=PROFILE["repetitions"], threads=THREADS,
                experiment_profile=PROFILE,simulator_lanes=FRAMEWORKS,setup_logs=str(LOGS),
                worker_invocation="Python -I bootstrap; explicit benchmark and selected library paths",
                dependency_lock_sha256=hashlib.sha256(LOCK_PATH.read_bytes()).hexdigest(),
                notebook_revision="requested_grid_24q_v8_1_desktop_mt16_t20min")
manifest_path = EXPERIMENT_ROOT/"manifest.json"; manifest_path.write_text(json.dumps(MANIFEST, indent=2))
CONF["manifest_sha256"] = hashlib.sha256(manifest_path.read_bytes()).hexdigest()
CONFIG_PATH.write_text(json.dumps(CONF, indent=2))
print("Manifest SHA256:", CONF["manifest_sha256"])
for installation in PROVENANCE:
    print(installation["framework"], installation["wheel_sha256"][:16], installation["extension_sha256"][:16])

AssertionError: 

### 10. Check every simulator before timing

At the configured validation qubits (4/8/12 for the requested grid), compare full first/last batch statevectors against
independent NumPy propagation for Angle, Phase, Pauli, ZZ, GHZ, random and
nontrivial-input QFT circuits. Check asymmetric endian order, mixed X/Y/Z
observables, native mean-Z/diagonal batches, and parameter-shift gradients
against finite differences. f64 tolerance is `3e-9`; qsim f32 uses `3e-5`,
scaled by observable magnitude where needed. Each later state job also
checks all norms and full first/last states outside timing.

A failed gate stops the campaign. Failed worker logs are retained. No
incomplete group is presented as a complete framework comparison.

The native smoke test checks actual imports, two parameter rows, nontrivial state
amplitudes, and mean-Z expectations in each installed backend. `backend_preflight.json`
records Python paths, package versions, module files and isolation flags. This
is preparation, not an extra timing comparison. A failure prints the original
worker traceback in this cell, in addition to saving it as JSON.

If this cell fails in your Colab runtime, read the **ROOT ERROR** displayed above
the final controller error. A missing import, native library loading error,
worker signal/timeout, and numerical mismatch are distinct failures; no generic
fallback or tolerance increase is applied. To inspect an earlier failed run:

```python
for item in json.loads((EXPERIMENT_ROOT/"validation_errors.json").read_text()):
    print(item["framework"], item["n_qubits"])
    print(item.get("traceback", item.get("log_tail", item.get("error"))))
```

In [ ]:
import os
import json

# Save the original helper once, so rerunning this cell is safe.
if "_sf_run_logged_original" not in globals():
    _sf_run_logged_original = run_logged

def run_logged(command, name, cwd=None, env=None, timeout=3600):
    worker_env = os.environ.copy() if env is None else dict(env)
    worker_env["MPLBACKEND"] = "Agg"

    return _sf_run_logged_original(
        command,
        name,
        cwd=cwd,
        env=worker_env,
        timeout=timeout,
    )


# 1. Check backend imports and isolation.
run_logged(
    [PYTHON, EXPERIMENT_ROOT / "study_driver.py", CONFIG_PATH, "preflight"],
    "backend_preflight",
    timeout=1800,
)

print(
    (LOGS / "backend_preflight.log")
    .read_text(errors="replace")[-2500:]
)

PREFLIGHT = json.loads(
    (EXPERIMENT_ROOT / "backend_preflight.json").read_text()
)

assert PREFLIGHT, "Preflight returned no results."
assert all(
    row["status"] == "ok"
    and row["isolated_python"]
    and row["user_site_disabled"]
    for row in PREFLIGHT
), "Backend preflight failed. Inspect backend_preflight.log."

for row in PREFLIGHT:
    print(
        row["framework"], "->", row["backend"],
        "; isolated:", row["isolated_python"],
    )


# 2. Validate correctness before benchmarking.
run_logged(
    [PYTHON, EXPERIMENT_ROOT / "study_driver.py", CONFIG_PATH, "validate"],
    "independent_framework_checks",
    timeout=7200,
)

print(
    (LOGS / "independent_framework_checks.log")
    .read_text(errors="replace")[-3500:]
)

show_table(
    "correctness",
    [
        "framework",
        "n_qubits",
        "case",
        "max_amplitude_error",
        "max_norm_error",
        "native_readout_error",
        "max_gradient_error",
        "passed",
    ],
    limit=20,
)

### 11. Preview all requested jobs and bounded allocations

Preview emits the 2,520 supported jobs, full logical bank requirements and the
bounded exported-bank estimates. Large kernels share the same strategy across
frameworks. Allocation guards remain active; their 512 MiB limit covers exported
banks rather than total process RSS. Read actual RAM from worker measurements.

Workers execute sequentially and retain per-job checkpoints. Failed/time-out
workers remain explicit and do not stop later benchmark jobs when `fail_fast=False`.
Preflight/independent correctness errors still stop the run. A free session may
finish only part of this grid; analysis uses available checkpoints and reports
pending/failed conditions instead of presenting a complete performance ranking.
Groups run in ascending qubit bands; order is shuffled within bands and within
framework groups. Reports refresh between jobs every 900 seconds. Computation
timers and worker RAM are unaffected by simultaneous rendering because rendering
runs in a separate process between workers. Campaign elapsed time includes it.


In [ ]:
PLAN = None
for name in ["campaign_plan.json", "statebank_budget.csv"]:
    (EXPERIMENT_ROOT/name).unlink(missing_ok=True)
try:
    run_logged([PYTHON,EXPERIMENT_ROOT/"study_driver.py",CONFIG_PATH,"preview"],"job_preview",timeout=600)
finally:
    plan_path=EXPERIMENT_ROOT/"campaign_plan.json"
    if plan_path.exists():
        PLAN=json.loads(plan_path.read_text())
        print(json.dumps(PLAN,indent=2))
        show_table("planned_jobs",limit=20)
        show_table("statebank_budget",["task","dataset","framework","n_qubits","track",
            "execution_strategy","logical_statebank_mib","estimated_statebank_mib","limit_mib","limit_exceeded"],
            filters={"limit_exceeded":"True"},limit=12)
if PLAN is None:
    raise RuntimeError("Preview did not produce campaign_plan.json; inspect job_preview.log.")
if PLAN["jobs"]>MAX_JOBS:
    raise RuntimeError(f"Preview has {PLAN['jobs']} jobs; MAX_JOBS={MAX_JOBS}. Review the plan and adjust the grid or visible cap.")
if RUN_PROFILE!="validation" and not ALLOW_LARGE_RUN:
    raise RuntimeError("Set ALLOW_LARGE_RUN=True to intentionally execute the displayed campaign")


### 12. Run QSVM, kernel regression, QKPCA and classical controls

**QSVM/QKRR:** encode each sample once, construct
\(K_{ij}=|\langle\phi(x_i)|\phi(x_j)\rangle|^2\), then use a precomputed
SVC (`C=1`) or KernelRidge (`alpha=0.1`). Both Angle and ZZ kernels run on
every dataset. The Angle map is independent RY gates plus a fixed entangler,
so \(K_{ij}=\prod_q\cos^2((x_{iq}-x_{jq})/2)\) is an exact separate classical
baseline. ZZ includes data-dependent two-wire phases and uses simulation.

**QKPCA:** center the training kernel, retain up to three positive
eigencomponents, and center held-out kernel rows with training column/grand
means. Train a logistic head only on training embeddings. Kernel encoding,
matrix algebra and the eigensolver/head are separate timings.

**Classical controls:** use the same projected angles and fixed split.
Full-original-feature classification accuracy is recorded separately to
show whether dimensionality reduction removed useful information. No
hyperparameter search or test-based model selection is performed.

At high qubit counts, kernel banks are processed by bounded re-encoding blocks.
Large memory requirements remain visible in the preview as logical banks; the
actual execution uses bounded exports without changing the full dataset split.


In [ ]:
run_logged([PYTHON,EXPERIMENT_ROOT/"study_driver.py",CONFIG_PATH,"run","qsvm,qkrr,qkpca,classical"],
           "kernel_and_classical_campaign",timeout=172800)
print((LOGS/"kernel_and_classical_campaign.log").read_text()[-2200:])

### 13. Train VQC and VQR with matched optimizer budgets

A hardware-efficient ansatz adds RY/RZ layers and a CNOT chain to the Angle
encoding. Its readout is the mean of single-qubit Z expectations. Binary
VQC uses \(\sigma(2\langle\bar Z\rangle+b)\). Multiclass VQC uses one
parameterized circuit per class and softmax-normalized scores. VQR uses
\(a\langle\bar Z\rangle+b\), trained on train-standardized targets and
evaluated in original target units.

The requested preset uses five full-batch SPSA updates on 4/8/12 qubits.
Every perturbation loss uses the complete 80% training partition. Initial
parameters, row orders, perturbation signs and update schedules match across
all backends. Other presets may explicitly use minibatches. The learning rates are fixed:
\(a_k=0.12/(k+1)^{0.602}\), \(c_k=0.1/(k+1)^{0.101}\). Full training loss
is measured after every update for interpretable traces. No backend gets
extra training steps, early stopping, or a different optimizer. The final
training loss is reused rather than recomputed. Small validation budgets
can produce poor predictions; that is retained in the quality results.

In [ ]:
run_logged([PYTHON,EXPERIMENT_ROOT/"study_driver.py",CONFIG_PATH,"run","vqc,vqr"],
           "variational_qml_campaign",timeout=172800)
print((LOGS/"variational_qml_campaign.log").read_text()[-2200:])

### 14. Run QAOA feature selection, MaxCut and Ising VQE

**Feature selection:** rank original features by train-only mutual
information. Minimize a shared QUBO combining negative relevance, absolute
correlation redundancy and a cardinality penalty. With binary variables
\(z_i\), the cost is
\(-\sum_i r_i z_i+0.25\sum_{i<j}|\rho_{ij}|z_i z_j+1.5(\sum_i z_i-k)^2\),
with \(k=\max(1,\lfloor m/2\rfloor)\), where \(m\) is the number of
real feature variables. Ancillas have zero coefficients, are excluded from
this cardinality penalty and do not enter the downstream feature selection. The QUBO→Ising transformation uses
\(z_i=(1-Z_i)/2\). A cached diagonal vector is given to all state-track
reductions and equivalent native Pauli observables to the native track.
The modal bitstring's original features train a logistic classifier;
an empty selection is explicitly scored with a majority-class classifier.

**MaxCut:** weighted ring and seeded Erdős–Rényi graphs. Minimize negative
cut weight. Enumerate all bitstrings for the exact optimum and report
the expected-cut/optimal-cut ratio, energy gap and optimum probability.
These quantities differ from the cost of the modal bitstring.

**VQE:** the open transverse-field Ising Hamiltonian
\(H=-\sum_{q=0}^{n-2}Z_q Z_{q+1}-0.7\sum_{q=0}^{n-1}X_q\), a two-layer
RY/RZ ansatz, and an exact open-chain free-fermion ground-state energy. Mixed
observables make this a different workload from diagonal QAOA readouts.
Native training, final state diagnostics and downstream classifier time
are kept separate. The same matched SPSA budget applies to every lane.

For the open TFIM only, the exact reference is now `-sum(svd(bidiagonal(field,
coupling)))`, using an n-by-n matrix. Small systems are checked independently
against the original dense spin Hamiltonian. Reference time is recorded separately;
full worker elapsed plots include setup/references/diagnostics. QAOA diagonals
and MaxCut costs retain full exact bitstring costs but construct them in blocks.
See [The quantum Ising chain for beginners](https://arxiv.org/abs/2009.09208)
for the Jordan-Wigner/free-fermion diagonalization background.


In [ ]:
run_logged([PYTHON,EXPERIMENT_ROOT/"study_driver.py",CONFIG_PATH,"run","qaoa_fs,maxcut,vqe"],
           "optimization_campaign",timeout=172800)
print((LOGS/"optimization_campaign.log").read_text()[-2200:])

### 15. Differentiate with a matched parameter-shift algorithm

Every trainable RY/RZ angle occurs in exactly one gate, so
\(\partial_{\theta_j}f=(f(\theta_j+\pi/2)-f(\theta_j-\pi/2))/2\).
Batch the same \(2P\) shifted circuits for every backend. Check the resulting
vector against independent NumPy central finite differences (`epsilon=1e-6`).
Compare common state export with native scalar reductions.

This measures the same differentiation algorithm across all frameworks.
It does **not** claim that parameter shift is faster than each framework's
best adjoint/backprop implementation. Native-adjoint and accelerator studies
would be separate, capability-matched experiments.

In [ ]:
run_logged([PYTHON,EXPERIMENT_ROOT/"study_driver.py",CONFIG_PATH,"run","gradient"],
           "gradient_campaign",timeout=172800)
print((LOGS/"gradient_campaign.log").read_text()[-2200:])

### 16. Scale qubits, feature maps, batches and circuit depth

GHZ exercises Clifford structure but uses the same general statevector
contract for all lanes. QFT starts from a seeded nontrivial product state;
controlled phases are decomposed into P/CNOT gates and final bit swaps.
Random circuits interleave RY/RZ layers and alternating entangling chains.
Angle, Phase, Pauli and ZZ map batches use identical generated inputs.

Batch sweeps expose binding/export amortization; depth sweeps expose gate
work and fusion. Repetition order is paired and randomised. The memory metric
is Linux process high-water RSS at the end of the measured stage, before
independent numerical audits. It includes imports, preparation and warmup;
state-bank bytes are recorded separately. It is not a device-memory metric.

Every family, batch-size case and depth case now spans all six qubit counts.
Large state outputs are exported/discarded in bounded chunks. Every measured
state norm is reduced inside the timer; first/last full states are independently
replayed after timing for amplitude checks. All backends share this output contract.


In [ ]:
run_logged([PYTHON,EXPERIMENT_ROOT/"study_driver.py",CONFIG_PATH,"run","circuit"],
           "circuit_scaling_campaign",timeout=172800)
print((LOGS/"circuit_scaling_campaign.log").read_text()[-2200:])

## Results

### 17. Reconcile coverage, timings and outcome agreement

Only complete simulator groups receive paired speedups. Analytical and
classical algorithms remain separate lanes. Compare f64-only leaders as
well as all-lane leaders because qsim computes in f32. The source/modified
comparison uses the same compiler and flags. Ratios below one mean the
competitor is faster; those losses remain in the tables and figures.

Repeated model metrics are numerical consistency checks. Independent split
seeds, not repeated copies of the same test split, are needed for broader
ML conclusions. Native and state tracks are different output contracts, so
compare within a track before interpreting cross-track savings.

Analysis also reconciles the entire requested plan against available checkpoints.
Successful complete groups alone receive paired ratios. Failure/pending conditions
remain in the all-experiment time/RAM table, and every cost figure retains requested
qubit ticks with gaps for missing measurements. Single-repeat IQR is not uncertainty.


In [ ]:
run_logged([PYTHON,EXPERIMENT_ROOT/"analyze_study.py",CONFIG_PATH],"analysis",timeout=1800)
OVERVIEW=json.loads((EXPERIMENT_ROOT/"overview.json").read_text())
print(json.dumps(OVERVIEW,indent=2))
if not OVERVIEW["complete"]:
    print("INCOMPLETE CAMPAIGN:", OVERVIEW["failed_jobs"], "failed jobs;", OVERVIEW["pending_jobs"], "pending. Only successful complete groups receive paired comparisons.")
FIGURES=json.loads((EXPERIMENT_ROOT/"figure_index.json").read_text())
def show_figure(key):
    if key not in FIGURES:
        print(f"No figure for {key}: inspect successful measurements and experiment_job_status.csv.")
        return
    display(Image(filename=FIGURES[key]))
show_table("condition_leaders",["task","dataset","n_qubits","feature_map","family","track",
    "depth","batch_size","fastest_all","fastest_f64","source_over_modified","qulacs_over_modified"],limit=18)

### Kernel classification across five datasets

The figures show the ZZ map at the smallest recorded QML qubit count,
first thread budget and smallest seed value. That fixed slice is chosen
before viewing the winning times. Full Angle/ZZ and all other configurations
are retained in `timing_summary.csv`. Read encoding versus kernel-algebra
columns when larger sample counts make BLAS work dominate.

Displayed validation slice: ZZ map, 24 training and 12 test rows, four qubits, seed 42, two process repeats.

In [ ]:
show_figure("qsvm_latency")
show_table("timing_summary",["dataset","framework","median_ms","q25_ms","q75_ms","repetitions",
    "accuracy","balanced_accuracy","f1_macro"],filters={"task":"qsvm","feature_map":"zz"},limit=40)

### Kernel regression and representation learning

Regression targets are evaluated in their original units; comparing RMSE
magnitudes across unrelated target scales is not a model ranking. QKPCA
timing separates the simulator/kernel stage from eigendecomposition and
the classical head. Its held-out transform uses training means and eigenvectors.

In [ ]:
show_figure("qkrr_latency")
show_figure("qkpca_latency")
show_table("model_and_solver_quality",["task","dataset","framework","feature_map","rmse","mae","r2",
    "balanced_accuracy","qkpca_variance_fraction"],filters={"framework":"superfermion_optimized"},limit=30)

### Variational classification and regression

Compare native-readout runtime and test quality together. A faster library
runs the same training procedure more quickly; it does not change the
ansatz's expressive power or guarantee convergence. The three-class Iris
VQC has one circuit per class, which is more work than the binary classifier.

In [ ]:
show_figure("vqc_latency")
show_figure("vqr_latency")
show_table("model_and_solver_quality",["task","dataset","track","framework","final_loss","accuracy",
    "balanced_accuracy","rmse","mae","r2"],filters={"framework":"superfermion_optimized","track":"native_readout"},limit=25)

### QAOA and physical VQE

Exact-optimum gaps, cut ratios and solution probabilities complement timing.
A short validation optimizer can retain a large gap; no convergence target
is silently relaxed to make one backend appear better. The final modal
feature subset and downstream accuracy are present in the raw job records.

In [ ]:
show_figure("qaoa_fs_latency")
show_figure("maxcut_latency")
show_figure("vqe_latency")
show_table("model_and_solver_quality",["task","dataset","framework","final_energy","ground_energy",
    "energy_gap","approximation_ratio","optimum_probability","fs_downstream_accuracy"],
    filters={"framework":"superfermion_optimized","track":"native_readout"},limit=25)

### Gradient and readout sensitivity

Parameter-shift uses the same shifted-circuit count on every backend. The
second figure compares state-export and native-readout variants of the
same workload. Above one means native scalar execution saved time; below
one means its overhead outweighed export savings in that configuration.

In [ ]:
show_figure("gradient_latency")
show_figure("readout_sensitivity")
show_table("outcome_agreement",["task","dataset","framework","gradient_max_difference",
    "kernel_probe_max_difference","prediction_mismatches","final_energy_difference","final_loss_difference"],limit=25)

### Scaling and feature-map sensitivity

Lines connect genuinely different qubit/depth settings; bands show fresh
process IQR. Single-configuration figures use points and whiskers. Inspect
the batch plot before attributing a large speedup to faster gate kernels:
wrapper and export amortization can dominate at small qubit counts.

Validation figures: GHZ/QFT/random use batch 1 and 4/8/12 qubits. The feature-map comparison uses 16 circuits at eight qubits. Angle batch sweeps use 1/8/32 circuits; random depth sweeps use 1/4/8 layers, at eight qubits.

In [ ]:
show_figure("circuit_scaling")
show_figure("feature_maps_latency")

In [ ]:
show_figure("batch_runtime")
show_figure("batch_per_circuit")
show_figure("depth_scaling")

### Model quality and optimizer traces

Classical estimators use the same projected inputs and split. Their fitting
times are end-to-end model workload timings, so compare those with quantum
end-to-end timings, not with a gate-only or encoding-only number. The quality
charts show one validation split without a confidence interval; absent
models are omitted, not plotted as zero.

The trace figure uses full training losses or full energies at each SPSA
iterate. Overlapping simulator traces indicate numerical agreement. They
do not show that one implementation learned a better model.

In [ ]:
show_figure("classification_quality")
show_figure("regression_quality")
show_figure("optimization_traces")

### RAM consumption per framework

Each job runs in its own CPU worker. RSS includes Python objects, native
Rust/C++ buffers and resident library pages. Units are **MiB (2²⁰ bytes)**.

| Metric | Meaning |
|---|---|
| `worker_peak_rss_mib` | OS high-water mark at job end, before JSON serialization; includes imports, setup, warmup, fitting/inference, diagnostics and audits performed by the job |
| `primary_stage_peak_rss_mib` | Existing OS high-water mark at the primary stage endpoint; includes earlier imports/setup/warmup |
| `worker_entry_rss_mib`, `worker_exit_rss_mib` | Current RSS snapshots before/after the task from Linux procfs; unavailable readings stay missing |
| `worker_rss_change_mib` | Exit minus entry RSS; retained change, can be negative, and is not peak allocated memory |

Primary-stage HWM is not a resettable, stage-only allocation counter. Median,
IQR, P95 and maximum describe **per-job measurements**, not time-averaged RAM.
Pooled summaries mix workload settings; use condition-level/matched tables
for comparisons. Snapshots do not measure GPU, whole-machine or child-process
RAM. Do not sum process RSS as physical-machine usage because resident pages
can be shared. Recording uses endpoint reads without a polling thread inside
the timed stages.


In [ ]:
show_figure("ram_framework_overview")
show_table("ram_framework_summary",["framework","threads","tasks","conditions","measured_jobs",
    "worker_peak_median_mib","worker_peak_q25_mib","worker_peak_q75_mib",
    "worker_peak_p95_mib","worker_peak_max_mib","missing_peak_jobs"],limit=20)


### RAM per framework and each task

The heatmaps keep statevector and native-readout tracks separate and share one
colour scale. Cells pool settings within each task; tables report job counts,
IQR, P95 and maximum. Grey cells are not run, not zero. Analytical and classical
controls appear in the CSV tables with their own task sets. Preflight/validation
footprints are reported separately from benchmark summaries.


In [ ]:
show_figure("ram_task_statevector")
show_figure("ram_task_native_readout")
show_table("ram_by_framework_task",["task","track","framework","threads","measured_jobs","conditions",
    "worker_peak_median_mib","worker_peak_q25_mib","worker_peak_q75_mib",
    "worker_peak_p95_mib","worker_peak_max_mib","primary_stage_peak_median_mib"],limit=60)


### Inspect RAM for an exact workload

Edit `RAM_FILTERS` to choose a task, dataset, qubit count, track or framework.
The complete CSVs retain every setting and repeat. Paired ratios use complete
eight-backend groups with identical conditions. A competitor/modified RAM
ratio below 1 means the competitor used less peak resident RAM in that job.
Shared import/setup costs are included; this does not establish a universal
simulator-memory ranking. qsim remains labelled as the f32 lane.


In [ ]:
RAM_FILTERS={"task":"qsvm"}  # e.g. add "dataset":"iris", "n_qubits":4
show_table("ram_by_condition",["task","dataset","n_qubits","feature_map","track","depth",
    "batch_size","seed","threads","framework","measured_jobs","worker_peak_median_mib",
    "worker_peak_q25_mib","worker_peak_q75_mib","worker_peak_max_mib",
    "primary_stage_peak_median_mib","entry_rss_median_mib","exit_rss_median_mib"],
    filters=RAM_FILTERS,limit=24)
PAIR_FILTERS={k:v for k,v in RAM_FILTERS.items() if k!="framework"}
show_table("ram_paired_comparisons",["task","dataset","n_qubits","feature_map","track","seed",
    "repetition","competitor","competitor_peak_mib","modified_peak_mib",
    "competitor_over_modified_ram","competitor_minus_modified_mib"],filters=PAIR_FILTERS,limit=24)


### Raw RAM records and backend-check footprints

Raw rows include status and missing measurements; failures are never turned
into zero RAM. `ram_backend_checks.csv` contains preflight and correctness jobs,
whose primary-stage metric is not defined. Full measurement definitions are
exported in `ram_methodology.json`. All RAM CSV/HTML files, raw worker JSONs
and PNG/SVG figures are included automatically in the final results ZIP.


In [ ]:
show_table("ram_per_job",["framework","task","dataset","n_qubits","track","seed","repetition",
    "status","worker_peak_rss_mib","primary_stage_peak_rss_mib","worker_entry_rss_mib",
    "worker_exit_rss_mib","worker_rss_change_mib"],filters=RAM_FILTERS,limit=24)
show_table("ram_backend_checks",["framework","task","n_qubits","status",
    "worker_peak_rss_mib","worker_entry_rss_mib","worker_exit_rss_mib"],limit=32)


### Circuit footprint and all paired runtime ratios

The original circuit-memory view remains available. Runtime-ratio spread
reflects different workloads and repeats, not uncertainty around one universal
speedup factor. Full condition-level tables retain losses as well as wins.


In [ ]:
show_figure("process_memory")
show_figure("paired_speedup_distribution")
show_table("group_coverage",["task","dataset","n_qubits","track","repetition","complete"],limit=15)

### Time and RAM for every experiment across the requested qubit grid

Every exact task/dataset/map/family/track/depth/batch/iteration/seed/thread/model
condition has its own three-panel figure. Curves compare framework runtimes and
process RAM at all requested qubits without pooling datasets or qubit counts.
The full worker timer includes startup, setup, warmups, reference calculations,
audits and serialization; primary computation time retains the experiment's timer.
RAM is Linux worker high-water resident RAM, including native allocations.
Missing, unsupported, failed or pending values appear as gaps, not zero.
All figures are saved as PNG and editable SVG and are included in the result ZIP.


In [ ]:
COST_FIGURES=json.loads((EXPERIMENT_ROOT/"all_experiment_figures.json").read_text())
print("Exact-condition time/RAM figures:",len(COST_FIGURES))
show_table("all_experiment_figures",["task","dataset","feature_map","family","track","depth","batch_size",
    "iterations","successful_jobs","failed_jobs","pending_jobs","key"],limit=25)
if SHOW_ALL_EXPERIMENT_FIGURES:
    for figure in COST_FIGURES:
        print(figure["figure_title"])
        display(Image(filename=figure["png"],width=1400))
else:
    print("All files are saved. Set SHOW_ALL_EXPERIMENT_FIGURES=True to display every figure here.")


### Inspect exact time/RAM values and failed or pending jobs

Filter the CSV table by task, dataset, track, framework or qubit count. The complete
CSV contains every supported planned condition and the attempted/successful/failed/
pending counts. Process peaks are not added together as physical machine RAM.


In [ ]:
COST_FILTERS={}  # e.g. {"task":"vqe", "n_qubits":24}
show_table("all_experiment_time_ram",["task","dataset","n_qubits","framework","track","feature_map","family",
    "depth","batch_size","iterations","primary_runtime_median_s","worker_wall_time_median_s",
    "worker_peak_median_mib","expected_jobs","successful_jobs","failed_jobs","pending_jobs"],
    filters=COST_FILTERS,limit=40)
show_table("experiment_job_status",["task","dataset","n_qubits","framework","track","status","error",
    "primary_runtime_s","worker_wall_time_s","worker_peak_rss_mib"],filters={"status":"error"},limit=20)
show_table("unsupported_feature_selection",limit=20)


## Takeaways

The generated text below follows the measured outputs. Continue with the
larger study only after execution and numerical agreement are clean.
If Qulacs wins particular workloads, profile those workloads; a Rust-to-C++
rewrite is not automatically the remedy. Investigate prepared-plan work,
state allocation/export, gate fusion and memory bandwidth before changing
implementation language. Include a baseline's best comparable execution
API before making broad performance claims.

In [ ]:
print(f"Successful jobs: {OVERVIEW['successful_jobs']}; failed: {OVERVIEW['failed_jobs']}; pending: {OVERVIEW['pending_jobs']}.")
print("Complete campaign:",OVERVIEW["complete"])
print(f"Modified SF led {OVERVIEW['modified_fastest_f64']}/{OVERVIEW['measured_conditions']} complete f64 configurations and {OVERVIEW['modified_fastest_all']}/{OVERVIEW['measured_conditions']} including qsim f32.")
if OVERVIEW["source_over_modified_median"] is not None:
    print(f"Source/modified median paired ratio: {OVERVIEW['source_over_modified_median']:.3f}x.")
    print(f"Qulacs/modified median paired ratio: {OVERVIEW['qulacs_over_modified_median']:.3f}x; below 1 means Qulacs faster.")
print("Read all_experiment_time_ram.csv, experiment_job_status.csv and each condition's scaling figure for the complete grid.")
print("VQC/VQR use 4/8/12 qubits and 5 updates. Other supported quantum tasks request 4/8/12/16/20/24. Every dataset uses its full 80/20 split.")
print("One seed/repeat and incomplete large-qubit coverage cannot establish universal performance or quantum ML advantage.")


### 18. Export a reproducible source and results bundle

Retain fixed splits, preprocessing arrays, hashes, source patch/changed
files, configuration, raw active jobs, failed-job evidence, logs, numerical
checks, tables, PNG/SVG charts and exact dependency pins. The bundle excludes
large toolchain/venv caches. In Colab, download the generated ZIP and save
the executed notebook with **File → Download → Download .ipynb**.

In [ ]:
import zipfile
RESULT_ZIP=EXPERIMENT_ROOT/f"Superfermion_{RUN_PROFILE}_Expanded_Results.zip"
active=json.loads((EXPERIMENT_ROOT/"all_results.json").read_text())+json.loads((EXPERIMENT_ROOT/"validation_results.json").read_text())+json.loads((EXPERIMENT_ROOT/"backend_preflight.json").read_text())
active_ids={row["job_id"] for row in active}
with zipfile.ZipFile(RESULT_ZIP,"w",compression=zipfile.ZIP_DEFLATED) as archive:
    for path in EXPERIMENT_ROOT.rglob("*"):
        if not path.is_file() or path==RESULT_ZIP or "__pycache__" in path.parts: continue
        rel=path.relative_to(EXPERIMENT_ROOT)
        if rel.parts[0]=="raw" and path.name.split(".")[0] not in active_ids: continue
        archive.write(path,"results/"+str(rel))
    archive.write(PATCH_PATH,"modifications/superfermion_performance.patch")
    archive.writestr("modifications/source_manifest.json",json.dumps({"base_commit":BASE_COMMIT,
        "patch_sha256":PATCH_SHA256,"modified_file_sha256":EXPECTED_SOURCE_SHA256},indent=2))
    for relative in EXPECTED_SOURCE_SHA256:
        archive.write(OPT/relative,"modifications/modified_files/"+relative)
        if (BASE/relative).exists():archive.write(BASE/relative,"modifications/original_files/"+relative)
print("Bundle:",RESULT_ZIP.name,"bytes:",RESULT_ZIP.stat().st_size)
try:
    from google.colab import files
except ImportError:
    print("Local execution: notebook and ZIP paths are retained in the execution receipt.")
else:
    files.download(str(RESULT_ZIP))

### References and scope

* [Superfermion source](https://github.com/Catstate101/superfermion), pinned to the commit above.
* [Qiskit Aer Simulator](https://qiskit.github.io/qiskit-aer/stubs/qiskit_aer.AerSimulator.html): runtime parameter binding and explicit CPU options.
* [PennyLane QNode](https://docs.pennylane.ai/en/stable/code/api/pennylane.qnode.html) and [Lightning device](https://docs.pennylane.ai/projects/lightning/en/stable/code/api/pennylane_lightning.lightning_qubit.LightningQubit.html): broadcasting is expanded by Lightning, not intrinsically fused.
* [Cirq simulation](https://quantumai.google/cirq/simulate/simulation) and [qsim](https://quantumai.google/qsim): explicit separate f64/f32 lanes.
* [Qulacs advanced guide](https://docs.qulacs.org/en/latest/guide/2.0_python_advanced.html) and [installation/thread controls](https://docs.qulacs.org/en/latest/intro/1_install.html): parametric circuit/state reuse and rotation conventions.
* [scikit-learn dataset documentation](https://scikit-learn.org/stable/datasets/toy_dataset.html) and [synthetic generators](https://scikit-learn.org/stable/datasets/sample_generators.html): built-in sources and seeded generated datasets. Read the datasets' descriptions for original attribution.

These are ideal CPU statevector/API benchmarks and small classical-data QML
experiments. They do not compare every algorithm available in each framework,
nor demonstrate quantum advantage. Saved results identify the actually
executed profile, build flags and precision. Local checks cover notebook schema/syntax, the pure grid, independent small
numerical helpers, NumPy-reference worker paths and pending-only report rendering.
The full native campaign and hosted Colab runtime have not been exercised here.